# F2 — Vision Features and Image Twins

This notebook extracts SigLIP 2 and CLIP ViT-L/14 embeddings for VERITE, M4FC and GenImage, creates VAE and SDXL image twins, screens them for similarity and faces, re-saves originals and twins as JPEG quality 85, and exports the shared-format twin records.

It reads the F1 outputs in `data/processed/vision/` (VERITE splits and the GenImage export manifest) and the M4FC images from Y1.

In [ ]:
!pip install pandas pillow tqdm requests

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Main project directory in Google Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/twincheck")

# Create required directories
DIRECTORIES = [
    "data/raw/verite",
    "data/raw/genimage",
    "data/processed/vision",
    "data/copies/vision",
    "features/vision",
    "models/vision",
    "results/vision",
    "src/vision",
    "notes"
]

for directory in DIRECTORIES:
    (PROJECT_ROOT / directory).mkdir(
        parents=True,
        exist_ok=True
    )

print("Project workspace created successfully!")
print(f"Project location: {PROJECT_ROOT}")

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os

zip_path = (
    "/content/drive/MyDrive/twincheck/"
    "data/raw/m4fc/Y1_outputs.zip"
)

if os.path.exists(zip_path):
    print("SUCCESS: ZIP file found!")
    print("File size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")
else:
    print("ZIP file not found. Please check the Drive folder.")


In [ ]:

import zipfile
import os
from collections import Counter

zip_path = "/content/drive/MyDrive/twincheck/data/raw/m4fc/Y1_outputs.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = [info for info in z.infolist() if not info.is_dir()]

    print("=" * 60)
    print("Y1_OUTPUTS.ZIP CONTENT INSPECTION")
    print("=" * 60)

    print("Total files:", len(files))

    # Count files by extension
    extensions = Counter(
        os.path.splitext(info.filename)[1].lower()
        for info in files
    )

    print("\nFiles by extension:")
    for ext, count in extensions.items():
        print(f"{ext or '[no extension]'}: {count}")

    print("\nFirst 30 file paths:")
    for info in files[:30]:
        print(info.filename)

    print("\nTotal uncompressed size:",
          round(sum(f.file_size for f in files) / (1024**2), 2), "MB")

    # Check for potentially unsafe archive paths
    unsafe = [
        f.filename for f in files
        if f.filename.startswith("/")
        or ".." in f.filename.replace("\\", "/").split("/")
    ]

    print("\nUnsafe paths detected:", len(unsafe))


In [ ]:

import os
from collections import Counter

extract_dir = "/content/drive/MyDrive/twincheck/data/raw/m4fc/extracted"

all_files = []
for root, dirs, files in os.walk(extract_dir):
    for filename in files:
        all_files.append(os.path.join(root, filename))

extensions = Counter(
    os.path.splitext(f)[1].lower() for f in all_files
)

print("=" * 55)
print("M4FC EXTRACTION VERIFICATION")
print("=" * 55)

print("Total extracted files:", len(all_files))
print("\nFiles by extension:")
for ext, count in sorted(extensions.items()):
    print(ext, ":", count)

images = [f for f in all_files if f.lower().endswith(".jpg")]
metadata = [f for f in all_files if f.lower().endswith((".csv", ".jsonl"))]

print("\nJPEG images:", len(images))
print("Metadata files:", len(metadata))

print("\nMetadata filenames:")
for f in sorted(metadata):
    print("-", os.path.relpath(f, extract_dir))


In [ ]:

import os
import zipfile

zip_path = "/content/drive/MyDrive/twincheck/data/raw/m4fc/Y1_outputs.zip"
extract_dir = "/content/m4fc_inspection"

print("ZIP EXISTS:", os.path.exists(zip_path))
print("EXTRACTION FOLDER EXISTS:", os.path.exists(extract_dir))

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        print("\nFiles inside original ZIP:", len([
            n for n in z.namelist() if not n.endswith("/")
        ]))

if os.path.exists(extract_dir):
    print("Items inside extraction folder:", os.listdir(extract_dir)[:20])
else:
    print("Extraction folder has not been created.")


In [ ]:

import os
import zipfile
from pathlib import PurePosixPath

zip_path = "/content/drive/MyDrive/twincheck/data/raw/m4fc/Y1_outputs.zip"
extract_dir = "/content/m4fc_inspection"

# Confirm original ZIP
assert os.path.isfile(zip_path), "ZIP file not found!"

# Create temporary extraction folder
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as archive:

    # Check archive paths
    for name in archive.namelist():
        path = PurePosixPath(name)

        if path.is_absolute() or ".." in path.parts:
            raise ValueError(f"Unsafe path: {name}")

    archive.extractall(extract_dir)

print("Extraction completed successfully!")
print("Original ZIP remains unchanged.")
print("Extraction location:", extract_dir)


In [ ]:

import os
from collections import Counter

extract_dir = "/content/m4fc_inspection"

all_files = []

for root, dirs, files in os.walk(extract_dir):
    for filename in files:
        all_files.append(os.path.join(root, filename))

extensions = Counter(
    os.path.splitext(f)[1].lower()
    for f in all_files
)

images = [f for f in all_files if f.lower().endswith(".jpg")]
metadata = [
    f for f in all_files
    if f.lower().endswith((".csv", ".jsonl"))
]

print("=" * 55)
print("M4FC EXTRACTION VERIFICATION")
print("=" * 55)

print("Total extracted files:", len(all_files))
print("JPEG images:", len(images))
print("Metadata files:", len(metadata))

print("\nFiles by extension:")
for ext, count in sorted(extensions.items()):
    print(ext, ":", count)

print("\nMetadata filenames:")
for f in sorted(metadata):
    print("-", os.path.relpath(f, extract_dir))


In [ ]:

import os
import json
import pandas as pd

extract_dir = "/content/m4fc_inspection"
metadata_dir = os.path.join(extract_dir, "ar", "uncleaned")
manifest_path = os.path.join(
    extract_dir, "vision", "m4fc_images", "manifest.csv"
)

# Inspect both CSV files
csv_files = [
    os.path.join(metadata_dir, "m4fc_ar_claims_en.csv"),
    manifest_path
]

for path in csv_files:
    print("\n" + "=" * 70)
    print("CSV FILE:", os.path.basename(path))
    print("=" * 70)

    df = pd.read_csv(path, nrows=10)

    print("Columns:", df.columns.tolist())
    print("Sample records:")
    display(df.head(10))

# Inspect all five JSONL files
jsonl_files = [
    os.path.join(metadata_dir, name)
    for name in [
        "m4fc_ar_test.jsonl",
        "m4fc_ar_test_ctx.jsonl",
        "xfact_ar_test.jsonl",
        "xfact_ar_train.jsonl",
        "xfact_ar_val.jsonl"
    ]
]

for path in jsonl_files:
    print("\n" + "=" * 70)
    print("JSONL FILE:", os.path.basename(path))
    print("=" * 70)

    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
            if len(records) == 10:
                break

    print("Sample records:", len(records))

    for record in records:
        print(json.dumps(
            record,
            ensure_ascii=False,
            indent=2
        ))


Verify

In [ ]:

import os
import pandas as pd
import json
from PIL import Image
from collections import Counter

base_dir = "/content/m4fc_inspection"
image_dir = os.path.join(base_dir, "vision", "m4fc_images")

manifest_path = os.path.join(image_dir, "manifest.csv")
test_path = os.path.join(
    base_dir, "ar", "uncleaned", "m4fc_ar_test.jsonl"
)
ctx_path = os.path.join(
    base_dir, "ar", "uncleaned", "m4fc_ar_test_ctx.jsonl"
)

# Load manifest
manifest = pd.read_csv(manifest_path)

# Resolve image paths using the sample ID
manifest["resolved_path"] = manifest["id"].apply(
    lambda x: os.path.join(image_dir, f"{x}.jpg")
)

# Check image files
existing = manifest["resolved_path"].apply(os.path.isfile)

valid_images = []
invalid_images = []

for path in manifest.loc[existing, "resolved_path"]:
    try:
        with Image.open(path) as img:
            img.verify()
        valid_images.append(path)
    except Exception as e:
        invalid_images.append((path, str(e)))

# Load M4FC JSONL records
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test_records = load_jsonl(test_path)
ctx_records = load_jsonl(ctx_path)

manifest_ids = set(manifest["id"].astype(str))
test_ids = set(str(r["id"]) for r in test_records)
ctx_ids = set(str(r["id"]) for r in ctx_records)

print("=" * 60)
print("M4FC IMAGE AND METADATA VALIDATION")
print("=" * 60)

print("Manifest records:", len(manifest))
print("Unique manifest IDs:", manifest["id"].nunique())
print("Existing image files:", int(existing.sum()))
print("Missing image files:", int((~existing).sum()))
print("Valid images:", len(valid_images))
print("Invalid images:", len(invalid_images))

print("\nM4FC test JSONL records:", len(test_records))
print("Unique test IDs:", len(test_ids))
print("Test IDs missing from manifest:", len(test_ids - manifest_ids))

print("\nM4FC context JSONL records:", len(ctx_records))
print("Unique context IDs:", len(ctx_ids))
print("Context IDs missing from manifest:", len(ctx_ids - manifest_ids))

print("\nManifest split counts:")
print(manifest["split"].value_counts(dropna=False))

print("\nManifest status counts:")
print(manifest["status"].value_counts(dropna=False))

print("\nInvalid image examples:", invalid_images[:5])

if (
    len(manifest) == 603
    and manifest["id"].nunique() == 603
    and existing.all()
    and len(valid_images) == 603
    and len(invalid_images) == 0
):
    print("\nPASS: All 603 manifest images exist and are valid.")
else:
    print("\nCHECK REQUIRED: Some image validation conditions failed.")


In [ ]:
# Investigate unmatched M4FC test IDs

manifest_ids = set(manifest["id"].astype(str))
test_lookup = {str(r["id"]): r for r in test_records}

missing_test_ids = sorted(set(test_lookup) - manifest_ids)

print("=" * 60)
print("UNMATCHED M4FC TEST RECORDS")
print("=" * 60)

for record_id in missing_test_ids[:20]:
    record = test_lookup[record_id]

    print({
        "id": record.get("id"),
        "image_path": record.get("image_path"),
        "label": record.get("label"),
        "orig_label": record.get("orig_label")
    })

# Inspect failed downloads
print("\n" + "=" * 60)
print("FAILED IMAGE DOWNLOADS")
print("=" * 60)

failed = manifest[manifest["status"] == "failed"]

print(
    failed[["id", "split", "status", "url_used"]]
    .to_string(index=False)
)

# Check whether unmatched IDs appear in any manifest image paths
print("\n" + "=" * 60)
print("UNMATCHED ID / IMAGE PATH SUMMARY")
print("=" * 60)

for record_id in missing_test_ids[:20]:
    record = test_lookup[record_id]
    print(record_id, "->", record.get("image_path"))

In [ ]:
import os
import pandas as pd
from collections import Counter

# Resolve JSONL image paths to the extracted image directory
def resolve_image_path(image_path):
    if not image_path:
        return None

    filename = os.path.basename(image_path)
    return os.path.join(image_dir, filename)

# Check all test records
test_mapping = []

for record in test_records:
    original_path = record.get("image_path")
    resolved_path = resolve_image_path(original_path)

    test_mapping.append({
        "claim_id": record.get("id"),
        "image_path": original_path,
        "resolved_path": resolved_path,
        "image_exists": (
            os.path.isfile(resolved_path)
            if resolved_path else False
        ),
        "label": record.get("label"),
        "orig_label": record.get("orig_label")
    })

mapping_df = pd.DataFrame(test_mapping)

print("=" * 60)
print("M4FC CLAIM-TO-IMAGE MAPPING VALIDATION")
print("=" * 60)

print("Total test records:", len(mapping_df))
print("Records with image paths:", mapping_df["image_path"].notna().sum())
print("Records with existing images:", mapping_df["image_exists"].sum())
print("Records with missing images:", (~mapping_df["image_exists"]).sum())
print(
    "Unique image paths:",
    mapping_df["image_path"].dropna().nunique()
)

print("\nMissing image examples:")
print(
    mapping_df.loc[
        ~mapping_df["image_exists"],
        ["claim_id", "image_path"]
    ].head(20).to_string(index=False)
)

# Specifically check the 156 previously unmatched claim IDs
unmatched_mapping = mapping_df[
    mapping_df["claim_id"].isin(missing_test_ids)
]

print("\nPreviously unmatched claim IDs:")
print("Total:", len(unmatched_mapping))
print("Mapped to existing images:", unmatched_mapping["image_exists"].sum())
print("Mapped to missing images:", (~unmatched_mapping["image_exists"]).sum())

# Save the mapping for later use
output_path = os.path.join(
    "/content/drive/MyDrive/twincheck/data/processed/vision",
    "m4fc_test_claim_image_mapping.csv"
)

mapping_df.to_csv(output_path, index=False)
print("\nMapping saved to:", output_path)

In [ ]:
# Compare M4FC test and context JSONL mappings

test_map = {
    str(r["id"]): r.get("image_path")
    for r in test_records
}

ctx_map = {
    str(r["id"]): r.get("image_path")
    for r in ctx_records
}

test_ids = set(test_map)
ctx_ids = set(ctx_map)

common_ids = test_ids & ctx_ids

different_paths = [
    record_id for record_id in common_ids
    if test_map[record_id] != ctx_map[record_id]
]

print("=" * 60)
print("M4FC TEST VS CONTEXT MAPPING CHECK")
print("=" * 60)

print("Test records:", len(test_map))
print("Context records:", len(ctx_map))
print("Common claim IDs:", len(common_ids))
print("IDs only in test:", len(test_ids - ctx_ids))
print("IDs only in context:", len(ctx_ids - test_ids))
print("IDs with different image paths:", len(different_paths))

print("\nExamples of different paths:")
for record_id in different_paths[:10]:
    print(
        record_id,
        "| Test:", test_map[record_id],
        "| Context:", ctx_map[record_id]
    )

if (
    test_ids == ctx_ids
    and len(different_paths) == 0
):
    print("\nPASS: Test and context JSONL mappings agree.")
else:
    print("\nCHECK REQUIRED: Mapping differences found.")

In [ ]:
import json
import os

report = {
    "dataset": "M4FC Arabic",
    "validation": "image_and_metadata_validation",
    "manifest_records": 614,
    "unique_manifest_ids": 614,
    "downloaded_images": 603,
    "missing_image_files": 11,
    "invalid_images": 0,
    "test_jsonl_records": 770,
    "context_jsonl_records": 770,
    "test_context_common_ids": 770,
    "test_context_different_image_paths": 0,
    "test_records_with_image_paths": 770,
    "test_records_with_existing_images": 758,
    "test_records_with_missing_images": 12,
    "unique_test_image_paths": 614,
    "mapping_validation": "passed",
    "image_integrity_validation": "passed_for_downloaded_images",
    "overall_status": "partially_validated_missing_11_images"
}

report_path = (
    "/content/drive/MyDrive/twincheck/"
    "data/processed/vision/m4fc_validation_report.json"
)

os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4)

print("Report saved:", report_path)

with open(report_path, "r", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=4))

In [ ]:

import os
import pandas as pd
from PIL import Image

# Project directory
PROJECT_DIR = "/content/drive/MyDrive/twincheck"

# VERITE split files
SPLIT_DIR = os.path.join(
    PROJECT_DIR, "data", "processed", "vision", "splits"
)

split_files = {
    "train": os.path.join(SPLIT_DIR, "train.csv"),
    "validation": os.path.join(SPLIT_DIR, "validation.csv"),
    "test": os.path.join(SPLIT_DIR, "test.csv")
}

print("=" * 60)
print("VERITE FEATURE EXTRACTION INPUT AUDIT")
print("=" * 60)

datasets = {}
all_image_paths = set()

for split_name, csv_path in split_files.items():

    print(f"\n--- {split_name.upper()} ---")

    if not os.path.isfile(csv_path):
        print("ERROR: Split file not found:", csv_path)
        continue

    df = pd.read_csv(csv_path)
    datasets[split_name] = df

    print("Records:", len(df))
    print("Columns:", df.columns.tolist())

    # Check required columns
    required_columns = ["caption", "resolved_image_path", "label"]
    missing_columns = [
        col for col in required_columns if col not in df.columns
    ]

    if missing_columns:
        print("Missing required columns:", missing_columns)
        continue

    # Check image files
    exists = df["resolved_image_path"].apply(
        lambda p: isinstance(p, str) and os.path.isfile(p)
    )

    print("Existing image paths:", int(exists.sum()))
    print("Missing image paths:", int((~exists).sum()))

    # Check labels
    print("Label counts:")
    print(df["label"].value_counts(dropna=False))

    all_image_paths.update(
        df.loc[exists, "resolved_image_path"].tolist()
    )

print("\n" + "=" * 60)
print("OVERALL SUMMARY")
print("=" * 60)

print("Splits loaded:", list(datasets.keys()))
print("Unique accessible image paths:", len(all_image_paths))

# Check a few actual images
print("\nSample image checks:")

for image_path in list(sorted(all_image_paths))[:5]:
    try:
        with Image.open(image_path) as img:
            img.verify()
        print("VALID:", os.path.basename(image_path))
    except Exception as e:
        print("INVALID:", image_path, str(e))

print("\nAudit completed.")


In [ ]:

import torch
import os

print("=" * 55)
print("COLAB HARDWARE CHECK")
print("=" * 55)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)
    print("GPU memory:", round(props.total_memory / (1024**3), 2), "GB")
    print("Free GPU memory:",
          round(torch.cuda.mem_get_info()[0] / (1024**3), 2), "GB")
else:
    print("No GPU detected. Do not start model extraction yet.")

print("\nDrive available space:")
!df -h /content/drive | tail -n 1


In [ ]:

import psutil
import platform

memory = psutil.virtual_memory()

print("=" * 50)
print("CPU ENVIRONMENT CHECK")
print("=" * 50)

print("Operating system:", platform.system())
print("Processor:", platform.processor())
print("Total RAM:", round(memory.total / (1024**3), 2), "GB")
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("Used RAM:", round(memory.used / (1024**3), 2), "GB")
print("CPU cores:", psutil.cpu_count(logical=True))


In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
import os
import time
import torch
import pandas as pd
import psutil
import gc

from PIL import Image
from transformers import AutoProcessor, AutoModel

MODEL_ID = "google/siglip2-so400m-patch14-384"

PROJECT_DIR = "/content/drive/MyDrive/twincheck"

train_csv = os.path.join(
    PROJECT_DIR,
    "data/processed/vision/splits/train.csv"
)

df = pd.read_csv(train_csv).head(5)

print("Samples selected:", len(df))

print("\nLoading processor...")
start = time.time()

processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Processor loading time:", round(time.time() - start, 2), "seconds")

print("\nLoading SigLIP 2 on CPU...")
start = time.time()

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
)

model.eval()

print("Model loading time:", round(time.time() - start, 2), "seconds")

# Load five images and captions
images = [
    Image.open(path).convert("RGB")
    for path in df["resolved_image_path"]
]

captions = df["caption"].tolist()

inputs = processor(
    images=images,
    text=captions,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

print("\nRunning inference...")
start = time.time()

with torch.inference_mode():
    outputs = model(**inputs)

inference_time = time.time() - start

print("\n" + "=" * 50)
print("SIGLIP 2 CPU TEST RESULTS")
print("=" * 50)

print("Image feature shape:", outputs.image_embeds.shape)
print("Text feature shape:", outputs.text_embeds.shape)
print("Inference time:", round(inference_time, 2), "seconds")
print(
    "Available RAM:",
    round(psutil.virtual_memory().available / (1024**3), 2),
    "GB"
)

print("Model device:", next(model.parameters()).device)

# Release resources
del outputs, inputs, images, captions
del model, processor
gc.collect()

print("\nTest completed.")

In [ ]:

import time
import torch
import pandas as pd
import psutil
import gc
import os

from PIL import Image
from transformers import AutoProcessor, AutoModel

MODEL_ID = "google/siglip2-so400m-patch14-384"

PROJECT_DIR = "/content/drive/MyDrive/twincheck"

df = pd.read_csv(
    os.path.join(
        PROJECT_DIR,
        "data/processed/vision/splits/train.csv"
    )
).head(8)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
).eval()

images = [
    Image.open(p).convert("RGB")
    for p in df["resolved_image_path"]
]

captions = df["caption"].tolist()

for batch_size in [1, 2]:
    print(f"\nTesting batch size: {batch_size}")

    batch_times = []

    for start_idx in range(0, len(images), batch_size):
        end_idx = start_idx + batch_size

        batch_images = images[start_idx:end_idx]
        batch_captions = captions[start_idx:end_idx]

        inputs = processor(
            images=batch_images,
            text=batch_captions,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        start = time.time()

        with torch.inference_mode():
            outputs = model(**inputs)

        elapsed = time.time() - start
        batch_times.append(elapsed)

        del inputs, outputs
        gc.collect()

    print("Total inference time:", round(sum(batch_times), 2), "seconds")
    print(
        "Average time per sample:",
        round(sum(batch_times) / len(images), 2),
        "seconds"
    )
    print(
        "Available RAM:",
        round(psutil.virtual_memory().available / 1024**3, 2),
        "GB"
    )

del model, processor, images, captions
gc.collect()


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/twincheck"

FEATURE_DIR = os.path.join(
    PROJECT_DIR, "features", "vision", "verite"
)

os.makedirs(
    os.path.join(FEATURE_DIR, "siglip2"),
    exist_ok=True
)

os.makedirs(
    os.path.join(FEATURE_DIR, "clip"),
    exist_ok=True
)

print("Feature directories prepared:")

for root, dirs, files in os.walk(FEATURE_DIR):
    print(root)

In [ ]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/twincheck"
SPLIT_DIR = os.path.join(
    PROJECT_DIR, "data", "processed", "vision", "splits"
)

split_names = ["train", "validation", "test"]
all_dfs = []

for split in split_names:
    path = os.path.join(SPLIT_DIR, f"{split}.csv")
    df = pd.read_csv(path)
    df["split"] = split
    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)

print("Total records:", len(combined_df))
print("Unique image IDs:", combined_df["image_id"].nunique())
print("Unique image paths:", combined_df["resolved_image_path"].nunique())
print("Unique captions:", combined_df["caption"].nunique())

# Check whether an image appears in multiple splits
image_split_counts = combined_df.groupby("image_id")["split"].nunique()

print(
    "Images appearing in multiple splits:",
    int((image_split_counts > 1).sum())
)

assert len(combined_df) == 1001
assert combined_df["image_id"].nunique() == 663
assert (image_split_counts > 1).sum() == 0

print("\nPASS: Image IDs are unique across the three splits.")

In [ ]:

import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/twincheck"

SPLIT_DIR = os.path.join(
    PROJECT_DIR, "data", "processed", "vision", "splits"
)

FEATURE_DIR = os.path.join(
    PROJECT_DIR, "features", "vision", "verite", "siglip2"
)

CHECKPOINT_DIR = os.path.join(
    FEATURE_DIR, "checkpoints", "image_batches"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Load all splits
frames = []

for split in ["train", "validation", "test"]:
    path = os.path.join(SPLIT_DIR, f"{split}.csv")
    df = pd.read_csv(path)
    df["split"] = split
    frames.append(df)

all_records = pd.concat(frames, ignore_index=True)

# One row per unique image
image_df = (
    all_records[
        ["image_id", "resolved_image_path", "split"]
    ]
    .drop_duplicates(subset=["image_id"])
    .sort_values("image_id")
    .reset_index(drop=True)
)

# Verify that each image belongs to exactly one split
assert image_df["image_id"].nunique() == 663
assert image_df["resolved_image_path"].nunique() == 663
assert image_df["resolved_image_path"].apply(os.path.isfile).all()

# Save image manifest
image_manifest_path = os.path.join(
    FEATURE_DIR, "verite_unique_image_manifest.csv"
)

image_df.to_csv(image_manifest_path, index=False)

print("Unique images:", len(image_df))
print("Train images:", (image_df["split"] == "train").sum())
print("Validation images:", (image_df["split"] == "validation").sum())
print("Test images:", (image_df["split"] == "test").sum())
print("Missing images:", (~image_df["resolved_image_path"].apply(os.path.isfile)).sum())
print("Manifest saved:", image_manifest_path)


In [ ]:
print("Model output inspection")

print("Output type:", type(
    model.get_image_features(**processor(
        images=[
            Image.open(image_df.iloc[0]["resolved_image_path"]).convert("RGB")
        ],
        return_tensors="pt"
    ))
))

with torch.inference_mode():
    test_inputs = processor(
        images=[
            Image.open(image_df.iloc[0]["resolved_image_path"]).convert("RGB")
        ],
        return_tensors="pt"
    )

    test_output = model.get_image_features(**test_inputs)

print("\nOutput attributes:")
print(test_output)

print("\nAvailable attributes:")
print([x for x in dir(test_output) if not x.startswith("_")])

In [ ]:
# Test the SigLIP2 vision encoder directly

with torch.inference_mode():
    test_image = Image.open(
        image_df.iloc[0]["resolved_image_path"]
    ).convert("RGB")

    test_inputs = processor(
        images=[test_image],
        return_tensors="pt"
    )

    vision_outputs = model.vision_model(
        pixel_values=test_inputs["pixel_values"]
    )

    pooled_features = vision_outputs.pooler_output

print("Vision output type:", type(vision_outputs))
print("Pooled feature shape:", pooled_features.shape)
print("Feature data type:", pooled_features.dtype)
print("Contains NaN:", torch.isnan(pooled_features).any().item())
print("Contains infinity:", torch.isinf(pooled_features).any().item())

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

# Directories
FEATURE_DIR = os.path.join(
    "/content/drive/MyDrive/twincheck/features/vision/verite/siglip2"
)

CHECKPOINT_DIR = os.path.join(FEATURE_DIR, "checkpoints")

os.makedirs(FEATURE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Settings
BATCH_SIZE = 2
EXPECTED_DIM = 1152

# Confirm manifest
print("Images to process:", len(image_df))
print("Batch size:", BATCH_SIZE)

# Checkpoint file
checkpoint_path = os.path.join(
    CHECKPOINT_DIR, "verite_image_embeddings_checkpoint.npy"
)

# Load existing checkpoint if present
if os.path.exists(checkpoint_path):
    image_embeddings = np.load(checkpoint_path)
    start_idx = len(image_embeddings)
    print("Resuming from checkpoint:", start_idx)
else:
    image_embeddings = np.empty((0, EXPECTED_DIM), dtype=np.float32)
    start_idx = 0

# Process remaining images
for start in tqdm(range(start_idx, len(image_df), BATCH_SIZE)):

    batch_df = image_df.iloc[start:start + BATCH_SIZE]

    images = []

    for path in batch_df["resolved_image_path"]:
        img = Image.open(path).convert("RGB")
        images.append(img)

    inputs = processor(
        images=images,
        return_tensors="pt"
    )

    with torch.inference_mode():
        outputs = model.vision_model(
            pixel_values=inputs["pixel_values"]
        )

        batch_features = (
            outputs.pooler_output
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

    # Validate output
    assert batch_features.shape == (
        len(batch_df), EXPECTED_DIM
    ), f"Unexpected shape: {batch_features.shape}"

    assert np.isfinite(batch_features).all(), \
        "NaN or infinite values detected"

    image_embeddings = np.concatenate(
        [image_embeddings, batch_features],
        axis=0
    )

    # Save checkpoint after each batch
    np.save(checkpoint_path, image_embeddings)

    # Release batch memory
    del images, inputs, outputs, batch_features

# Save final embeddings
final_path = os.path.join(
    FEATURE_DIR, "verite_siglip2_image_embeddings.npy"
)

np.save(final_path, image_embeddings)

print("\nExtraction completed.")
print("Embedding shape:", image_embeddings.shape)
print("Saved to:", final_path)

In [ ]:
print("Image DataFrame columns:")
print(image_df.columns.tolist())

print("\nFirst 5 rows:")
display(image_df.head())

print("\nDataFrame shape:")
print(image_df.shape)

In [ ]:
import os
import pandas as pd

VERITE_DIR = "/content/drive/MyDrive/twincheck/data/processed/vision"

print("Files in VERITE processed directory:\n")

for root, dirs, files in os.walk(VERITE_DIR):
    for file in files:
        if file.endswith((".csv", ".jsonl")):
            print(os.path.join(root, file))

In [ ]:
verite_df = pd.read_csv(
    "/content/drive/MyDrive/twincheck/data/processed/vision/verite_cleaned.csv"
)

print("Dataset shape:", verite_df.shape)

print("\nColumn names:")
print(verite_df.columns.tolist())

print("\nFirst 5 records:")
display(verite_df.head())

print("\nMissing values:")
print(verite_df.isnull().sum())

In [ ]:

print("Model class:", type(model).__name__)
print("\nConfiguration class:", type(model.config).__name__)

print("\nConfiguration attributes:")
print(model.config.to_dict().keys())

print("\nModel attributes:")
print([name for name in dir(model) if "project" in name or "embed" in name])

print("\nVision configuration:")
print(model.config.vision_config)

print("\nText configuration:")
print(model.config.text_config)


In [ ]:
with torch.inference_mode():
    outputs = model(**inputs)

print("Output type:", type(outputs).__name__)
print("\nAvailable output fields:")
print(outputs.keys())

for name in ["image_embeds", "text_embeds"]:
    value = getattr(outputs, name, None)

    if value is not None:
        print(f"\n{name}:")
        print("Shape:", tuple(value.shape))
        print("Data type:", value.dtype)
        print("All values finite:", torch.isfinite(value).all().item())
        print("Sample values:", value[0, :5].tolist())
    else:
        print(f"\n{name}: Not available")

print("\nVision pooled output:")
with torch.inference_mode():
    vision_outputs = model.vision_model(
        pixel_values=inputs["pixel_values"]
    )

print("Shape:", tuple(vision_outputs.pooler_output.shape))

Test image-only projected embeddings

In [ ]:

# Test SigLIP2 projected image embeddings

with torch.inference_mode():
    test_image_features = model.get_image_features(
        pixel_values=image_inputs["pixel_values"]
    )

print("Output type:", type(test_image_features).__name__)

if hasattr(test_image_features, "pooler_output"):
    test_image_features = test_image_features.pooler_output

print("Embedding shape:", test_image_features.shape)
print(
    "All values finite:",
    torch.isfinite(test_image_features).all().item()
)


Test the text feature method

In [ ]:
# Inspect the text processor output

text_inputs = processor(
    text=[verite_df.iloc[0]["caption"]],
    padding="max_length",
    return_tensors="pt"
)

print("Available keys:", text_inputs.keys())

for key, value in text_inputs.items():
    print(key, value.shape, value.dtype)

In [ ]:
with torch.inference_mode():
    test_text_features = model.get_text_features(
        input_ids=text_inputs["input_ids"]
    )

if hasattr(test_text_features, "pooler_output"):
    test_text_features = test_text_features.pooler_output

print("Output type:", type(test_text_features).__name__)
print("Embedding shape:", test_text_features.shape)

print(
    "All values finite:",
    torch.isfinite(test_text_features).all().item()
)

In [ ]:

# Compare dedicated feature methods with joint model outputs

with torch.inference_mode():
    joint_outputs = model(**inputs)

    dedicated_image = model.get_image_features(
        pixel_values=inputs["pixel_values"]
    )

    dedicated_text = model.get_text_features(
        input_ids=inputs["input_ids"]
    )

# Extract pooled output if returned as a model output object
if hasattr(dedicated_image, "pooler_output"):
    dedicated_image = dedicated_image.pooler_output

if hasattr(dedicated_text, "pooler_output"):
    dedicated_text = dedicated_text.pooler_output

image_difference = torch.max(
    torch.abs(joint_outputs.image_embeds - dedicated_image)
).item()

text_difference = torch.max(
    torch.abs(joint_outputs.text_embeds - dedicated_text)
).item()

print("Maximum image embedding difference:", image_difference)
print("Maximum text embedding difference:", text_difference)

print(
    "Image embeddings match:",
    torch.allclose(
        joint_outputs.image_embeds,
        dedicated_image,
        atol=1e-5
    )
)

print(
    "Text embeddings match:",
    torch.allclose(
        joint_outputs.text_embeds,
        dedicated_text,
        atol=1e-5
    )
)


In [ ]:
import inspect

print("TRANSFORMERS VERSION")
import transformers
print(transformers.__version__)

print("\nMODEL COMPONENTS")
print(model)

print("\nPROJECTION-RELATED PARAMETERS")
for name, parameter in model.named_parameters():
    if any(word in name.lower() for word in ["projection", "head"]):
        print(name, tuple(parameter.shape))

print("\nGET_IMAGE_FEATURES IMPLEMENTATION")
print(inspect.getsource(model.get_image_features))

print("\nGET_TEXT_FEATURES IMPLEMENTATION")
print(inspect.getsource(model.get_text_features))

In [ ]:

# Verify projected image and text extraction separately
# using the complete SigLIP2 model.

with torch.inference_mode():

    # Reference: image and text together
    reference = model(**inputs)

    # Image extraction: real image + dummy text
    image_test = model(
        pixel_values=inputs["pixel_values"],
        input_ids=inputs["input_ids"]
    )

    # Text extraction: dummy image + real text
    text_test = model(
        pixel_values=inputs["pixel_values"],
        input_ids=inputs["input_ids"]
    )

print("Image reference shape:", reference.image_embeds.shape)
print("Text reference shape:", reference.text_embeds.shape)

print(
    "Image outputs match:",
    torch.allclose(
        reference.image_embeds,
        image_test.image_embeds,
        atol=1e-5
    )
)

print(
    "Text outputs match:",
    torch.allclose(
        reference.text_embeds,
        text_test.text_embeds,
        atol=1e-5
    )
)


#  Recovery Cell

In [ ]:

# ============================================================
# TWINCHECK — F2 RUNTIME RECOVERY
# Restore project variables, datasets, manifest and SigLIP2
# ============================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd
import torch

from google.colab import drive
from PIL import Image
from transformers import AutoProcessor, AutoModel

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive", force_remount=True)

# ------------------------------------------------------------
# 2. Restore project directory paths
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/twincheck"

VERITE_DIR = os.path.join(
    PROJECT_DIR, "data", "processed", "vision"
)

FEATURE_DIR = os.path.join(
    PROJECT_DIR, "features", "vision", "verite", "siglip2"
)

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR, "checkpoints"
)

# Create directories if they do not already exist
os.makedirs(FEATURE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 3. Restore dataset and image manifest
# ------------------------------------------------------------

VERITE_CSV = os.path.join(
    VERITE_DIR, "verite_cleaned.csv"
)

IMAGE_MANIFEST = os.path.join(
    FEATURE_DIR, "verite_unique_image_manifest.csv"
)

if not os.path.isfile(VERITE_CSV):
    raise FileNotFoundError(f"VERITE CSV not found: {VERITE_CSV}")

if not os.path.isfile(IMAGE_MANIFEST):
    raise FileNotFoundError(
        f"Image manifest not found: {IMAGE_MANIFEST}"
    )

verite_df = pd.read_csv(VERITE_CSV)
image_df = pd.read_csv(IMAGE_MANIFEST)

# ------------------------------------------------------------
# 4. Restore references to previous checkpoints
#    These are preserved; do not assume their row order is valid.
# ------------------------------------------------------------

OLD_PROJECTED_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "verite_image_projected_checkpoint.npy"
)

OLD_POOLED_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "verite_image_embeddings_checkpoint.npy"
)

OLD_POOLED_FINAL = os.path.join(
    FEATURE_DIR,
    "verite_siglip2_image_embeddings.npy"
)

# Load old checkpoints only for inspection, not for the new
# verified extraction.
old_projected_checkpoint = (
    np.load(OLD_PROJECTED_CHECKPOINT)
    if os.path.isfile(OLD_PROJECTED_CHECKPOINT)
    else None
)

old_pooled_checkpoint = (
    np.load(OLD_POOLED_CHECKPOINT)
    if os.path.isfile(OLD_POOLED_CHECKPOINT)
    else None
)

# ------------------------------------------------------------
# 5. Restore verified extraction output paths
# ------------------------------------------------------------

VERIFIED_DIR = os.path.join(
    FEATURE_DIR, "verified_image_extraction"
)

os.makedirs(VERIFIED_DIR, exist_ok=True)

VERIFIED_CHECKPOINT = os.path.join(
    VERIFIED_DIR,
    "verite_siglip2_image_projected_verified_checkpoint.npz"
)

VERIFIED_MANIFEST = os.path.join(
    VERIFIED_DIR,
    "verite_siglip2_image_projected_verified_manifest.csv"
)

VERIFIED_CONFIG = os.path.join(
    VERIFIED_DIR,
    "extraction_config.json"
)

# ------------------------------------------------------------
# 6. Validate manifest and image files
# ------------------------------------------------------------

required_columns = {
    "image_id",
    "resolved_image_path",
    "split"
}

missing_columns = required_columns - set(image_df.columns)

if missing_columns:
    raise ValueError(
        f"Manifest is missing columns: {missing_columns}"
    )

missing_images = [
    path for path in image_df["resolved_image_path"]
    if not os.path.isfile(path)
]

if len(image_df) != 663:
    raise ValueError(
        f"Expected 663 manifest rows, found {len(image_df)}"
    )

if image_df["image_id"].nunique() != 663:
    raise ValueError("Image IDs are not unique.")

if image_df["resolved_image_path"].nunique() != 663:
    raise ValueError("Image paths are not unique.")

if missing_images:
    raise FileNotFoundError(
        f"{len(missing_images)} image files are missing. "
        f"First missing paths: {missing_images[:5]}"
    )

# Fingerprint the exact ordered manifest
manifest_hash = hashlib.sha256(
    image_df.to_csv(index=False).encode("utf-8")
).hexdigest()

# ------------------------------------------------------------
# 7. Restore SigLIP2 model and processor
# ------------------------------------------------------------

MODEL_ID = "google/siglip2-so400m-patch14-384"

device = torch.device("cpu")

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
)

model.to(device)
model.eval()

# ------------------------------------------------------------
# 8. Display recovery summary
# ------------------------------------------------------------

print("\n========== RUNTIME RECOVERY COMPLETE ==========")

print("\nProject directory:", PROJECT_DIR)

print("\nVERITE records:", len(verite_df))
print("Unique images:", len(image_df))

print("\nImage split counts:")
print(image_df["split"].value_counts())

print("\nMissing image files:", len(missing_images))
print("Manifest SHA-256:", manifest_hash)

print("\nModel:", MODEL_ID)
print("Model class:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("Vision hidden size:", model.config.vision_config.hidden_size)
print("Text hidden size:", model.config.text_config.hidden_size)

print("\nPrevious projected checkpoint:",
      None if old_projected_checkpoint is None
      else old_projected_checkpoint.shape)

print("Previous pooled checkpoint:",
      None if old_pooled_checkpoint is None
      else old_pooled_checkpoint.shape)

print("\nVerified checkpoint already exists:",
      os.path.isfile(VERIFIED_CHECKPOINT))

print("Verified manifest already exists:",
      os.path.isfile(VERIFIED_MANIFEST))

print("\nVerified extraction directory:", VERIFIED_DIR)

print("\n========== READY FOR NEXT STEP ==========")


In [ ]:

# ============================================================
# TEST: SIGLIP2 PROJECTED IMAGE EMBEDDING EXTRACTION
# Only two images — no full extraction yet
# ============================================================

import torch
import numpy as np
from PIL import Image

# Select the first two images from the verified manifest
test_rows = image_df.iloc[:2].copy()

# Load images as RGB
test_images = [
    Image.open(path).convert("RGB")
    for path in test_rows["resolved_image_path"]
]

# A fixed dummy caption is required by this model's forward API.
# Earlier tests confirmed that changing the caption does not
# change the image embeddings.
dummy_texts = ["An image."] * len(test_images)

# Preprocess images and dummy texts
test_inputs = processor(
    images=test_images,
    text=dummy_texts,
    return_tensors="pt",
    padding=True
)

# Move inputs to CPU
test_inputs = {
    key: value.to(device)
    for key, value in test_inputs.items()
}

# Extract projected image embeddings
with torch.no_grad():
    test_outputs = model(**test_inputs)
    test_embeddings = test_outputs.image_embeds

# Convert to NumPy
test_embeddings = test_embeddings.cpu().numpy()

# ------------------------------------------------------------
# Validate output
# ------------------------------------------------------------

print("Image IDs:")
print(test_rows["image_id"].tolist())

print("\nEmbedding shape:", test_embeddings.shape)
print("Expected shape:", (2, 1152))

print("\nAll values finite:",
      np.isfinite(test_embeddings).all())

print("Embedding norms:",
      np.linalg.norm(test_embeddings, axis=1))

print("\nFirst 10 values of first embedding:")
print(test_embeddings[0][:10])

assert test_embeddings.shape == (2, 1152)
assert np.isfinite(test_embeddings).all()

print("\nSUCCESS: Two-image projected embedding test passed.")


In [ ]:

import torch
import subprocess

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("\nGPU name:", torch.cuda.get_device_name(0))

    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3), 2
        ),
        "GB"
    )

    print(
        "Currently allocated:",
        round(torch.cuda.memory_allocated(0) / (1024 ** 3), 2),
        "GB"
    )

    print(
        "Currently reserved:",
        round(torch.cuda.memory_reserved(0) / (1024 ** 3), 2),
        "GB"
    )
else:
    print("\nNo CUDA GPU detected.")


In [ ]:

# Move SigLIP2 model to NVIDIA L4 GPU

import torch

device = torch.device("cuda:0")

model = model.to(device)
model.eval()

print("Model device:", next(model.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated(0) / (1024**3), 2),
    "GB"
)

assert next(model.parameters()).is_cuda

print("\nSUCCESS: SigLIP2 is now running on GPU.")


In [ ]:

import time
import torch
import numpy as np
from PIL import Image

# Use the same eight images for each benchmark
benchmark_rows = image_df.iloc[:8]

benchmark_images = [
    Image.open(path).convert("RGB")
    for path in benchmark_rows["resolved_image_path"]
]

batch_sizes = [8, 16, 32]
benchmark_results = []

for batch_size in batch_sizes:

    # Clear previous temporary GPU allocations
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    images = benchmark_images[:batch_size]
    texts = ["An image."] * len(images)

    inputs = processor(
        images=images,
        text=texts,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Warm-up
    with torch.no_grad():
        _ = model(**inputs)

    torch.cuda.synchronize()
    start_time = time.time()

    # Timed inference
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.image_embeds

    torch.cuda.synchronize()
    elapsed = time.time() - start_time

    peak_memory = (
        torch.cuda.max_memory_allocated() / (1024 ** 3)
    )

    benchmark_results.append({
        "batch_size": batch_size,
        "images_processed": len(images),
        "time_seconds": round(elapsed, 2),
        "seconds_per_image": round(
            elapsed / len(images), 2
        ),
        "peak_gpu_memory_GB": round(peak_memory, 2),
        "embedding_shape": tuple(embeddings.shape),
        "finite_values": bool(
            torch.isfinite(embeddings).all().item()
        )
    })

    print(f"\nCompleted batch size {batch_size}")
    print("Time:", round(elapsed, 2), "seconds")
    print("Peak GPU memory:", round(peak_memory, 2), "GB")
    print("Embedding shape:", tuple(embeddings.shape))

print("\n========== BENCHMARK SUMMARY ==========")
display(pd.DataFrame(benchmark_results))


In [ ]:

import time
import torch
import pandas as pd
import numpy as np
from PIL import Image

# Load 32 different images
benchmark_rows = image_df.iloc[:32]

benchmark_images = [
    Image.open(path).convert("RGB")
    for path in benchmark_rows["resolved_image_path"]
]

batch_sizes = [8, 16, 32]
benchmark_results = []

for batch_size in batch_sizes:

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    images = benchmark_images[:batch_size]
    texts = ["An image."] * batch_size

    inputs = processor(
        images=images,
        text=texts,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Warm-up
    with torch.no_grad():
        _ = model(**inputs)

    torch.cuda.synchronize()
    start = time.time()

    # Timed inference
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.image_embeds

    torch.cuda.synchronize()
    elapsed = time.time() - start

    peak_memory = (
        torch.cuda.max_memory_allocated() / (1024 ** 3)
    )

    benchmark_results.append({
        "batch_size": batch_size,
        "images_processed": len(images),
        "time_seconds": round(elapsed, 3),
        "seconds_per_image": round(
            elapsed / len(images), 4
        ),
        "images_per_second": round(
            len(images) / elapsed, 2
        ),
        "peak_gpu_memory_GB": round(peak_memory, 2),
        "embedding_shape": str(tuple(embeddings.shape)),
        "finite_values": bool(
            torch.isfinite(embeddings).all().item()
        )
    })

print("========== PROPER GPU BENCHMARK ==========")
display(pd.DataFrame(benchmark_results))


In [ ]:



print("F2-related variables currently defined:\n")

for name, value in sorted(globals().items()):
    if any(word in name.upper() for word in
           ["VERIFIED", "BATCH", "MANIFEST", "CONFIG", "CHECKPOINT"]):
        if not name.startswith("_"):
            print(f"{name} = {value}")

print("\nExisting verified directory:")
print(VERIFIED_DIR)

import os

if os.path.isdir(VERIFIED_DIR):
    print("\nContents of verified directory:")
    print(os.listdir(VERIFIED_DIR))


In [ ]:

# ============================================================
# F2: INITIALISE VERIFIED SIGLIP2 EXTRACTION
# ============================================================

import os
import json
import hashlib
import pandas as pd

# Use the existing verified directory
BATCH_DIR = os.path.join(
    VERIFIED_DIR,
    "image_embedding_batches"
)

os.makedirs(BATCH_DIR, exist_ok=True)

# Confirm the selected batch size
BATCH_SIZE = 32

# Recalculate the ordered manifest fingerprint
manifest_hash = hashlib.sha256(
    image_df.to_csv(index=False).encode("utf-8")
).hexdigest()

# Save a copy of the exact manifest
image_df.to_csv(
    VERIFIED_MANIFEST,
    index=False
)

# Save extraction configuration
extraction_config = {
    "model_id": MODEL_ID,
    "embedding_type": "projected_image_embeddings",
    "embedding_dimension": 1152,
    "batch_size": BATCH_SIZE,
    "total_images": len(image_df),
    "manifest_sha256": manifest_hash,
    "device": str(device)
}

with open(VERIFIED_CONFIG, "w") as f:
    json.dump(extraction_config, f, indent=4)

# Verify files
print("Preparation completed.")

print("\nVerified directory:", VERIFIED_DIR)
print("Batch directory:", BATCH_DIR)

print("\nManifest exists:",
      os.path.isfile(VERIFIED_MANIFEST))

print("Configuration exists:",
      os.path.isfile(VERIFIED_CONFIG))

print("Batch size:", BATCH_SIZE)
print("Total images:", len(image_df))
print("Expected batches:",
      (len(image_df) + BATCH_SIZE - 1) // BATCH_SIZE)

print("Manifest fingerprint:", manifest_hash)


In [ ]:

# ============================================================
# F2: FIXED 10-IMAGE EXTRACTION AND CHECKPOINT TEST
# ============================================================

import os
import numpy as np
import torch
from PIL import Image

TEST_BATCH_FILE = os.path.join(
    BATCH_DIR,
    "batch_000_test.npz"
)

# Select 10 images
test_rows = image_df.iloc[:10].copy()

test_images = [
    Image.open(path).convert("RGB")
    for path in test_rows["resolved_image_path"]
]

test_texts = ["An image."] * len(test_images)

# Preprocess
inputs = processor(
    images=test_images,
    text=test_texts,
    return_tensors="pt",
    padding=True
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# Extract projected embeddings
with torch.no_grad():
    outputs = model(**inputs)
    embeddings = outputs.image_embeds.cpu().numpy()

# Convert metadata to Unicode arrays
image_ids = np.asarray(
    test_rows["image_id"].tolist(),
    dtype=str
)

image_paths = np.asarray(
    test_rows["resolved_image_path"].tolist(),
    dtype=str
)

splits = np.asarray(
    test_rows["split"].tolist(),
    dtype=str
)

# Validate before saving
assert embeddings.shape == (10, 1152)
assert np.isfinite(embeddings).all()

# Save checkpoint
np.savez_compressed(
    TEST_BATCH_FILE,
    embeddings=embeddings,
    image_ids=image_ids,
    image_paths=image_paths,
    splits=splits,
    batch_index=np.array([-1], dtype=np.int64),
    manifest_sha256=np.asarray([manifest_hash], dtype=str)
)

# Reload safely (no pickle required)
with np.load(TEST_BATCH_FILE, allow_pickle=False) as saved:

    assert saved["embeddings"].shape == (10, 1152)
    assert np.isfinite(saved["embeddings"]).all()

    assert np.array_equal(saved["image_ids"], image_ids)
    assert np.array_equal(saved["image_paths"], image_paths)
    assert np.array_equal(saved["splits"], splits)

    print("Embedding shape:", saved["embeddings"].shape)
    print("All embedding values finite:",
          np.isfinite(saved["embeddings"]).all())
    print("Image IDs preserved:",
          np.array_equal(saved["image_ids"], image_ids))
    print("Image paths preserved:",
          np.array_equal(saved["image_paths"], image_paths))
    print("Splits preserved:",
          np.array_equal(saved["splits"], splits))

print("\nCheckpoint exists:", os.path.isfile(TEST_BATCH_FILE))
print("Checkpoint size (MB):",
      round(os.path.getsize(TEST_BATCH_FILE) / (1024**2), 3))

print("\nSUCCESS: 10-image extraction and checkpoint test passed.")


In [ ]:

# ============================================================
# TWINCHECK F2: FULL VERITE SIGLIP2 IMAGE EXTRACTION
#
# Input:
#   image_df: 663-image manifest
#   model: loaded SigLIP2 model
#   processor: SigLIP2 processor
#
# Output:
#   21 individual NPZ batch checkpoints
#   Each contains embeddings and matching metadata.
#
# Resume:
#   Completed and validated batches are skipped automatically.
# ============================================================

import os
import time
import numpy as np
import torch
from PIL import Image

# ------------------------------------------------------------
# 1. Configuration and safety checks
# ------------------------------------------------------------

BATCH_SIZE = 32
TOTAL_IMAGES = len(image_df)

assert TOTAL_IMAGES == 663
assert torch.cuda.is_available()
assert next(model.parameters()).is_cuda

os.makedirs(BATCH_DIR, exist_ok=True)

# Ensure the current manifest matches the saved configuration
assert manifest_hash == extraction_config["manifest_sha256"]

print("GPU:", torch.cuda.get_device_name(0))
print("Total images:", TOTAL_IMAGES)
print("Batch size:", BATCH_SIZE)

# ------------------------------------------------------------
# 2. Validate any existing completed batch
# ------------------------------------------------------------

def validate_batch(batch_path, expected_rows):
    """Check that a saved batch matches its expected manifest rows."""

    with np.load(batch_path, allow_pickle=False) as saved:
        assert saved["embeddings"].shape == (
            len(expected_rows), 1152
        )
        assert np.isfinite(saved["embeddings"]).all()

        assert np.array_equal(
            saved["image_ids"],
            expected_rows["image_id"].to_numpy(dtype=str)
        )

        assert np.array_equal(
            saved["image_paths"],
            expected_rows["resolved_image_path"].to_numpy(dtype=str)
        )

        assert np.array_equal(
            saved["splits"],
            expected_rows["split"].to_numpy(dtype=str)
        )

        assert saved["manifest_sha256"][0] == manifest_hash

    return True

# ------------------------------------------------------------
# 3. Extract batches
# ------------------------------------------------------------

start_total = time.time()

completed_batches = 0
skipped_batches = 0

number_of_batches = (
    TOTAL_IMAGES + BATCH_SIZE - 1
) // BATCH_SIZE

for batch_index in range(number_of_batches):

    start_index = batch_index * BATCH_SIZE
    end_index = min(start_index + BATCH_SIZE, TOTAL_IMAGES)

    batch_rows = image_df.iloc[start_index:end_index].copy()

    batch_path = os.path.join(
        BATCH_DIR,
        f"batch_{batch_index:03d}.npz"
    )

    # Skip only if existing checkpoint passes validation
    if os.path.isfile(batch_path):
        try:
            validate_batch(batch_path, batch_rows)
            skipped_batches += 1

            print(
                f"Batch {batch_index + 1}/{number_of_batches}: "
                "already completed and validated."
            )
            continue

        except Exception as error:
            raise RuntimeError(
                f"Existing batch {batch_index} failed validation. "
                "Stop and inspect it before proceeding."
            ) from error

    batch_start = time.time()

    images = []
    for path in batch_rows["resolved_image_path"]:
        with Image.open(path) as image:
            images.append(image.convert("RGB"))

    texts = ["An image."] * len(images)

    inputs = processor(
        images=images,
        text=texts,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model(**inputs)
        embeddings = outputs.image_embeds.cpu().numpy()

    # Prepare safe Unicode metadata
    image_ids = batch_rows["image_id"].to_numpy(dtype=str)
    image_paths = batch_rows["resolved_image_path"].to_numpy(dtype=str)
    splits = batch_rows["split"].to_numpy(dtype=str)

    # Validate before saving
    assert embeddings.shape == (len(batch_rows), 1152)
    assert np.isfinite(embeddings).all()

    # Save to temporary file first, then rename
    temp_path = batch_path + ".tmp.npz"

    np.savez_compressed(
        temp_path,
        embeddings=embeddings,
        image_ids=image_ids,
        image_paths=image_paths,
        splits=splits,
        batch_index=np.array([batch_index], dtype=np.int64),
        manifest_sha256=np.asarray([manifest_hash], dtype=str)
    )

    os.replace(temp_path, batch_path)

    # Reload and validate saved checkpoint
    validate_batch(batch_path, batch_rows)

    completed_batches += 1

    elapsed = time.time() - batch_start

    print(
        f"Batch {batch_index + 1}/{number_of_batches} "
        f"| Images: {len(batch_rows)} "
        f"| Time: {elapsed:.2f}s "
        f"| Saved and validated"
    )

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

total_time = time.time() - start_total

print("\n========== EXTRACTION SUMMARY ==========")
print("Total images:", TOTAL_IMAGES)
print("Total batches:", number_of_batches)
print("Newly completed batches:", completed_batches)
print("Previously completed batches skipped:", skipped_batches)
print("Elapsed time (minutes):", round(total_time / 60, 2))

print("\nBatch files:")
print(sorted([
    f for f in os.listdir(BATCH_DIR)
    if f.endswith(".npz") and f.startswith("batch_")
]))

print("\nExtraction loop completed.")
print("Next step: combine and validate all 663 embeddings.")


In [ ]:

import os
import numpy as np
import pandas as pd

expected_total = len(image_df)
expected_batches = 21
batch_size = 32
embedding_dim = 1152

all_embeddings = []
all_ids = []
all_paths = []
all_splits = []

for i in range(expected_batches):
    batch_path = os.path.join(BATCH_DIR, f"batch_{i:03d}.npz")

    assert os.path.isfile(batch_path), f"Missing batch: {batch_path}"

    batch = np.load(batch_path, allow_pickle=False)

    start = i * batch_size
    end = min(start + batch_size, expected_total)
    expected_rows = image_df.iloc[start:end]

    embeddings = batch["embeddings"]
    ids = batch["image_ids"].astype(str)
    paths = batch["image_paths"].astype(str)
    splits = batch["splits"].astype(str)

    assert embeddings.shape == (len(expected_rows), embedding_dim), \
        f"Incorrect shape in batch {i}"

    assert np.isfinite(embeddings).all(), \
        f"Non-finite values in batch {i}"

    assert np.array_equal(ids, expected_rows["image_id"].astype(str).to_numpy())
    assert np.array_equal(paths, expected_rows["resolved_image_path"].astype(str).to_numpy())
    assert np.array_equal(splits, expected_rows["split"].astype(str).to_numpy())

    norms = np.linalg.norm(embeddings, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-4), \
        f"Unexpected embedding norms in batch {i}"

    all_embeddings.append(embeddings)
    all_ids.extend(ids.tolist())
    all_paths.extend(paths.tolist())
    all_splits.extend(splits.tolist())

    batch.close()

combined_embeddings = np.concatenate(all_embeddings, axis=0)

assert combined_embeddings.shape == (expected_total, embedding_dim)
assert len(set(all_ids)) == expected_total
assert np.isfinite(combined_embeddings).all()

print("ALL BATCHES VALIDATED")
print("Combined shape:", combined_embeddings.shape)
print("Total image IDs:", len(all_ids))
print("Unique image IDs:", len(set(all_ids)))
print("All values finite:", np.isfinite(combined_embeddings).all())
print("All embeddings normalized:",
      np.allclose(np.linalg.norm(combined_embeddings, axis=1), 1.0, atol=1e-4))
print("Split counts:")
print(pd.Series(all_splits).value_counts())


In [ ]:

import os
import numpy as np
import pandas as pd

FINAL_EMBEDDINGS = os.path.join(
    VERIFIED_DIR,
    "verite_siglip2_image_projected_embeddings.npy"
)

FINAL_MANIFEST = os.path.join(
    VERIFIED_DIR,
    "verite_siglip2_image_projected_embeddings_manifest.csv"
)

# Save the combined embeddings
np.save(FINAL_EMBEDDINGS, combined_embeddings)

# Save corresponding metadata in exactly the same order
final_manifest = pd.DataFrame({
    "image_id": all_ids,
    "resolved_image_path": all_paths,
    "split": all_splits
})

final_manifest.to_csv(FINAL_MANIFEST, index=False)

# Reload both files
reloaded_embeddings = np.load(FINAL_EMBEDDINGS)
reloaded_manifest = pd.read_csv(FINAL_MANIFEST)

# Verify saved files
assert reloaded_embeddings.shape == (663, 1152)
assert len(reloaded_manifest) == 663
assert np.array_equal(reloaded_embeddings, combined_embeddings)
assert reloaded_manifest["image_id"].astype(str).tolist() == all_ids
assert reloaded_manifest["resolved_image_path"].astype(str).tolist() == all_paths
assert reloaded_manifest["split"].astype(str).tolist() == all_splits

print("FINAL FILES SAVED AND RELOADED SUCCESSFULLY")
print("Embedding file:", FINAL_EMBEDDINGS)
print("Manifest file:", FINAL_MANIFEST)
print("Embedding shape:", reloaded_embeddings.shape)
print("Manifest rows:", len(reloaded_manifest))
print("Exact embedding reload:", np.array_equal(
    reloaded_embeddings, combined_embeddings
))
print("All metadata mappings preserved:", True)
print("Embedding file size (MB):",
      round(os.path.getsize(FINAL_EMBEDDINGS) / (1024**2), 2))


VERITE SigLIP2 caption embeddings.

In [ ]:

print("VERITE dataset columns:")
print(verite_df.columns.tolist())

print("\nDataset shape:", verite_df.shape)

print("\nFirst 5 records:")
display(verite_df.head())

print("\nCaption missing values:")
print(verite_df["caption"].isna().sum())

print("\nUnique resolved image paths:",
      verite_df["resolved_image_path"].nunique())


Create the caption-to-image mapping

In [ ]:

# Create a caption-level manifest by joining VERITE records
# with the verified unique-image manifest.

caption_df = verite_df.merge(
    image_df[["image_id", "resolved_image_path", "split"]],
    on="resolved_image_path",
    how="left",
    validate="many_to_one"
)

# Assign a stable ID to every caption record
caption_df.insert(
    0,
    "caption_id",
    [f"verite_caption_{i:04d}" for i in range(len(caption_df))]
)

# Validate the mapping
assert len(caption_df) == 1001
assert caption_df["caption"].notna().all()
assert caption_df["image_id"].notna().all()
assert caption_df["split"].notna().all()
assert caption_df["caption_id"].is_unique

print("CAPTION MAPPING VALIDATION")
print("Total caption records:", len(caption_df))
print("Unique caption IDs:", caption_df["caption_id"].nunique())
print("Unique image IDs:", caption_df["image_id"].nunique())
print("Missing image mappings:", caption_df["image_id"].isna().sum())
print("\nSplit counts:")
print(caption_df["split"].value_counts())
print("\nLabel counts:")
print(caption_df["label"].value_counts())

print("\nSample mapped records:")
display(
    caption_df[
        ["caption_id", "caption", "image_id", "split", "label"]
    ].head()
)


In [ ]:

import os
import pandas as pd

CAPTION_DIR = os.path.join(
    FEATURE_DIR,
    "verified_caption_extraction"
)
os.makedirs(CAPTION_DIR, exist_ok=True)

CAPTION_MANIFEST = os.path.join(
    CAPTION_DIR,
    "verite_siglip2_caption_manifest.csv"
)

# Save the caption-to-image mapping
caption_df.to_csv(CAPTION_MANIFEST, index=False)

# Reload and verify
saved_caption_df = pd.read_csv(CAPTION_MANIFEST)

assert len(saved_caption_df) == 1001
assert saved_caption_df["caption_id"].is_unique
assert saved_caption_df["image_id"].notna().all()
assert saved_caption_df["split"].notna().all()
assert saved_caption_df["caption"].notna().all()

print("CAPTION MANIFEST SAVED AND VERIFIED")
print("File:", CAPTION_MANIFEST)
print("Rows:", len(saved_caption_df))
print("Unique caption IDs:", saved_caption_df["caption_id"].nunique())
print("Unique image IDs:", saved_caption_df["image_id"].nunique())
print("Missing mappings:", saved_caption_df["image_id"].isna().sum())
print("Split counts:")
print(saved_caption_df["split"].value_counts())


In [ ]:

import inspect

print("Model class:", type(model).__name__)

print("\nText model available:", hasattr(model, "text_model"))
print("Text projection available:", hasattr(model, "text_projection"))
print("get_text_features available:", hasattr(model, "get_text_features"))

if hasattr(model, "get_text_features"):
    print("\nget_text_features signature:")
    print(inspect.signature(model.get_text_features))

if hasattr(model, "text_model"):
    print("\nText model forward signature:")
    print(inspect.signature(model.text_model.forward))

if hasattr(model, "text_projection"):
    print("\nText projection module:")
    print(model.text_projection)


In [ ]:

import inspect

print("GET_TEXT_FEATURES IMPLEMENTATION:\n")
print(inspect.getsource(type(model).get_text_features))

print("\nTEXT MODEL OUTPUT CONFIGURATION:\n")
print("Text hidden size:", model.config.text_config.hidden_size)
print("Projection dimension:",
      getattr(model.config, "projection_size", "Not available"))

print("\nTEXT MODEL CONFIGURATION:\n")
print(model.config.text_config)


In [ ]:

import inspect

print("MODEL FORWARD IMPLEMENTATION:\n")
print(inspect.getsource(type(model).forward))

print("\nMODEL COMPONENTS:\n")
for name, module in model.named_children():
    print(name, ":", type(module).__name__)

print("\nTEXT MODEL COMPONENTS:\n")
for name, module in model.text_model.named_children():
    print(name, ":", type(module).__name__)


In [ ]:

import torch
import numpy as np

# Select 10 captions
test_captions = saved_caption_df["caption"].head(10).tolist()

# Tokenize captions
text_inputs = processor(
    text=test_captions,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

# Move tensors to GPU
text_inputs = {
    key: value.to(device)
    for key, value in text_inputs.items()
}

model.eval()

with torch.inference_mode():
    # Extract text model output
    text_outputs = model.get_text_features(**text_inputs)

    # Retrieve pooled text features
    raw_text_features = text_outputs.pooler_output

    # Apply the same L2 normalization as model.forward()
    test_text_embeddings = torch.nn.functional.normalize(
        raw_text_features,
        p=2,
        dim=-1
    ).cpu().numpy()

# Validate
assert test_text_embeddings.shape == (10, 1152)
assert np.isfinite(test_text_embeddings).all()

test_norms = np.linalg.norm(test_text_embeddings, axis=1)

print("CORRECTED SIGLIP2 TEXT EXTRACTION TEST")
print("Embedding shape:", test_text_embeddings.shape)
print("All values finite:", np.isfinite(test_text_embeddings).all())
print("Embedding norms:", np.round(test_norms, 4))
print(
    "All embeddings normalized:",
    np.allclose(test_norms, 1.0, atol=1e-4)
)
print("Device used:", device)


In [ ]:

import os
import time
import numpy as np
import torch

# Confirm the previous extraction setup is still available
assert "caption_df" in globals()
assert "CAPTION_BATCH_DIR" in globals()
assert "manifest_hash" in globals()
assert "expected_batches" in globals()
assert "validate_caption_batch" in globals()
assert "model" in globals()
assert "processor" in globals()

# Remove only an incomplete temporary checkpoint from the failed batch
for filename in os.listdir(CAPTION_BATCH_DIR):
    if filename.endswith(".tmp.npz"):
        os.remove(os.path.join(CAPTION_BATCH_DIR, filename))
        print("Removed incomplete temporary file:", filename)

# Replace validator with a version that checks Unicode string arrays
def validate_caption_batch(batch_path, expected_rows):
    with np.load(batch_path, allow_pickle=False) as batch:
        embeddings = batch["embeddings"]
        caption_ids = batch["caption_ids"].astype(str)
        image_ids = batch["image_ids"].astype(str)
        splits = batch["splits"].astype(str)
        saved_hash = str(batch["manifest_sha256"].item())

    assert embeddings.shape == (len(expected_rows), 1152)
    assert np.isfinite(embeddings).all()

    norms = np.linalg.norm(embeddings, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-4)

    assert np.array_equal(
        caption_ids,
        expected_rows["caption_id"].astype(str).to_numpy()
    )
    assert np.array_equal(
        image_ids,
        expected_rows["image_id"].astype(str).to_numpy()
    )
    assert np.array_equal(
        splits,
        expected_rows["split"].astype(str).to_numpy()
    )
    assert saved_hash == manifest_hash

    return len(embeddings)

# Run extraction, reusing valid existing batches
model.eval()
newly_completed = 0
previously_completed = 0
start_time = time.time()

for batch_index in range(expected_batches):
    start = batch_index * 32
    end = min(start + 32, len(caption_df))
    batch_rows = caption_df.iloc[start:end].reset_index(drop=True)

    batch_path = os.path.join(
        CAPTION_BATCH_DIR,
        f"caption_batch_{batch_index:03d}.npz"
    )
    temp_path = batch_path + ".tmp.npz"

    if os.path.isfile(batch_path):
        validate_caption_batch(batch_path, batch_rows)
        previously_completed += 1
        print(f"Batch {batch_index + 1}/{expected_batches}: validated, skipped")
        continue

    batch_start = time.time()

    text_inputs = processor(
        text=batch_rows["caption"].astype(str).tolist(),
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    text_inputs = {
        key: value.to(device)
        for key, value in text_inputs.items()
    }

    with torch.inference_mode():
        text_outputs = model.get_text_features(**text_inputs)
        embeddings = torch.nn.functional.normalize(
            text_outputs.pooler_output,
            p=2,
            dim=-1
        ).float().cpu().numpy().astype(np.float32)

    # Explicitly save Unicode arrays, not object arrays
    caption_ids = batch_rows["caption_id"].astype(str).to_numpy(dtype=str)
    image_ids = batch_rows["image_id"].astype(str).to_numpy(dtype=str)
    splits = batch_rows["split"].astype(str).to_numpy(dtype=str)

    np.savez_compressed(
        temp_path,
        embeddings=embeddings,
        caption_ids=caption_ids,
        image_ids=image_ids,
        splits=splits,
        manifest_sha256=np.array(manifest_hash)
    )

    validate_caption_batch(temp_path, batch_rows)
    os.replace(temp_path, batch_path)
    validate_caption_batch(batch_path, batch_rows)

    newly_completed += 1
    print(
        f"Batch {batch_index + 1}/{expected_batches}: "
        f"{len(batch_rows)} captions saved and validated "
        f"({time.time() - batch_start:.2f}s)"
    )

print("\n========== EXTRACTION SUMMARY ==========")
print("Total captions:", len(caption_df))
print("Expected batches:", expected_batches)
print("Newly completed:", newly_completed)
print("Previously completed and skipped:", previously_completed)
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("Output directory:", CAPTION_BATCH_DIR)
print("All extraction batches processed.")


In [ ]:

import os
import numpy as np
import pandas as pd
import hashlib

# --------------------------------------------------
# 1. Define final output paths
# --------------------------------------------------

FINAL_CAPTION_EMBEDDINGS = os.path.join(
    CAPTION_DIR,
    "verite_siglip2_caption_projected_embeddings.npy"
)

FINAL_CAPTION_MANIFEST = os.path.join(
    CAPTION_DIR,
    "verite_siglip2_caption_projected_embeddings_manifest.csv"
)

# --------------------------------------------------
# 2. Load and validate every batch in order
# --------------------------------------------------

embedding_batches = []

for batch_index in range(expected_batches):

    batch_path = os.path.join(
        CAPTION_BATCH_DIR,
        f"caption_batch_{batch_index:03d}.npz"
    )

    assert os.path.isfile(batch_path), (
        f"Missing batch: {batch_path}"
    )

    start = batch_index * 32
    end = min(start + 32, len(caption_df))
    expected_rows = caption_df.iloc[start:end].reset_index(drop=True)

    # Validate IDs, splits, dimensions, normalization and fingerprint
    validate_caption_batch(batch_path, expected_rows)

    with np.load(batch_path, allow_pickle=False) as batch:
        embeddings = batch["embeddings"].copy()

    embedding_batches.append(embeddings)

    print(
        f"Loaded and validated batch "
        f"{batch_index + 1}/{expected_batches}: "
        f"{len(embeddings)} embeddings"
    )

# --------------------------------------------------
# 3. Combine batches
# --------------------------------------------------

combined_caption_embeddings = np.concatenate(
    embedding_batches,
    axis=0
).astype(np.float32, copy=False)

print("\nCombined embedding shape:",
      combined_caption_embeddings.shape)

# --------------------------------------------------
# 4. Independently validate combined matrix
# --------------------------------------------------

assert combined_caption_embeddings.shape == (1001, 1152)
assert np.isfinite(combined_caption_embeddings).all()

combined_norms = np.linalg.norm(
    combined_caption_embeddings,
    axis=1
)

assert np.allclose(
    combined_norms,
    1.0,
    atol=1e-4
)

assert len(caption_df) == 1001
assert caption_df["caption_id"].is_unique

print("All values finite: True")
print("All embeddings L2-normalized: True")
print("Minimum embedding norm:", float(combined_norms.min()))
print("Maximum embedding norm:", float(combined_norms.max()))

# --------------------------------------------------
# 5. Save final embeddings and metadata manifest
# --------------------------------------------------

temporary_embedding_path = FINAL_CAPTION_EMBEDDINGS + ".tmp.npy"

np.save(
    temporary_embedding_path,
    combined_caption_embeddings
)

os.replace(
    temporary_embedding_path,
    FINAL_CAPTION_EMBEDDINGS
)

# Preserve the full caption-to-image mapping and other metadata
caption_output_manifest = caption_df.copy()

caption_output_manifest.to_csv(
    FINAL_CAPTION_MANIFEST,
    index=False
)

# --------------------------------------------------
# 6. Reload final files and verify saved outputs
# --------------------------------------------------

reloaded_embeddings = np.load(
    FINAL_CAPTION_EMBEDDINGS,
    allow_pickle=False
)

reloaded_manifest = pd.read_csv(FINAL_CAPTION_MANIFEST)

assert reloaded_embeddings.shape == (1001, 1152)
assert np.array_equal(
    reloaded_embeddings,
    combined_caption_embeddings
)

assert len(reloaded_manifest) == 1001
assert reloaded_manifest["caption_id"].is_unique
assert reloaded_manifest["image_id"].notna().all()

assert np.array_equal(
    reloaded_manifest["caption_id"].astype(str).to_numpy(),
    caption_df["caption_id"].astype(str).to_numpy()
)

assert np.array_equal(
    reloaded_manifest["image_id"].astype(str).to_numpy(),
    caption_df["image_id"].astype(str).to_numpy()
)

# --------------------------------------------------
# 7. Final report
# --------------------------------------------------

print("\n========== FINAL CAPTION EMBEDDING REPORT ==========")
print("Total caption embeddings:", len(reloaded_embeddings))
print("Embedding dimension:", reloaded_embeddings.shape[1])
print("Embedding matrix shape:", reloaded_embeddings.shape)
print("Caption IDs unique:", reloaded_manifest["caption_id"].is_unique)
print("Image mappings preserved:", True)
print("Embedding values finite:", np.isfinite(reloaded_embeddings).all())
print("Embedding norms valid:", np.allclose(
    np.linalg.norm(reloaded_embeddings, axis=1),
    1.0,
    atol=1e-4
))
print("Exact save/reload equality:", np.array_equal(
    reloaded_embeddings,
    combined_caption_embeddings
))

print("\nSplit counts:")
print(reloaded_manifest["split"].value_counts())

print("\nSaved embedding file:")
print(FINAL_CAPTION_EMBEDDINGS)

print("\nSaved metadata manifest:")
print(FINAL_CAPTION_MANIFEST)

print("\nEmbedding file size (MB):",
      round(os.path.getsize(FINAL_CAPTION_EMBEDDINGS) / (1024**2), 2))

print("\nCaption embedding combination and verification completed.")


In [ ]:

import os
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Define the verified file paths
# --------------------------------------------------

IMAGE_EMBEDDINGS_PATH = os.path.join(
    FEATURE_DIR,
    "verified_image_extraction",
    "verite_siglip2_image_projected_embeddings.npy"
)

IMAGE_MANIFEST_PATH = os.path.join(
    FEATURE_DIR,
    "verified_image_extraction",
    "verite_siglip2_image_projected_embeddings_manifest.csv"
)

CAPTION_EMBEDDINGS_PATH = os.path.join(
    CAPTION_DIR,
    "verite_siglip2_caption_projected_embeddings.npy"
)

CAPTION_MANIFEST_PATH = os.path.join(
    CAPTION_DIR,
    "verite_siglip2_caption_projected_embeddings_manifest.csv"
)

# --------------------------------------------------
# 2. Load the saved data
# --------------------------------------------------

image_embeddings = np.load(
    IMAGE_EMBEDDINGS_PATH,
    allow_pickle=False
)

caption_embeddings = np.load(
    CAPTION_EMBEDDINGS_PATH,
    allow_pickle=False
)

image_manifest = pd.read_csv(IMAGE_MANIFEST_PATH)
caption_manifest = pd.read_csv(CAPTION_MANIFEST_PATH)

# --------------------------------------------------
# 3. Check expected sizes and required columns
# --------------------------------------------------

assert image_embeddings.shape == (663, 1152)
assert caption_embeddings.shape == (1001, 1152)

required_image_columns = {"image_id", "split"}
required_caption_columns = {
    "caption_id", "image_id", "split", "caption"
}

assert required_image_columns.issubset(image_manifest.columns)
assert required_caption_columns.issubset(caption_manifest.columns)

assert len(image_manifest) == 663
assert len(caption_manifest) == 1001

assert image_manifest["image_id"].is_unique
assert caption_manifest["caption_id"].is_unique

# --------------------------------------------------
# 4. Check that all caption image IDs exist
# --------------------------------------------------

image_ids = set(image_manifest["image_id"].astype(str))
caption_image_ids = set(caption_manifest["image_id"].astype(str))

missing_image_ids = caption_image_ids - image_ids
unused_image_ids = image_ids - caption_image_ids

assert len(missing_image_ids) == 0, (
    f"Caption records refer to unknown images: {missing_image_ids}"
)

# --------------------------------------------------
# 5. Check split consistency for every image
# --------------------------------------------------

image_split_map = (
    image_manifest
    .assign(image_id=image_manifest["image_id"].astype(str))
    .set_index("image_id")["split"]
    .to_dict()
)

caption_split_map = (
    caption_manifest
    .assign(image_id=caption_manifest["image_id"].astype(str))
    .groupby("image_id")["split"]
    .agg(lambda values: set(values.astype(str)))
)

inconsistent_splits = {}

for image_id, caption_splits in caption_split_map.items():
    expected_split = str(image_split_map[image_id])

    if caption_splits != {expected_split}:
        inconsistent_splits[image_id] = {
            "image_split": expected_split,
            "caption_splits": sorted(caption_splits)
        }

assert len(inconsistent_splits) == 0, (
    f"Split inconsistencies found: {inconsistent_splits}"
)

# --------------------------------------------------
# 6. Check embedding validity
# --------------------------------------------------

assert np.isfinite(image_embeddings).all()
assert np.isfinite(caption_embeddings).all()

image_norms = np.linalg.norm(image_embeddings, axis=1)
caption_norms = np.linalg.norm(caption_embeddings, axis=1)

assert np.allclose(image_norms, 1.0, atol=1e-4)
assert np.allclose(caption_norms, 1.0, atol=1e-4)

# --------------------------------------------------
# 7. Report results
# --------------------------------------------------

print("========== IMAGE-CAPTION CONSISTENCY REPORT ==========")

print("\nImage embeddings:")
print("Shape:", image_embeddings.shape)
print("Unique image IDs:", image_manifest["image_id"].nunique())

print("\nCaption embeddings:")
print("Shape:", caption_embeddings.shape)
print("Unique caption IDs:", caption_manifest["caption_id"].nunique())

print("\nMapping checks:")
print("Unique caption image IDs:", len(caption_image_ids))
print("Missing image IDs:", len(missing_image_ids))
print("Images without captions:", len(unused_image_ids))
print("Split inconsistencies:", len(inconsistent_splits))

print("\nCaption split counts:")
print(caption_manifest["split"].value_counts().to_string())

print("\nImage split counts:")
print(image_manifest["split"].value_counts().to_string())

print("\nEmbedding checks:")
print("Image embeddings finite:", np.isfinite(image_embeddings).all())
print("Caption embeddings finite:", np.isfinite(caption_embeddings).all())
print("Image embeddings normalized:", np.allclose(image_norms, 1.0, atol=1e-4))
print("Caption embeddings normalized:", np.allclose(caption_norms, 1.0, atol=1e-4))

print("\nAll image-caption mapping checks passed.")


In [ ]:

import torch
import gc
from transformers import CLIPModel, CLIPProcessor
from PIL import Image

CLIP_MODEL_ID = "openai/clip-vit-large-patch14"

# Load CLIP ViT-L/14 onto the existing GPU
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)

clip_model = CLIPModel.from_pretrained(
    CLIP_MODEL_ID
).to(device)

clip_model.eval()

print("Model:", CLIP_MODEL_ID)
print("Model class:", type(clip_model).__name__)
print("Device:", next(clip_model.parameters()).device)
print("Vision hidden size:", clip_model.config.vision_config.hidden_size)
print("Projection dimension:", clip_model.config.projection_dim)

# Test one image from the verified VERITE manifest
test_image_path = image_manifest.iloc[0]["resolved_image_path"]

assert os.path.isfile(test_image_path), (
    f"Test image not found: {test_image_path}"
)

with Image.open(test_image_path) as img:
    test_image = img.convert("RGB")

image_inputs = clip_processor(
    images=test_image,
    return_tensors="pt"
)

image_inputs = {
    key: value.to(device)
    for key, value in image_inputs.items()
}

with torch.inference_mode():
    clip_test_output = clip_model.get_image_features(**image_inputs)

print("\nImage feature output type:", type(clip_test_output))

if hasattr(clip_test_output, "pooler_output"):
    print("Output contains pooler_output:", True)
    print("Pooler output shape:", clip_test_output.pooler_output.shape)
elif isinstance(clip_test_output, torch.Tensor):
    print("Output is a tensor:", True)
    print("Tensor shape:", clip_test_output.shape)
else:
    print("Output structure:", clip_test_output)

print("\nCLIP test inference completed.")


In [ ]:

import inspect
import torch
import numpy as np

# Inspect the actual feature-extraction implementation
print("========== CLIP FEATURE METHOD ==========")

print(inspect.getsource(clip_model.get_image_features))

# Run the same test image again
with torch.inference_mode():
    clip_test_output = clip_model.get_image_features(**image_inputs)

raw_clip_features = clip_test_output.pooler_output

# Measure the norm before any additional normalization
raw_norm = torch.linalg.vector_norm(
    raw_clip_features,
    ord=2,
    dim=-1
)

# Create an explicitly L2-normalized version for comparison
normalized_clip_features = torch.nn.functional.normalize(
    raw_clip_features,
    p=2,
    dim=-1
)

normalized_norm = torch.linalg.vector_norm(
    normalized_clip_features,
    ord=2,
    dim=-1
)

print("\n========== NORMALIZATION TEST ==========")
print("Output shape:", tuple(raw_clip_features.shape))
print("Raw feature norm:", raw_norm.cpu().numpy())
print("Normalized feature norm:", normalized_norm.cpu().numpy())
print(
    "Raw features already normalized:",
    bool(torch.allclose(
        raw_norm,
        torch.ones_like(raw_norm),
        atol=1e-4
    ))
)
print(
    "Explicitly normalized features have unit norm:",
    bool(torch.allclose(
        normalized_norm,
        torch.ones_like(normalized_norm),
        atol=1e-4
    ))
)
print("All raw values finite:", bool(torch.isfinite(raw_clip_features).all()))
print("All normalized values finite:", bool(torch.isfinite(normalized_clip_features).all()))


In [ ]:

import os
import json
import time
import hashlib
import numpy as np
import pandas as pd
import torch
from PIL import Image

# --------------------------------------------------
# 1. Verify model and image manifest
# --------------------------------------------------

assert "clip_model" in globals()
assert "clip_processor" in globals()
assert "image_manifest" in globals()
assert len(image_manifest) == 663

CLIP_BATCH_SIZE = 32
CLIP_EMBEDDING_DIM = 768
TOTAL_IMAGES = len(image_manifest)

clip_image_df = image_manifest.reset_index(drop=True).copy()

required_columns = {
    "image_id", "resolved_image_path", "split"
}
assert required_columns.issubset(clip_image_df.columns)
assert clip_image_df["image_id"].is_unique
assert clip_image_df["resolved_image_path"].notna().all()
assert clip_image_df["split"].notna().all()
assert clip_image_df["resolved_image_path"].map(os.path.isfile).all()

clip_model.eval()

# --------------------------------------------------
# 2. Configure output directories
# --------------------------------------------------

CLIP_FEATURE_DIR = os.path.join(
    PROJECT_DIR,
    "features",
    "vision",
    "verite",
    "clip_vit_l14"
)

CLIP_VERIFIED_DIR = os.path.join(
    CLIP_FEATURE_DIR,
    "verified_image_extraction"
)

CLIP_BATCH_DIR = os.path.join(
    CLIP_VERIFIED_DIR,
    "image_embedding_batches"
)

os.makedirs(CLIP_BATCH_DIR, exist_ok=True)

CLIP_CONFIG_PATH = os.path.join(
    CLIP_VERIFIED_DIR,
    "clip_extraction_config.json"
)

manifest_hash = hashlib.sha256(
    clip_image_df.to_csv(index=False).encode("utf-8")
).hexdigest()

expected_batches = (
    TOTAL_IMAGES + CLIP_BATCH_SIZE - 1
) // CLIP_BATCH_SIZE

clip_config = {
    "model_id": CLIP_MODEL_ID,
    "embedding_type": "projected_image_embeddings",
    "embedding_dimension": CLIP_EMBEDDING_DIM,
    "normalization": "L2",
    "batch_size": CLIP_BATCH_SIZE,
    "total_images": TOTAL_IMAGES,
    "expected_batches": expected_batches,
    "manifest_sha256": manifest_hash,
    "device": str(device)
}

if os.path.isfile(CLIP_CONFIG_PATH):
    with open(CLIP_CONFIG_PATH, "r") as f:
        previous_config = json.load(f)

    assert previous_config == clip_config, (
        "Existing CLIP extraction configuration differs. "
        "Stop and inspect before continuing."
    )
else:
    with open(CLIP_CONFIG_PATH, "w") as f:
        json.dump(clip_config, f, indent=2)

print("Model:", CLIP_MODEL_ID)
print("Device:", device)
print("Images:", TOTAL_IMAGES)
print("Batch size:", CLIP_BATCH_SIZE)
print("Expected batches:", expected_batches)
print("Output:", CLIP_BATCH_DIR)

# --------------------------------------------------
# 3. Batch validation function
# --------------------------------------------------

def validate_clip_batch(batch_path, expected_rows):

    with np.load(batch_path, allow_pickle=False) as batch:
        embeddings = batch["embeddings"]
        image_ids = batch["image_ids"].astype(str)
        splits = batch["splits"].astype(str)
        saved_hash = str(batch["manifest_sha256"].item())

    assert embeddings.shape == (
        len(expected_rows), CLIP_EMBEDDING_DIM
    ), f"Incorrect embedding shape: {batch_path}"

    assert np.isfinite(embeddings).all(), (
        f"Non-finite embedding values: {batch_path}"
    )

    norms = np.linalg.norm(embeddings, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-4), (
        f"Embedding normalization failed: {batch_path}"
    )

    assert np.array_equal(
        image_ids,
        expected_rows["image_id"].astype(str).to_numpy()
    ), f"Image ID mismatch: {batch_path}"

    assert np.array_equal(
        splits,
        expected_rows["split"].astype(str).to_numpy()
    ), f"Split mismatch: {batch_path}"

    assert saved_hash == manifest_hash, (
        f"Manifest fingerprint mismatch: {batch_path}"
    )

    return len(embeddings)

# --------------------------------------------------
# 4. Extract images in batches
# --------------------------------------------------

newly_completed = 0
previously_completed = 0
start_time = time.time()

for batch_index in range(expected_batches):

    start = batch_index * CLIP_BATCH_SIZE
    end = min(start + CLIP_BATCH_SIZE, TOTAL_IMAGES)

    batch_rows = clip_image_df.iloc[start:end].reset_index(drop=True)

    batch_path = os.path.join(
        CLIP_BATCH_DIR,
        f"clip_batch_{batch_index:03d}.npz"
    )

    temp_path = batch_path + ".tmp.npz"

    # Remove only an incomplete temporary checkpoint
    if os.path.isfile(temp_path):
        os.remove(temp_path)

    # Reuse only a fully validated existing batch
    if os.path.isfile(batch_path):
        validate_clip_batch(batch_path, batch_rows)
        previously_completed += 1
        print(
            f"Batch {batch_index + 1}/{expected_batches}: "
            "already validated and skipped"
        )
        continue

    batch_start = time.time()

    images = []

    for image_path in batch_rows["resolved_image_path"]:
        with Image.open(image_path) as img:
            images.append(img.convert("RGB"))

    image_inputs = clip_processor(
        images=images,
        return_tensors="pt"
    )

    image_inputs = {
        key: value.to(device)
        for key, value in image_inputs.items()
    }

    with torch.inference_mode():
        clip_outputs = clip_model.get_image_features(
            **image_inputs
        )

        raw_features = clip_outputs.pooler_output

        normalized_features = torch.nn.functional.normalize(
            raw_features,
            p=2,
            dim=-1
        )

        embeddings = (
            normalized_features.float().cpu().numpy()
        ).astype(np.float32, copy=False)

    image_ids = batch_rows["image_id"].astype(str).to_numpy(dtype=str)
    splits = batch_rows["split"].astype(str).to_numpy(dtype=str)

    # Validate before saving
    assert embeddings.shape == (
        len(batch_rows), CLIP_EMBEDDING_DIM
    )
    assert np.isfinite(embeddings).all()
    assert np.allclose(
        np.linalg.norm(embeddings, axis=1),
        1.0,
        atol=1e-4
    )

    # Save fixed-width Unicode metadata
    np.savez_compressed(
        temp_path,
        embeddings=embeddings,
        image_ids=image_ids,
        splits=splits,
        manifest_sha256=np.array(manifest_hash)
    )

    # Validate temporary file, then promote it
    validate_clip_batch(temp_path, batch_rows)
    os.replace(temp_path, batch_path)

    # Verify the final saved batch
    validate_clip_batch(batch_path, batch_rows)

    newly_completed += 1

    print(
        f"Batch {batch_index + 1}/{expected_batches}: "
        f"{len(batch_rows)} images saved and validated "
        f"({time.time() - batch_start:.2f}s)"
    )

# --------------------------------------------------
# 5. Extraction summary
# --------------------------------------------------

print("\n========== CLIP EXTRACTION SUMMARY ==========")
print("Total images:", TOTAL_IMAGES)
print("Expected batches:", expected_batches)
print("Newly completed:", newly_completed)
print("Previously completed and skipped:", previously_completed)
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("Output directory:", CLIP_BATCH_DIR)
print("All extraction batches processed.")


In [ ]:

import os
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Define final output paths
# --------------------------------------------------

CLIP_FINAL_EMBEDDINGS = os.path.join(
    CLIP_VERIFIED_DIR,
    "verite_clip_vit_l14_image_projected_embeddings.npy"
)

CLIP_FINAL_MANIFEST = os.path.join(
    CLIP_VERIFIED_DIR,
    "verite_clip_vit_l14_image_projected_embeddings_manifest.csv"
)

# --------------------------------------------------
# 2. Load and validate all 21 batches in order
# --------------------------------------------------

clip_embedding_batches = []

for batch_index in range(expected_batches):

    batch_path = os.path.join(
        CLIP_BATCH_DIR,
        f"clip_batch_{batch_index:03d}.npz"
    )

    assert os.path.isfile(batch_path), (
        f"Missing CLIP batch: {batch_path}"
    )

    start = batch_index * CLIP_BATCH_SIZE
    end = min(start + CLIP_BATCH_SIZE, TOTAL_IMAGES)

    expected_rows = (
        clip_image_df.iloc[start:end].reset_index(drop=True)
    )

    validate_clip_batch(batch_path, expected_rows)

    with np.load(batch_path, allow_pickle=False) as batch:
        embeddings = batch["embeddings"].copy()

    clip_embedding_batches.append(embeddings)

    print(
        f"Loaded and validated batch "
        f"{batch_index + 1}/{expected_batches}: "
        f"{len(embeddings)} embeddings"
    )

# --------------------------------------------------
# 3. Combine batches
# --------------------------------------------------

combined_clip_embeddings = np.concatenate(
    clip_embedding_batches,
    axis=0
).astype(np.float32, copy=False)

print("\nCombined CLIP embedding shape:",
      combined_clip_embeddings.shape)

# --------------------------------------------------
# 4. Validate the complete embedding matrix
# --------------------------------------------------

assert combined_clip_embeddings.shape == (663, 768)
assert np.isfinite(combined_clip_embeddings).all()

clip_norms = np.linalg.norm(
    combined_clip_embeddings,
    axis=1
)

assert np.allclose(
    clip_norms,
    1.0,
    atol=1e-4
)

assert len(clip_image_df) == 663
assert clip_image_df["image_id"].is_unique

print("All values finite:",
      np.isfinite(combined_clip_embeddings).all())

print("All embeddings L2-normalized:",
      np.allclose(clip_norms, 1.0, atol=1e-4))

print("Minimum embedding norm:", float(clip_norms.min()))
print("Maximum embedding norm:", float(clip_norms.max()))

# --------------------------------------------------
# 5. Save final embeddings and manifest
# --------------------------------------------------

temporary_path = CLIP_FINAL_EMBEDDINGS + ".tmp.npy"

np.save(temporary_path, combined_clip_embeddings)

os.replace(
    temporary_path,
    CLIP_FINAL_EMBEDDINGS
)

clip_output_manifest = clip_image_df.copy()

clip_output_manifest.to_csv(
    CLIP_FINAL_MANIFEST,
    index=False
)

# --------------------------------------------------
# 6. Reload and verify saved files
# --------------------------------------------------

reloaded_clip_embeddings = np.load(
    CLIP_FINAL_EMBEDDINGS,
    allow_pickle=False
)

reloaded_clip_manifest = pd.read_csv(
    CLIP_FINAL_MANIFEST
)

assert reloaded_clip_embeddings.shape == (663, 768)

assert np.array_equal(
    reloaded_clip_embeddings,
    combined_clip_embeddings
)

assert len(reloaded_clip_manifest) == 663
assert reloaded_clip_manifest["image_id"].is_unique

assert np.array_equal(
    reloaded_clip_manifest["image_id"].astype(str).to_numpy(),
    clip_image_df["image_id"].astype(str).to_numpy()
)

assert np.array_equal(
    reloaded_clip_manifest["split"].astype(str).to_numpy(),
    clip_image_df["split"].astype(str).to_numpy()
)

# --------------------------------------------------
# 7. Final report
# --------------------------------------------------

print("\n========== FINAL CLIP EMBEDDING REPORT ==========")
print("Model:", CLIP_MODEL_ID)
print("Total image embeddings:", len(reloaded_clip_embeddings))
print("Embedding dimension:", reloaded_clip_embeddings.shape[1])
print("Embedding matrix shape:", reloaded_clip_embeddings.shape)
print("Unique image IDs:", reloaded_clip_manifest["image_id"].nunique())

print("\nSplit counts:")
print(reloaded_clip_manifest["split"].value_counts().to_string())

print("\nValidation:")
print("Values finite:", np.isfinite(reloaded_clip_embeddings).all())
print("Embeddings normalized:", np.allclose(
    np.linalg.norm(reloaded_clip_embeddings, axis=1),
    1.0,
    atol=1e-4
))
print("Exact save/reload equality:", np.array_equal(
    reloaded_clip_embeddings,
    combined_clip_embeddings
))
print("Image ID mapping preserved:", True)
print("Split mapping preserved:", True)

print("\nSaved embedding file:")
print(CLIP_FINAL_EMBEDDINGS)

print("\nSaved manifest:")
print(CLIP_FINAL_MANIFEST)

print(
    "\nEmbedding file size (MB):",
    round(os.path.getsize(CLIP_FINAL_EMBEDDINGS) / (1024**2), 2)
)

print("\nCLIP combination and verification completed.")


In [ ]:

import os
import json
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Load both final manifests
# --------------------------------------------------

SIGLIP_IMAGE_MANIFEST = os.path.join(
    FEATURE_DIR,
    "verified_image_extraction",
    "verite_siglip2_image_projected_embeddings_manifest.csv"
)

CLIP_IMAGE_MANIFEST = os.path.join(
    CLIP_VERIFIED_DIR,
    "verite_clip_vit_l14_image_projected_embeddings_manifest.csv"
)

siglip_manifest = pd.read_csv(SIGLIP_IMAGE_MANIFEST)
clip_manifest = pd.read_csv(CLIP_IMAGE_MANIFEST)

# --------------------------------------------------
# 2. Check required columns and row counts
# --------------------------------------------------

required_columns = {
    "image_id",
    "resolved_image_path",
    "split"
}

assert required_columns.issubset(siglip_manifest.columns)
assert required_columns.issubset(clip_manifest.columns)

assert len(siglip_manifest) == 663
assert len(clip_manifest) == 663

assert siglip_manifest["image_id"].is_unique
assert clip_manifest["image_id"].is_unique

# --------------------------------------------------
# 3. Compare image IDs in exact row order
# --------------------------------------------------

siglip_ids = siglip_manifest["image_id"].astype(str).to_numpy()
clip_ids = clip_manifest["image_id"].astype(str).to_numpy()

same_image_ids_same_order = np.array_equal(
    siglip_ids,
    clip_ids
)

# --------------------------------------------------
# 4. Compare image paths in exact row order
# --------------------------------------------------

siglip_paths = (
    siglip_manifest["resolved_image_path"].astype(str).to_numpy()
)

clip_paths = (
    clip_manifest["resolved_image_path"].astype(str).to_numpy()
)

same_image_paths_same_order = np.array_equal(
    siglip_paths,
    clip_paths
)

# --------------------------------------------------
# 5. Compare split assignments in exact row order
# --------------------------------------------------

siglip_splits = siglip_manifest["split"].astype(str).to_numpy()
clip_splits = clip_manifest["split"].astype(str).to_numpy()

same_splits_same_order = np.array_equal(
    siglip_splits,
    clip_splits
)

# --------------------------------------------------
# 6. Identify any differences
# --------------------------------------------------

id_mismatches = np.where(siglip_ids != clip_ids)[0].tolist()
path_mismatches = np.where(siglip_paths != clip_paths)[0].tolist()
split_mismatches = np.where(siglip_splits != clip_splits)[0].tolist()

# --------------------------------------------------
# 7. Compare split distributions
# --------------------------------------------------

siglip_split_counts = (
    siglip_manifest["split"].value_counts().sort_index()
)

clip_split_counts = (
    clip_manifest["split"].value_counts().sort_index()
)

same_split_counts = siglip_split_counts.equals(
    clip_split_counts
)

# --------------------------------------------------
# 8. Final validation
# --------------------------------------------------

all_checks_passed = all([
    same_image_ids_same_order,
    same_image_paths_same_order,
    same_splits_same_order,
    same_split_counts
])

print("========== SIGLIP2 vs CLIP MANIFEST COMPARISON ==========")

print("\nDataset sizes:")
print("SigLIP2 images:", len(siglip_manifest))
print("CLIP images:", len(clip_manifest))

print("\nConsistency checks:")
print("Same image IDs in same order:", same_image_ids_same_order)
print("Same image paths in same order:", same_image_paths_same_order)
print("Same split assignments in same order:", same_splits_same_order)
print("Same split distributions:", same_split_counts)

print("\nMismatch counts:")
print("Image ID mismatches:", len(id_mismatches))
print("Image path mismatches:", len(path_mismatches))
print("Split mismatches:", len(split_mismatches))

print("\nSigLIP2 split counts:")
print(siglip_split_counts.to_string())

print("\nCLIP split counts:")
print(clip_split_counts.to_string())

print("\nOverall result:", "PASS" if all_checks_passed else "FAIL")

# --------------------------------------------------
# 9. Save comparison report
# --------------------------------------------------

COMPARISON_REPORT_PATH = os.path.join(
    FEATURE_DIR,
    "siglip2_clip_manifest_comparison_report.json"
)

comparison_report = {
    "dataset": "VERITE",
    "image_count_siglip2": len(siglip_manifest),
    "image_count_clip": len(clip_manifest),
    "same_image_ids_same_order": bool(same_image_ids_same_order),
    "same_image_paths_same_order": bool(same_image_paths_same_order),
    "same_splits_same_order": bool(same_splits_same_order),
    "same_split_distributions": bool(same_split_counts),
    "image_id_mismatch_count": len(id_mismatches),
    "image_path_mismatch_count": len(path_mismatches),
    "split_mismatch_count": len(split_mismatches),
    "overall_status": "passed" if all_checks_passed else "failed"
}

with open(COMPARISON_REPORT_PATH, "w") as f:
    json.dump(comparison_report, f, indent=2)

print("\nComparison report saved:")
print(COMPARISON_REPORT_PATH)

assert all_checks_passed, (
    "Manifest comparison failed. Inspect the mismatch counts."
)

print("\nBoth models use the same verified image ordering and splits.")


In [ ]:

import os
import zipfile
import pandas as pd
from io import BytesIO

M4FC_ZIP = (
    "/content/drive/MyDrive/twincheck/"
    "data/raw/m4fc/Y1_outputs.zip"
)

assert os.path.isfile(M4FC_ZIP), (
    f"M4FC ZIP not found: {M4FC_ZIP}"
)

with zipfile.ZipFile(M4FC_ZIP, "r") as z:
    entries = z.namelist()

    image_entries = [
        name for name in entries
        if name.startswith("vision/m4fc_images/")
        and name.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    manifest_candidates = [
        name for name in entries
        if name.endswith("vision/m4fc_images/manifest.csv")
    ]

    print("Total ZIP entries:", len(entries))
    print("Image files in ZIP:", len(image_entries))
    print("Manifest candidates:", manifest_candidates)

    assert len(manifest_candidates) == 1, (
        "Expected exactly one M4FC image manifest."
    )

    manifest_bytes = z.read(manifest_candidates[0])
    m4fc_archive_manifest = pd.read_csv(
        BytesIO(manifest_bytes)
    )

print("\nManifest rows:", len(m4fc_archive_manifest))
print("Manifest columns:", m4fc_archive_manifest.columns.tolist())

print("\nFirst five manifest rows:")
display(m4fc_archive_manifest.head())

print("\nZIP image count:", len(image_entries))
print("M4FC archive inspection completed.")


In [ ]:

import os
import zipfile
import pandas as pd
from io import BytesIO

# --------------------------------------------------
# 1. Define paths
# --------------------------------------------------

M4FC_ZIP = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "m4fc",
    "Y1_outputs.zip"
)

M4FC_IMAGE_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "vision",
    "m4fc_images"
)

M4FC_AUDIT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "vision",
    "m4fc"
)

os.makedirs(M4FC_IMAGE_DIR, exist_ok=True)
os.makedirs(M4FC_AUDIT_DIR, exist_ok=True)

assert os.path.isfile(M4FC_ZIP), "M4FC ZIP not found."

# --------------------------------------------------
# 2. Read the original manifest and image entries
# --------------------------------------------------

with zipfile.ZipFile(M4FC_ZIP, "r") as z:

    entries = z.namelist()

    image_entries = [
        name for name in entries
        if name.startswith("vision/m4fc_images/")
        and name.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    manifest_path = "vision/m4fc_images/manifest.csv"
    assert manifest_path in entries

    original_manifest = pd.read_csv(
        BytesIO(z.read(manifest_path))
    )

    assert len(image_entries) == 603
    assert len(original_manifest) == 614
    assert original_manifest["id"].is_unique

    # Map image IDs to the actual image files supplied in the ZIP.
    zip_image_map = {}

    for entry in image_entries:
        filename = os.path.basename(entry)
        image_id = os.path.splitext(filename)[0]
        assert image_id not in zip_image_map, (
            f"Duplicate image ID in ZIP: {image_id}"
        )
        zip_image_map[image_id] = entry

    # --------------------------------------------------
    # 3. Extract the 603 supplied images
    # --------------------------------------------------

    extracted_count = 0

    for image_id, entry in zip_image_map.items():

        filename = os.path.basename(entry)
        destination = os.path.join(M4FC_IMAGE_DIR, filename)

        if not os.path.isfile(destination):
            with z.open(entry) as source, open(destination, "wb") as target:
                target.write(source.read())
            extracted_count += 1

# --------------------------------------------------
# 4. Build a complete 614-record audit manifest
# --------------------------------------------------

audit_manifest = original_manifest.copy()

audit_manifest["id"] = audit_manifest["id"].astype(str)

audit_manifest["resolved_image_path"] = audit_manifest["id"].map(
    lambda image_id: (
        os.path.join(
            M4FC_IMAGE_DIR,
            os.path.basename(zip_image_map[image_id])
        )
        if image_id in zip_image_map
        else None
    )
)

audit_manifest["image_exists"] = audit_manifest[
    "resolved_image_path"
].apply(
    lambda path: bool(path) and os.path.isfile(path)
)

# --------------------------------------------------
# 5. Verify the counts and missing IDs
# --------------------------------------------------

available_manifest = audit_manifest[
    audit_manifest["image_exists"]
].copy().reset_index(drop=True)

missing_manifest = audit_manifest[
    ~audit_manifest["image_exists"]
].copy().reset_index(drop=True)

assert len(audit_manifest) == 614
assert len(available_manifest) == 603
assert len(missing_manifest) == 11

assert available_manifest["id"].is_unique
assert available_manifest["resolved_image_path"].is_unique
assert available_manifest["resolved_image_path"].map(os.path.isfile).all()

# --------------------------------------------------
# 6. Save the three manifests
# --------------------------------------------------

AUDIT_PATH = os.path.join(
    M4FC_AUDIT_DIR,
    "m4fc_image_availability_audit.csv"
)

AVAILABLE_PATH = os.path.join(
    M4FC_AUDIT_DIR,
    "m4fc_available_603_image_manifest.csv"
)

MISSING_PATH = os.path.join(
    M4FC_AUDIT_DIR,
    "m4fc_missing_11_image_records.csv"
)

audit_manifest.to_csv(AUDIT_PATH, index=False)
available_manifest.to_csv(AVAILABLE_PATH, index=False)
missing_manifest.to_csv(MISSING_PATH, index=False)

# --------------------------------------------------
# 7. Reload and verify saved manifests
# --------------------------------------------------

reloaded_audit = pd.read_csv(AUDIT_PATH)
reloaded_available = pd.read_csv(AVAILABLE_PATH)
reloaded_missing = pd.read_csv(MISSING_PATH)

assert len(reloaded_audit) == 614
assert len(reloaded_available) == 603
assert len(reloaded_missing) == 11

assert reloaded_available["id"].is_unique
assert reloaded_missing["id"].is_unique

print("========== M4FC IMAGE AVAILABILITY REPORT ==========")
print("Original manifest records:", len(original_manifest))
print("Images supplied in ZIP:", len(image_entries))
print("Newly extracted images:", extracted_count)
print("Verified available images:", len(available_manifest))
print("Missing image records:", len(missing_manifest))

print("\nAvailable images by split:")
print(available_manifest["split"].value_counts(dropna=False).to_string())

print("\nMissing image IDs:")
print(missing_manifest["id"].to_list())

print("\nAll available image paths exist:",
      available_manifest["resolved_image_path"].map(os.path.isfile).all())

print("\nSaved files:")
print("Complete audit:", AUDIT_PATH)
print("Available images:", AVAILABLE_PATH)
print("Missing records:", MISSING_PATH)

print("\nM4FC availability audit completed successfully.")


In [ ]:

import os
import torch
import numpy as np
from PIL import Image
import inspect

# --------------------------------------------------
# 1. Load the verified M4FC manifest
# --------------------------------------------------

M4FC_AVAILABLE_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "vision",
    "m4fc",
    "m4fc_available_603_image_manifest.csv"
)

m4fc_image_df = pd.read_csv(M4FC_AVAILABLE_PATH)

assert len(m4fc_image_df) == 603
assert m4fc_image_df["id"].is_unique
assert m4fc_image_df["resolved_image_path"].map(os.path.isfile).all()

# --------------------------------------------------
# 2. Confirm SigLIP2 model is available
# --------------------------------------------------

assert "model" in globals()
assert "processor" in globals()

model.eval()

print("SigLIP2 model:", MODEL_ID)
print("Device:", next(model.parameters()).device)

# Inspect the image feature method
print("\n========== SIGLIP2 IMAGE FEATURE METHOD ==========")
print(inspect.getsource(model.get_image_features))

# --------------------------------------------------
# 3. Load one real M4FC image
# --------------------------------------------------

test_row = m4fc_image_df.iloc[0]
test_path = test_row["resolved_image_path"]

with Image.open(test_path) as img:
    test_image = img.convert("RGB")

print("\nTest image ID:", test_row["id"])
print("Test image path:", test_path)
print("Image dimensions:", test_image.size)

# --------------------------------------------------
# 4. Run image-only inference
# --------------------------------------------------

image_inputs = processor(
    images=test_image,
    return_tensors="pt"
)

image_inputs = {
    key: value.to(device)
    for key, value in image_inputs.items()
}

with torch.inference_mode():
    siglip_test_output = model.get_image_features(**image_inputs)

print("\nOutput type:", type(siglip_test_output))

if hasattr(siglip_test_output, "pooler_output"):
    raw_features = siglip_test_output.pooler_output
elif isinstance(siglip_test_output, torch.Tensor):
    raw_features = siglip_test_output
else:
    raise TypeError(
        f"Unexpected SigLIP2 output type: {type(siglip_test_output)}"
    )

print("Raw feature shape:", tuple(raw_features.shape))
print("Raw features finite:", bool(torch.isfinite(raw_features).all()))

raw_norm = torch.linalg.vector_norm(raw_features, ord=2, dim=-1)

normalized_features = torch.nn.functional.normalize(
    raw_features,
    p=2,
    dim=-1
)

normalized_norm = torch.linalg.vector_norm(
    normalized_features,
    ord=2,
    dim=-1
)

print("Raw feature norm:", raw_norm.cpu().numpy())
print("Normalized feature norm:", normalized_norm.cpu().numpy())
print("Normalized feature shape:", tuple(normalized_features.shape))
print("Normalized features finite:", bool(torch.isfinite(normalized_features).all()))

print("\nM4FC SigLIP2 one-image test completed.")


In [ ]:

import os
import json
import time
import hashlib
import numpy as np
import pandas as pd
import torch
from PIL import Image

# --------------------------------------------------
# 1. Verify model and available image manifest
# --------------------------------------------------

assert "model" in globals()
assert "processor" in globals()
assert "m4fc_image_df" in globals()

M4FC_BATCH_SIZE = 32
M4FC_EMBEDDING_DIM = 1152
TOTAL_M4FC_IMAGES = len(m4fc_image_df)

m4fc_image_df = m4fc_image_df.reset_index(drop=True).copy()

assert TOTAL_M4FC_IMAGES == 603
assert m4fc_image_df["id"].is_unique
assert m4fc_image_df["resolved_image_path"].map(os.path.isfile).all()

model.eval()

# --------------------------------------------------
# 2. Configure output directories
# --------------------------------------------------

M4FC_FEATURE_DIR = os.path.join(
    PROJECT_DIR,
    "features",
    "vision",
    "m4fc",
    "siglip2"
)

M4FC_BATCH_DIR = os.path.join(
    M4FC_FEATURE_DIR,
    "image_embedding_batches"
)

os.makedirs(M4FC_BATCH_DIR, exist_ok=True)

M4FC_CONFIG_PATH = os.path.join(
    M4FC_FEATURE_DIR,
    "m4fc_siglip2_extraction_config.json"
)

m4fc_manifest_hash = hashlib.sha256(
    m4fc_image_df.to_csv(index=False).encode("utf-8")
).hexdigest()

m4fc_expected_batches = (
    TOTAL_M4FC_IMAGES + M4FC_BATCH_SIZE - 1
) // M4FC_BATCH_SIZE

m4fc_config = {
    "dataset": "M4FC",
    "model_id": MODEL_ID,
    "embedding_type": "vision_pooler_output",
    "embedding_dimension": M4FC_EMBEDDING_DIM,
    "normalization": "L2",
    "batch_size": M4FC_BATCH_SIZE,
    "total_images": TOTAL_M4FC_IMAGES,
    "expected_batches": m4fc_expected_batches,
    "manifest_sha256": m4fc_manifest_hash,
    "device": str(device)
}

if os.path.isfile(M4FC_CONFIG_PATH):
    with open(M4FC_CONFIG_PATH, "r") as f:
        previous_config = json.load(f)

    assert previous_config == m4fc_config, (
        "Existing M4FC extraction configuration differs. "
        "Stop and inspect before continuing."
    )
else:
    with open(M4FC_CONFIG_PATH, "w") as f:
        json.dump(m4fc_config, f, indent=2)

print("Dataset: M4FC")
print("Model:", MODEL_ID)
print("Device:", device)
print("Available images:", TOTAL_M4FC_IMAGES)
print("Batch size:", M4FC_BATCH_SIZE)
print("Expected batches:", m4fc_expected_batches)
print("Output:", M4FC_BATCH_DIR)

# --------------------------------------------------
# 3. Batch validation function
# --------------------------------------------------

def validate_m4fc_siglip_batch(batch_path, expected_rows):

    with np.load(batch_path, allow_pickle=False) as batch:
        embeddings = batch["embeddings"]
        image_ids = batch["image_ids"].astype(str)
        splits = batch["splits"].astype(str)
        saved_hash = str(batch["manifest_sha256"].item())

    assert embeddings.shape == (
        len(expected_rows), M4FC_EMBEDDING_DIM
    ), f"Incorrect embedding shape: {batch_path}"

    assert np.isfinite(embeddings).all(), (
        f"Non-finite embedding values: {batch_path}"
    )

    norms = np.linalg.norm(embeddings, axis=1)

    assert np.allclose(norms, 1.0, atol=1e-4), (
        f"Normalization failed: {batch_path}"
    )

    assert np.array_equal(
        image_ids,
        expected_rows["id"].astype(str).to_numpy()
    ), f"Image ID mismatch: {batch_path}"

    assert np.array_equal(
        splits,
        expected_rows["split"].astype(str).to_numpy()
    ), f"Split mismatch: {batch_path}"

    assert saved_hash == m4fc_manifest_hash, (
        f"Manifest fingerprint mismatch: {batch_path}"
    )

    return len(embeddings)

# --------------------------------------------------
# 4. Extract image embeddings in batches
# --------------------------------------------------

newly_completed = 0
previously_completed = 0
start_time = time.time()

for batch_index in range(m4fc_expected_batches):

    start = batch_index * M4FC_BATCH_SIZE
    end = min(
        start + M4FC_BATCH_SIZE,
        TOTAL_M4FC_IMAGES
    )

    batch_rows = (
        m4fc_image_df.iloc[start:end].reset_index(drop=True)
    )

    batch_path = os.path.join(
        M4FC_BATCH_DIR,
        f"m4fc_siglip2_batch_{batch_index:03d}.npz"
    )

    temp_path = batch_path + ".tmp.npz"

    # Remove incomplete temporary checkpoint only
    if os.path.isfile(temp_path):
        os.remove(temp_path)

    # Reuse an existing batch only if every check passes
    if os.path.isfile(batch_path):
        validate_m4fc_siglip_batch(batch_path, batch_rows)
        previously_completed += 1

        print(
            f"Batch {batch_index + 1}/{m4fc_expected_batches}: "
            "already validated and skipped"
        )
        continue

    batch_start = time.time()

    # Load images
    images = []

    for image_path in batch_rows["resolved_image_path"]:
        with Image.open(image_path) as img:
            images.append(img.convert("RGB"))

    # Preprocess images
    image_inputs = processor(
        images=images,
        return_tensors="pt"
    )

    image_inputs = {
        key: value.to(device)
        for key, value in image_inputs.items()
    }

    # Extract projected vision features
    with torch.inference_mode():
        outputs = model.get_image_features(**image_inputs)

        raw_features = outputs.pooler_output

        embeddings = torch.nn.functional.normalize(
            raw_features,
            p=2,
            dim=-1
        ).float().cpu().numpy().astype(np.float32, copy=False)

    # Preserve original IDs and split assignments
    image_ids = batch_rows["id"].astype(str).to_numpy(dtype=str)
    splits = batch_rows["split"].astype(str).to_numpy(dtype=str)

    # Validate before saving
    assert embeddings.shape == (
        len(batch_rows), M4FC_EMBEDDING_DIM
    )

    assert np.isfinite(embeddings).all()

    assert np.allclose(
        np.linalg.norm(embeddings, axis=1),
        1.0,
        atol=1e-4
    )

    # Save Unicode metadata without object arrays
    np.savez_compressed(
        temp_path,
        embeddings=embeddings,
        image_ids=image_ids,
        splits=splits,
        manifest_sha256=np.array(m4fc_manifest_hash)
    )

    # Validate temporary checkpoint
    validate_m4fc_siglip_batch(temp_path, batch_rows)

    # Promote temporary file to completed checkpoint
    os.replace(temp_path, batch_path)

    # Validate the final saved checkpoint
    validate_m4fc_siglip_batch(batch_path, batch_rows)

    newly_completed += 1

    print(
        f"Batch {batch_index + 1}/{m4fc_expected_batches}: "
        f"{len(batch_rows)} images saved and validated "
        f"({time.time() - batch_start:.2f}s)"
    )

# --------------------------------------------------
# 5. Extraction summary
# --------------------------------------------------

print("\n========== M4FC SIGLIP2 EXTRACTION SUMMARY ==========")
print("Total available images:", TOTAL_M4FC_IMAGES)
print("Expected batches:", m4fc_expected_batches)
print("Newly completed:", newly_completed)
print("Previously completed and skipped:", previously_completed)
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("Output directory:", M4FC_BATCH_DIR)
print("All extraction batches processed.")


In [ ]:

import os
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Define final output paths
# --------------------------------------------------

M4FC_FINAL_EMBEDDINGS = os.path.join(
    M4FC_FEATURE_DIR,
    "m4fc_siglip2_image_projected_embeddings.npy"
)

M4FC_FINAL_MANIFEST = os.path.join(
    M4FC_FEATURE_DIR,
    "m4fc_siglip2_image_projected_embeddings_manifest.csv"
)

# --------------------------------------------------
# 2. Load and validate all 19 batches in order
# --------------------------------------------------

m4fc_embedding_batches = []

for batch_index in range(m4fc_expected_batches):

    batch_path = os.path.join(
        M4FC_BATCH_DIR,
        f"m4fc_siglip2_batch_{batch_index:03d}.npz"
    )

    assert os.path.isfile(batch_path), (
        f"Missing M4FC batch: {batch_path}"
    )

    start = batch_index * M4FC_BATCH_SIZE
    end = min(
        start + M4FC_BATCH_SIZE,
        TOTAL_M4FC_IMAGES
    )

    expected_rows = (
        m4fc_image_df.iloc[start:end].reset_index(drop=True)
    )

    validate_m4fc_siglip_batch(batch_path, expected_rows)

    with np.load(batch_path, allow_pickle=False) as batch:
        embeddings = batch["embeddings"].copy()

    m4fc_embedding_batches.append(embeddings)

    print(
        f"Loaded and validated batch "
        f"{batch_index + 1}/{m4fc_expected_batches}: "
        f"{len(embeddings)} embeddings"
    )

# --------------------------------------------------
# 3. Combine batches
# --------------------------------------------------

combined_m4fc_embeddings = np.concatenate(
    m4fc_embedding_batches,
    axis=0
).astype(np.float32, copy=False)

print("\nCombined embedding shape:",
      combined_m4fc_embeddings.shape)

# --------------------------------------------------
# 4. Validate the complete matrix
# --------------------------------------------------

assert combined_m4fc_embeddings.shape == (603, 1152)
assert np.isfinite(combined_m4fc_embeddings).all()

m4fc_norms = np.linalg.norm(
    combined_m4fc_embeddings,
    axis=1
)

assert np.allclose(
    m4fc_norms,
    1.0,
    atol=1e-4
)

assert len(m4fc_image_df) == 603
assert m4fc_image_df["id"].is_unique

print("All values finite:",
      np.isfinite(combined_m4fc_embeddings).all())

print("All embeddings L2-normalized:",
      np.allclose(m4fc_norms, 1.0, atol=1e-4))

print("Minimum embedding norm:", float(m4fc_norms.min()))
print("Maximum embedding norm:", float(m4fc_norms.max()))

# --------------------------------------------------
# 5. Save final embeddings and manifest
# --------------------------------------------------

temporary_path = M4FC_FINAL_EMBEDDINGS + ".tmp.npy"

np.save(temporary_path, combined_m4fc_embeddings)

os.replace(
    temporary_path,
    M4FC_FINAL_EMBEDDINGS
)

m4fc_output_manifest = m4fc_image_df.copy()

m4fc_output_manifest.to_csv(
    M4FC_FINAL_MANIFEST,
    index=False
)

# --------------------------------------------------
# 6. Reload and verify saved files
# --------------------------------------------------

reloaded_m4fc_embeddings = np.load(
    M4FC_FINAL_EMBEDDINGS,
    allow_pickle=False
)

reloaded_m4fc_manifest = pd.read_csv(
    M4FC_FINAL_MANIFEST
)

assert reloaded_m4fc_embeddings.shape == (603, 1152)

assert np.array_equal(
    reloaded_m4fc_embeddings,
    combined_m4fc_embeddings
)

assert len(reloaded_m4fc_manifest) == 603
assert reloaded_m4fc_manifest["id"].is_unique

assert np.array_equal(
    reloaded_m4fc_manifest["id"].astype(str).to_numpy(),
    m4fc_image_df["id"].astype(str).to_numpy()
)

assert np.array_equal(
    reloaded_m4fc_manifest["split"].astype(str).to_numpy(),
    m4fc_image_df["split"].astype(str).to_numpy()
)

# --------------------------------------------------
# 7. Final report
# --------------------------------------------------

print("\n========== FINAL M4FC SIGLIP2 REPORT ==========")
print("Model:", MODEL_ID)
print("Total image embeddings:", len(reloaded_m4fc_embeddings))
print("Embedding dimension:", reloaded_m4fc_embeddings.shape[1])
print("Embedding matrix shape:", reloaded_m4fc_embeddings.shape)
print("Unique image IDs:", reloaded_m4fc_manifest["id"].nunique())

print("\nOriginal M4FC split counts:")
print(reloaded_m4fc_manifest["split"].value_counts().to_string())

print("\nValidation:")
print("Values finite:", np.isfinite(reloaded_m4fc_embeddings).all())
print("Embeddings normalized:", np.allclose(
    np.linalg.norm(reloaded_m4fc_embeddings, axis=1),
    1.0,
    atol=1e-4
))
print("Exact save/reload equality:", np.array_equal(
    reloaded_m4fc_embeddings,
    combined_m4fc_embeddings
))
print("Image ID mapping preserved:", True)
print("Split mapping preserved:", True)

print("\nSaved embedding file:")
print(M4FC_FINAL_EMBEDDINGS)

print("\nSaved manifest:")
print(M4FC_FINAL_MANIFEST)

print(
    "\nEmbedding file size (MB):",
    round(os.path.getsize(M4FC_FINAL_EMBEDDINGS) / (1024**2), 2)
)

print("\nM4FC embedding combination and verification completed.")


In [ ]:

import os
import json
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Load the source availability records and final feature manifest
# --------------------------------------------------

M4FC_AVAILABLE_PATH = os.path.join(
    PROJECT_DIR,
    "data", "processed", "vision", "m4fc",
    "m4fc_available_603_image_manifest.csv"
)

M4FC_MISSING_PATH = os.path.join(
    PROJECT_DIR,
    "data", "processed", "vision", "m4fc",
    "m4fc_missing_11_image_records.csv"
)

M4FC_AUDIT_PATH = os.path.join(
    PROJECT_DIR,
    "data", "processed", "vision", "m4fc",
    "m4fc_image_availability_audit.csv"
)

available_df = pd.read_csv(M4FC_AVAILABLE_PATH)
missing_df = pd.read_csv(M4FC_MISSING_PATH)
audit_df = pd.read_csv(M4FC_AUDIT_PATH)
feature_manifest = pd.read_csv(M4FC_FINAL_MANIFEST)

feature_matrix = np.load(
    M4FC_FINAL_EMBEDDINGS,
    allow_pickle=False
)

# --------------------------------------------------
# 2. Verify dataset counts
# --------------------------------------------------

assert len(audit_df) == 614
assert len(available_df) == 603
assert len(missing_df) == 11
assert len(feature_manifest) == 603
assert feature_matrix.shape == (603, 1152)

# --------------------------------------------------
# 3. Compare IDs in exact order
# --------------------------------------------------

available_ids = available_df["id"].astype(str).to_numpy()
feature_ids = feature_manifest["id"].astype(str).to_numpy()

same_ids_same_order = np.array_equal(
    available_ids,
    feature_ids
)

# --------------------------------------------------
# 4. Compare paths and split assignments
# --------------------------------------------------

available_paths = (
    available_df["resolved_image_path"].astype(str).to_numpy()
)

feature_paths = (
    feature_manifest["resolved_image_path"].astype(str).to_numpy()
)

same_paths_same_order = np.array_equal(
    available_paths,
    feature_paths
)

available_splits = available_df["split"].astype(str).to_numpy()
feature_splits = feature_manifest["split"].astype(str).to_numpy()

same_splits_same_order = np.array_equal(
    available_splits,
    feature_splits
)

# --------------------------------------------------
# 5. Verify no missing ID entered feature extraction
# --------------------------------------------------

missing_ids = set(missing_df["id"].astype(str))
feature_id_set = set(feature_ids)

missing_ids_in_features = missing_ids.intersection(feature_id_set)

assert len(missing_ids_in_features) == 0

# --------------------------------------------------
# 6. Verify every feature path exists
# --------------------------------------------------

all_feature_paths_exist = all(
    os.path.isfile(path) for path in feature_paths
)

# --------------------------------------------------
# 7. Verify split distributions and feature validity
# --------------------------------------------------

available_split_counts = (
    available_df["split"].value_counts().sort_index()
)

feature_split_counts = (
    feature_manifest["split"].value_counts().sort_index()
)

same_split_counts = available_split_counts.equals(
    feature_split_counts
)

embeddings_finite = bool(np.isfinite(feature_matrix).all())

embedding_norms = np.linalg.norm(feature_matrix, axis=1)

embeddings_normalized = bool(
    np.allclose(embedding_norms, 1.0, atol=1e-4)
)

# --------------------------------------------------
# 8. Overall result
# --------------------------------------------------

checks = {
    "all_603_available_images_have_features": same_ids_same_order,
    "image_paths_match_in_same_order": same_paths_same_order,
    "split_assignments_match": same_splits_same_order,
    "no_missing_image_ids_in_features": len(missing_ids_in_features) == 0,
    "all_feature_image_paths_exist": all_feature_paths_exist,
    "split_distributions_match": same_split_counts,
    "embeddings_finite": embeddings_finite,
    "embeddings_l2_normalized": embeddings_normalized
}

overall_pass = all(checks.values())

# --------------------------------------------------
# 9. Save verification report
# --------------------------------------------------

M4FC_COVERAGE_REPORT = os.path.join(
    M4FC_FEATURE_DIR,
    "m4fc_siglip2_feature_coverage_report.json"
)

report = {
    "dataset": "M4FC",
    "model": MODEL_ID,
    "original_manifest_records": len(audit_df),
    "available_images": len(available_df),
    "missing_images": len(missing_df),
    "feature_embeddings": len(feature_matrix),
    "embedding_dimension": int(feature_matrix.shape[1]),
    "missing_ids_in_features": sorted(missing_ids_in_features),
    "checks": checks,
    "overall_status": "passed" if overall_pass else "failed"
}

with open(M4FC_COVERAGE_REPORT, "w") as f:
    json.dump(report, f, indent=2)

# --------------------------------------------------
# 10. Print results
# --------------------------------------------------

print("========== M4FC SIGLIP2 COVERAGE REPORT ==========")

print("\nCounts:")
print("Original manifest:", len(audit_df))
print("Available images:", len(available_df))
print("Missing images:", len(missing_df))
print("Final feature embeddings:", len(feature_matrix))

print("\nConsistency checks:")
for name, result in checks.items():
    print(f"{name}: {result}")

print("\nAvailable-image split counts:")
print(available_split_counts.to_string())

print("\nFeature-manifest split counts:")
print(feature_split_counts.to_string())

print("\nMissing image IDs in feature matrix:",
      len(missing_ids_in_features))

print("\nOverall status:",
      "PASS" if overall_pass else "FAIL")

print("\nReport saved:")
print(M4FC_COVERAGE_REPORT)

assert overall_pass, (
    "M4FC coverage verification failed. Inspect the report."
)

print("\nM4FC feature coverage verification completed.")


In [ ]:

import os
import pandas as pd

M4FC_CLAIM_MAPPING_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "vision",
    "m4fc_test_claim_image_mapping.csv"
)

assert os.path.isfile(M4FC_CLAIM_MAPPING_PATH), (
    f"Claim mapping file not found: {M4FC_CLAIM_MAPPING_PATH}"
)

m4fc_claim_df = pd.read_csv(M4FC_CLAIM_MAPPING_PATH)

print("========== M4FC CLAIM MAPPING INSPECTION ==========")

print("\nTotal claim records:", len(m4fc_claim_df))
print("\nColumns:")
print(m4fc_claim_df.columns.tolist())

print("\nData types:")
print(m4fc_claim_df.dtypes.to_string())

print("\nFirst five records:")
display(m4fc_claim_df.head())

print("\nMissing values per column:")
print(m4fc_claim_df.isna().sum().to_string())

print("\nNumber of unique claim IDs:")
claim_id_candidates = [
    col for col in m4fc_claim_df.columns
    if "id" in col.lower() and "image" not in col.lower()
]
for col in claim_id_candidates:
    print(col, ":", m4fc_claim_df[col].nunique())

print("\nPotential text columns:")
for col in m4fc_claim_df.columns:
    if (
        "claim" in col.lower()
        or "text" in col.lower()
        or "caption" in col.lower()
    ):
        print(f"\nColumn: {col}")
        print(m4fc_claim_df[col].dropna().astype(str).head(3).tolist())

print("\nM4FC claim mapping inspection completed.")


In [ ]:

import os
import json
import zipfile
import pandas as pd

M4FC_ZIP = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "m4fc",
    "Y1_outputs.zip"
)

M4FC_AR_TEST_FILE = "ar/uncleaned/m4fc_ar_test.jsonl"

# --------------------------------------------------
# 1. Read the Arabic test claims
# --------------------------------------------------

with zipfile.ZipFile(M4FC_ZIP, "r") as z:

    raw_text = z.read(M4FC_AR_TEST_FILE).decode(
        "utf-8-sig"
    )

    m4fc_ar_records = [
        json.loads(line)
        for line in raw_text.splitlines()
        if line.strip()
    ]

m4fc_ar_claims_df = pd.DataFrame(m4fc_ar_records)

# --------------------------------------------------
# 2. Validate the claim records
# --------------------------------------------------

required_columns = {
    "id", "claim", "image_path", "label", "orig_label", "split"
}

assert required_columns.issubset(
    m4fc_ar_claims_df.columns
)

assert len(m4fc_ar_claims_df) == 770
assert m4fc_ar_claims_df["id"].is_unique
assert m4fc_ar_claims_df["claim"].notna().all()

# --------------------------------------------------
# 3. Derive image IDs from the actual image paths
# --------------------------------------------------

m4fc_ar_claims_df["image_filename"] = (
    m4fc_ar_claims_df["image_path"]
    .apply(lambda path: os.path.basename(str(path)))
)

m4fc_ar_claims_df["image_id"] = (
    m4fc_ar_claims_df["image_filename"]
    .apply(lambda filename: os.path.splitext(filename)[0])
)

# Map to the persistent 603-image manifest
available_image_map = dict(
    zip(
        m4fc_image_df["id"].astype(str),
        m4fc_image_df["resolved_image_path"].astype(str)
    )
)

m4fc_ar_claims_df["resolved_image_path"] = (
    m4fc_ar_claims_df["image_id"].map(available_image_map)
)

m4fc_ar_claims_df["image_available"] = (
    m4fc_ar_claims_df["resolved_image_path"].notna()
)

# --------------------------------------------------
# 4. Inspect claim-to-image coverage
# --------------------------------------------------

available_claims = m4fc_ar_claims_df[
    m4fc_ar_claims_df["image_available"]
].copy()

unavailable_claims = m4fc_ar_claims_df[
    ~m4fc_ar_claims_df["image_available"]
].copy()

print("========== M4FC ARABIC CLAIM-IMAGE MAPPING ==========")

print("\nClaim records:", len(m4fc_ar_claims_df))
print("Unique claim IDs:", m4fc_ar_claims_df["id"].nunique())
print("Unique associated image IDs:", m4fc_ar_claims_df["image_id"].nunique())

print("\nImage availability:")
print("Claims with available images:", len(available_claims))
print("Claims with missing images:", len(unavailable_claims))

print("\nUnique available image IDs used by claims:",
      available_claims["image_id"].nunique())

print("\nClaims by original split:")
print(m4fc_ar_claims_df["split"].value_counts(dropna=False).to_string())

print("\nClaims by label:")
print(m4fc_ar_claims_df["label"].value_counts(dropna=False).to_string())

print("\nClaims with available images by label:")
print(available_claims["label"].value_counts(dropna=False).to_string())

print("\nFirst five claim-to-image mappings:")
display(
    m4fc_ar_claims_df[
        [
            "id",
            "claim",
            "image_id",
            "image_available",
            "label"
        ]
    ].head()
)

print("\nClaims with missing images:")
display(
    unavailable_claims[
        ["id", "image_path", "image_id", "claim", "label"]
    ]
)

print("\nAll available claim image paths exist:",
      available_claims["resolved_image_path"].map(os.path.isfile).all())

print("\nArabic claim-image inspection completed.")


In [ ]:

import os
import pandas as pd

# --------------------------------------------------
# 1. Prepare the Arabic claim manifest
# --------------------------------------------------

m4fc_claim_manifest = m4fc_ar_claims_df.copy()

# Preserve original claim IDs and Arabic text
m4fc_claim_manifest["claim_id"] = (
    m4fc_claim_manifest["id"].astype(str)
)

m4fc_claim_manifest["claim_text"] = (
    m4fc_claim_manifest["claim"].astype(str)
)

# Keep image mapping and availability information
m4fc_claim_manifest["image_id"] = (
    m4fc_claim_manifest["image_id"].astype(str)
)

m4fc_claim_manifest["image_available"] = (
    m4fc_claim_manifest["image_available"].astype(bool)
)

# Retain the original split, labels and image paths.
# No missing-image records are removed.

# --------------------------------------------------
# 2. Validate the claim manifest
# --------------------------------------------------

assert len(m4fc_claim_manifest) == 770
assert m4fc_claim_manifest["claim_id"].is_unique
assert m4fc_claim_manifest["claim_text"].notna().all()
assert m4fc_claim_manifest["claim_text"].str.strip().ne("").all()

assert m4fc_claim_manifest["image_available"].sum() == 758
assert (~m4fc_claim_manifest["image_available"]).sum() == 12

available_claim_manifest = m4fc_claim_manifest[
    m4fc_claim_manifest["image_available"]
].copy()

missing_image_claim_manifest = m4fc_claim_manifest[
    ~m4fc_claim_manifest["image_available"]
].copy()

assert available_claim_manifest["resolved_image_path"].map(
    os.path.isfile
).all()

# --------------------------------------------------
# 3. Save permanent manifests
# --------------------------------------------------

M4FC_TEXT_DIR = os.path.join(
    PROJECT_DIR,
    "features",
    "vision",
    "m4fc",
    "siglip2",
    "verified_text_extraction"
)

os.makedirs(M4FC_TEXT_DIR, exist_ok=True)

M4FC_ARABIC_CLAIM_MANIFEST = os.path.join(
    M4FC_TEXT_DIR,
    "m4fc_arabic_claim_manifest.csv"
)

M4FC_ARABIC_PAIRED_MANIFEST = os.path.join(
    M4FC_TEXT_DIR,
    "m4fc_arabic_claims_with_available_images.csv"
)

M4FC_ARABIC_MISSING_IMAGE_CLAIMS = os.path.join(
    M4FC_TEXT_DIR,
    "m4fc_arabic_claims_missing_images.csv"
)

m4fc_claim_manifest.to_csv(
    M4FC_ARABIC_CLAIM_MANIFEST,
    index=False,
    encoding="utf-8-sig"
)

available_claim_manifest.to_csv(
    M4FC_ARABIC_PAIRED_MANIFEST,
    index=False,
    encoding="utf-8-sig"
)

missing_image_claim_manifest.to_csv(
    M4FC_ARABIC_MISSING_IMAGE_CLAIMS,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# 4. Reload and verify saved manifests
# --------------------------------------------------

reloaded_claims = pd.read_csv(
    M4FC_ARABIC_CLAIM_MANIFEST
)

reloaded_paired = pd.read_csv(
    M4FC_ARABIC_PAIRED_MANIFEST
)

reloaded_missing = pd.read_csv(
    M4FC_ARABIC_MISSING_IMAGE_CLAIMS
)

assert len(reloaded_claims) == 770
assert len(reloaded_paired) == 758
assert len(reloaded_missing) == 12

assert reloaded_claims["claim_id"].is_unique
assert reloaded_claims["claim_text"].notna().all()

assert (
    reloaded_claims["image_available"].astype(bool).sum()
    == 758
)

# --------------------------------------------------
# 5. Final report
# --------------------------------------------------

print("========== M4FC ARABIC CLAIM MANIFEST REPORT ==========")

print("Total Arabic claims:", len(reloaded_claims))
print("Unique claim IDs:", reloaded_claims["claim_id"].nunique())
print("Claims with available images:", len(reloaded_paired))
print("Claims with missing images:", len(reloaded_missing))

print("\nClaim labels:")
print(reloaded_claims["label"].value_counts().to_string())

print("\nImage-linked claim labels:")
print(reloaded_paired["label"].value_counts().to_string())

print("\nSaved files:")
print(M4FC_ARABIC_CLAIM_MANIFEST)
print(M4FC_ARABIC_PAIRED_MANIFEST)
print(M4FC_ARABIC_MISSING_IMAGE_CLAIMS)

print("\nAll Arabic claim manifest checks passed.")


In [ ]:

import torch
import torch.nn.functional as F
import numpy as np

# Load five Arabic claims from the verified manifest
test_arabic_claims = (
    m4fc_claim_manifest["claim_text"]
    .head(5)
    .tolist()
)

# Tokenize Arabic text
text_inputs = processor(
    text=test_arabic_claims,
    padding="max_length",
    max_length=64,
    truncation=True,
    return_tensors="pt"
)

# Move inputs to the model's device
model.eval()
text_inputs = {
    key: value.to(device)
    for key, value in text_inputs.items()
}

# Extract SigLIP2 text features
with torch.inference_mode():
    text_outputs = model.get_text_features(**text_inputs)

    # Extract pooled projected features
    raw_embeddings = text_outputs.pooler_output

    # Explicit L2 normalization
    normalized_embeddings = F.normalize(
        raw_embeddings.float(),
        p=2,
        dim=1
    )

# Move results to CPU for inspection
test_embeddings = normalized_embeddings.cpu().numpy()

# Validation
assert test_embeddings.shape == (5, 1152)
assert np.isfinite(test_embeddings).all()

embedding_norms = np.linalg.norm(test_embeddings, axis=1)

assert np.allclose(embedding_norms, 1.0, atol=1e-5)

print("========== M4FC ARABIC SIGLIP2 TEST ==========")
print("Number of test claims:", len(test_arabic_claims))
print("Embedding shape:", test_embeddings.shape)
print("All values finite:", np.isfinite(test_embeddings).all())
print("Embedding norms:", embedding_norms)
print("All embeddings normalized:", np.allclose(embedding_norms, 1.0, atol=1e-5))

print("\nSample Arabic claim:")
print(test_arabic_claims[0])

print("\nFirst 10 embedding values:")
print(test_embeddings[0][:10])

print("\nArabic text embedding test passed.")


In [ ]:

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

BATCH_SIZE = 32
MAX_LENGTH = 64

claims_df = m4fc_claim_manifest.copy()

claim_ids = claims_df["claim_id"].astype(str).tolist()
claim_texts = claims_df["claim_text"].astype(str).tolist()

assert len(claim_texts) == 770
assert len(claim_ids) == 770
assert len(set(claim_ids)) == 770

# Output directories
TEXT_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "features",
    "vision",
    "m4fc",
    "siglip2",
    "verified_text_extraction"
)

BATCH_OUTPUT_DIR = os.path.join(
    TEXT_OUTPUT_DIR,
    "text_embedding_batches"
)

os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)

# --------------------------------------------------
# 2. Extract embeddings in batches
# --------------------------------------------------

model.eval()

embedding_batches = []

for batch_number, start in enumerate(
    tqdm(range(0, len(claim_texts), BATCH_SIZE))
):

    end = min(start + BATCH_SIZE, len(claim_texts))

    batch_texts = claim_texts[start:end]
    batch_ids = claim_ids[start:end]

    text_inputs = processor(
        text=batch_texts,
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
        return_tensors="pt"
    )

    text_inputs = {
        key: value.to(device)
        for key, value in text_inputs.items()
    }

    with torch.inference_mode():
        text_outputs = model.get_text_features(**text_inputs)

        batch_embeddings = F.normalize(
            text_outputs.pooler_output.float(),
            p=2,
            dim=1
        )

    batch_embeddings = batch_embeddings.cpu().numpy()

    # Validate each batch
    assert batch_embeddings.shape == (
        len(batch_texts), 1152
    )
    assert np.isfinite(batch_embeddings).all()

    assert np.allclose(
        np.linalg.norm(batch_embeddings, axis=1),
        1.0,
        atol=1e-5
    )

    # Save checkpoint
    batch_file = os.path.join(
        BATCH_OUTPUT_DIR,
        f"batch_{batch_number:03d}.npz"
    )

    np.savez_compressed(
        batch_file,
        claim_ids=np.array(batch_ids, dtype=str),
        embeddings=batch_embeddings
    )

    embedding_batches.append(batch_embeddings)

# --------------------------------------------------
# 3. Combine all batches
# --------------------------------------------------

final_embeddings = np.vstack(embedding_batches)

assert final_embeddings.shape == (770, 1152)
assert np.isfinite(final_embeddings).all()

final_norms = np.linalg.norm(final_embeddings, axis=1)

assert np.allclose(final_norms, 1.0, atol=1e-5)

# --------------------------------------------------
# 4. Save final embedding matrix
# --------------------------------------------------

FINAL_EMBEDDING_FILE = os.path.join(
    TEXT_OUTPUT_DIR,
    "m4fc_siglip2_arabic_claim_projected_embeddings.npy"
)

FINAL_MANIFEST_FILE = os.path.join(
    TEXT_OUTPUT_DIR,
    "m4fc_siglip2_arabic_claim_projected_embeddings_manifest.csv"
)

np.save(FINAL_EMBEDDING_FILE, final_embeddings)

embedding_manifest = claims_df.copy()

embedding_manifest["embedding_row"] = np.arange(
    len(embedding_manifest)
)

embedding_manifest.to_csv(
    FINAL_MANIFEST_FILE,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# 5. Reload and verify exact save
# --------------------------------------------------

reloaded_embeddings = np.load(FINAL_EMBEDDING_FILE)

assert reloaded_embeddings.shape == (770, 1152)
assert np.array_equal(
    final_embeddings,
    reloaded_embeddings
)

assert len(embedding_manifest) == len(reloaded_embeddings)
assert embedding_manifest["claim_id"].is_unique

# --------------------------------------------------
# 6. Final report
# --------------------------------------------------

print("\n========== M4FC ARABIC TEXT EXTRACTION REPORT ==========")

print("Model: google/siglip2-so400m-patch14-384")
print("Total Arabic claims:", len(claim_texts))
print("Batch size:", BATCH_SIZE)
print("Total batches:", len(embedding_batches))
print("Embedding matrix shape:", final_embeddings.shape)

print("All values finite:", np.isfinite(final_embeddings).all())
print("All embeddings normalized:", np.allclose(
    final_norms, 1.0, atol=1e-5
))

print("Exact save/reload equality:", np.array_equal(
    final_embeddings, reloaded_embeddings
))

print("\nClaims with available images:",
      claims_df["image_available"].sum())

print("Claims with missing images:",
      (~claims_df["image_available"]).sum())

print("\nFinal embedding file:")
print(FINAL_EMBEDDING_FILE)

print("\nEmbedding manifest:")
print(FINAL_MANIFEST_FILE)

print("\nBatch checkpoint directory:")
print(BATCH_OUTPUT_DIR)

print("\nM4FC Arabic SigLIP2 text extraction completed.")


In [ ]:

import os
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Load saved outputs
# --------------------------------------------------

embedding_file = os.path.join(
    PROJECT_DIR,
    "features", "vision", "m4fc", "siglip2",
    "verified_text_extraction",
    "m4fc_siglip2_arabic_claim_projected_embeddings.npy"
)

manifest_file = os.path.join(
    PROJECT_DIR,
    "features", "vision", "m4fc", "siglip2",
    "verified_text_extraction",
    "m4fc_siglip2_arabic_claim_projected_embeddings_manifest.csv"
)

batch_dir = os.path.join(
    PROJECT_DIR,
    "features", "vision", "m4fc", "siglip2",
    "verified_text_extraction",
    "text_embedding_batches"
)

embeddings = np.load(embedding_file)
manifest = pd.read_csv(manifest_file)

# --------------------------------------------------
# 2. Verify row-to-claim alignment
# --------------------------------------------------

assert len(manifest) == embeddings.shape[0] == 770
assert manifest["embedding_row"].tolist() == list(range(770))

assert manifest["claim_id"].astype(str).tolist() == (
    m4fc_claim_manifest["claim_id"].astype(str).tolist()
)

assert manifest["claim_text"].tolist() == (
    m4fc_claim_manifest["claim_text"].tolist()
)

# --------------------------------------------------
# 3. Reconstruct embeddings from batch files
# --------------------------------------------------

batch_files = sorted(
    f for f in os.listdir(batch_dir)
    if f.endswith(".npz")
)

reconstructed_batches = []
reconstructed_ids = []

for filename in batch_files:
    batch = np.load(os.path.join(batch_dir, filename))

    reconstructed_batches.append(batch["embeddings"])
    reconstructed_ids.extend(
        batch["claim_ids"].astype(str).tolist()
    )

reconstructed_embeddings = np.vstack(reconstructed_batches)

assert len(batch_files) == 25
assert reconstructed_embeddings.shape == (770, 1152)
assert np.array_equal(reconstructed_embeddings, embeddings)

assert reconstructed_ids == manifest["claim_id"].astype(str).tolist()

# --------------------------------------------------
# 4. Final validation report
# --------------------------------------------------

print("========== M4FC ARABIC EMBEDDING ALIGNMENT ==========")
print("Embedding rows:", len(embeddings))
print("Manifest records:", len(manifest))
print("Unique claim IDs:", manifest["claim_id"].nunique())
print("Embedding row order correct:",
      manifest["embedding_row"].tolist() == list(range(770)))
print("Claim IDs correctly aligned:", True)
print("Arabic text correctly aligned:", True)
print("Batch checkpoints:", len(batch_files))
print("Batch reconstruction exact:",
      np.array_equal(reconstructed_embeddings, embeddings))
print("Batch claim ID order correct:", True)

print("\nAll embedding alignment checks passed.")


In [ ]:
import os
import torch
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/twincheck")
VISION_DIR = PROJECT_DIR / "data/processed/vision"

# Load the verified GenImage manifests
trainval = pd.read_csv(
    VISION_DIR / "genimage_train_validation_export_manifest.csv"
)

# Standardise image paths
trainval["exported_path"] = trainval["exported_path"].astype(str)

# Verify image paths
trainval_exists = trainval["exported_path"].map(os.path.isfile)

print("========== GENIMAGE EXTRACTION READINESS ==========")

print("\nTraining/validation images:", len(trainval))
print("Existing training/validation files:", trainval_exists.sum())
print("Missing training/validation files:", (~trainval_exists).sum())

print("\nTraining/validation split counts:")
print(trainval["split"].value_counts().to_string())

print("\nClass counts:")
print(trainval["class_name"].value_counts().to_string())

print("\nGenerator counts:")
print(trainval["generator"].value_counts().to_string())

# Check model availability
print("\n========== MODEL STATUS ==========")

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Allocated GPU memory:",
        round(torch.cuda.memory_allocated(0) / 1024**3, 2),
        "GB"
    )

for name in ["model", "processor", "clip_model", "clip_processor"]:
    obj = globals().get(name)
    print(f"{name} loaded:", obj is not None)

if "model" in globals():
    print("SigLIP2 device:", next(model.parameters()).device)

if "clip_model" in globals():
    print("CLIP device:", next(clip_model.parameters()).device)

assert trainval_exists.all(), "Some GenImage image files are missing."
assert len(trainval) == 18000

print("\nGenImage manifests and image paths verified.")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

BATCH_SIZE = 32

GENIMAGE_OUTPUT = os.path.join(
    PROJECT_DIR,
    "features", "vision", "genimage"
)

BATCH_DIR = os.path.join(
    GENIMAGE_OUTPUT,
    "image_embedding_batches"
)

os.makedirs(BATCH_DIR, exist_ok=True)

df = trainval.reset_index(drop=True).copy()

assert len(df) == 18000
assert df["exported_path"].map(os.path.isfile).all()

model.eval()
clip_model.eval()

# --------------------------------------------------
# 2. Extract both models' embeddings
# --------------------------------------------------

for batch_no, start in enumerate(
    tqdm(range(0, len(df), BATCH_SIZE))
):
    end = min(start + BATCH_SIZE, len(df))

    batch_df = df.iloc[start:end]
    batch_file = os.path.join(
        BATCH_DIR, f"batch_{batch_no:04d}.npz"
    )

    # Resume: skip only a complete, valid checkpoint
    if os.path.exists(batch_file):
        saved = np.load(batch_file)

        if (
            len(saved["image_ids"]) == len(batch_df)
            and np.array_equal(
                saved["image_ids"].astype(str),
                batch_df["md5"].astype(str).to_numpy()
            )
            and saved["siglip_embeddings"].shape ==
                (len(batch_df), 1152)
            and saved["clip_embeddings"].shape ==
                (len(batch_df), 768)
        ):
            continue

    images = []

    for path in batch_df["exported_path"]:
        with Image.open(path) as image:
            images.append(image.convert("RGB"))

    # SigLIP2
    siglip_inputs = processor(
        images=images,
        return_tensors="pt"
    )

    siglip_inputs = {
        key: value.to(device)
        for key, value in siglip_inputs.items()
    }

    with torch.inference_mode():
        siglip_output = model.get_image_features(
            **siglip_inputs
        )

        siglip_embeddings = F.normalize(
            siglip_output.pooler_output.float(),
            p=2,
            dim=1
        ).cpu().numpy()

    # CLIP ViT-L/14
    clip_inputs = clip_processor(
        images=images,
        return_tensors="pt"
    )

    clip_inputs = {
        key: value.to(device)
        for key, value in clip_inputs.items()
    }

    with torch.inference_mode():
        clip_output = clip_model.get_image_features(
            **clip_inputs
        )

        clip_embeddings = F.normalize(
            clip_output.pooler_output.float(),
            p=2,
            dim=1
        ).cpu().numpy()

    # Validate batch
    assert siglip_embeddings.shape == (
        len(batch_df), 1152
    )
    assert clip_embeddings.shape == (
        len(batch_df), 768
    )

    assert np.isfinite(siglip_embeddings).all()
    assert np.isfinite(clip_embeddings).all()

    # Save both feature matrices and row identifiers
    np.savez_compressed(
        batch_file,
        image_ids=batch_df["md5"].astype(str).to_numpy(),
        image_paths=batch_df["exported_path"].astype(str).to_numpy(),
        splits=batch_df["split"].astype(str).to_numpy(),
        class_names=batch_df["class_name"].astype(str).to_numpy(),
        generators=batch_df["generator"].astype(str).to_numpy(),
        siglip_embeddings=siglip_embeddings,
        clip_embeddings=clip_embeddings
    )

    del images, siglip_inputs, clip_inputs
    del siglip_embeddings, clip_embeddings

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll GenImage extraction batches processed.")
print("Checkpoint directory:", BATCH_DIR)
print("Expected batches:", (len(df) + BATCH_SIZE - 1) // BATCH_SIZE)

In [ ]:

import torch

# Extract the actual feature tensors
image_features = image_features.pooler_output
text_features = text_features.pooler_output

# L2-normalize the embeddings
image_features = torch.nn.functional.normalize(
    image_features.float(), p=2, dim=-1
)
text_features = torch.nn.functional.normalize(
    text_features.float(), p=2, dim=-1
)

print("IMAGE EMBEDDING")
print("Shape:", tuple(image_features.shape))
print("Finite:", torch.isfinite(image_features).all().item())
print("L2 norm:", image_features.norm(dim=-1).item())

print("\nTEXT EMBEDDING")
print("Shape:", tuple(text_features.shape))
print("Finite:", torch.isfinite(text_features).all().item())
print("L2 norm:", text_features.norm(dim=-1).item())

assert image_features.shape == (1, 1152)
assert text_features.shape == (1, 1152)
assert torch.isfinite(image_features).all()
assert torch.isfinite(text_features).all()

print("\nSUCCESS: Image and text embedding tests passed.")


In [ ]:

from pathlib import Path
import numpy as np

PROJECT_DIR = Path("/content/drive/MyDrive/twincheck")
FEATURE_DIR = PROJECT_DIR / "features/vision/genimage"

print("=" * 60)
print("SEARCHING FOR PREVIOUS GENIMAGE CHECKPOINTS")
print("=" * 60)

print("\nFeature directory exists:", FEATURE_DIR.exists())

all_npz = sorted(FEATURE_DIR.rglob("*.npz"))

print("\nTotal NPZ files found:", len(all_npz))

# Display all checkpoint locations
if all_npz:
    print("\nCHECKPOINT LOCATIONS:")
    for f in all_npz:
        print(f)

# Identify files containing image embedding batches
batch_files = [
    f for f in all_npz
    if "batch" in f.stem.lower()
]

print("\n" + "=" * 60)
print("BATCH CHECKPOINT SUMMARY")
print("=" * 60)

print("Batch files found:", len(batch_files))

valid_batches = []
invalid_batches = []

for f in batch_files:
    try:
        with np.load(f, allow_pickle=False) as data:
            required = [
                "image_ids",
                "siglip_embeddings",
                "clip_embeddings"
            ]

            if not all(k in data.files for k in required):
                invalid_batches.append((str(f), "Missing required arrays"))
                continue

            ids = data["image_ids"]
            sig = data["siglip_embeddings"]
            clip = data["clip_embeddings"]

            valid = (
                sig.shape == (len(ids), 1152)
                and clip.shape == (len(ids), 768)
                and np.isfinite(sig).all()
                and np.isfinite(clip).all()
            )

            if valid:
                valid_batches.append(f)
            else:
                invalid_batches.append((str(f), "Shape or finite-value check failed"))

    except Exception as e:
        invalid_batches.append((str(f), str(e)))

print("Structurally valid batches:", len(valid_batches))
print("Invalid or incomplete batches:", len(invalid_batches))

print("\nVALID BATCH FILES:")
for f in valid_batches:
    print(f.name, "|", f.parent)

print("\nINVALID FILES:")
for item in invalid_batches:
    print(item)

# Check for previously consolidated final matrices
print("\n" + "=" * 60)
print("EXISTING FINAL EMBEDDING FILES")
print("=" * 60)

for f in sorted(FEATURE_DIR.rglob("*.npy")):
    print(f.name, "|", f)

print("\nSEARCH COMPLETE. No files have been modified.")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/twincheck")
FEATURE_DIR = PROJECT_DIR / "features/vision/genimage"
BATCH_DIR = FEATURE_DIR / "image_embedding_batches"

trainval = pd.read_csv(
    PROJECT_DIR / "data/processed/vision/genimage_train_validation_export_manifest.csv"
).reset_index(drop=True)

valid_batches = []
invalid_batches = []

print("Verifying saved GenImage checkpoints...")

for batch_no in range(563):
    batch_file = BATCH_DIR / f"batch_{batch_no:04d}.npz"

    if not batch_file.exists():
        invalid_batches.append((batch_no, "File missing"))
        continue

    try:
        # These are our own previously generated checkpoint files.
        with np.load(batch_file, allow_pickle=True) as saved:

            start = batch_no * 32
            end = min(start + 32, len(trainval))
            expected = trainval.iloc[start:end]

            ids = saved["image_ids"].astype(str)
            expected_ids = expected["md5"].astype(str).to_numpy()

            sig = saved["siglip_embeddings"]
            clip = saved["clip_embeddings"]

            checks_pass = (
                np.array_equal(ids, expected_ids)
                and sig.shape == (len(expected), 1152)
                and clip.shape == (len(expected), 768)
                and np.isfinite(sig).all()
                and np.isfinite(clip).all()
            )

            if checks_pass:
                valid_batches.append(batch_no)
            else:
                invalid_batches.append((batch_no, "Data or alignment check failed"))

    except Exception as e:
        invalid_batches.append((batch_no, str(e)))

print("\n========== VERIFICATION RESULTS ==========")
print("Valid saved batches:", len(valid_batches))
print("Invalid or missing batches:", len(invalid_batches))

if valid_batches:
    print("First valid batch:", valid_batches[0])
    print("Last valid batch:", valid_batches[-1])

print("\nInvalid/missing batch numbers:")
print([x[0] for x in invalid_batches])

print("\nImages already covered:", sum(
    min(32, len(trainval) - b * 32) for b in valid_batches
))

print("Images remaining:", len(trainval) - sum(
    min(32, len(trainval) - b * 32) for b in valid_batches
))


In [ ]:

import gc
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

# ==================================================
# 1. Paths and runtime checks
# ==================================================
PROJECT_DIR = Path("/content/drive/MyDrive/twincheck")
VISION_DIR = PROJECT_DIR / "data/processed/vision"
FEATURE_DIR = PROJECT_DIR / "features/vision/genimage"
BATCH_DIR = FEATURE_DIR / "image_embedding_batches"

BATCH_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "GPU is not available."
assert model is not None and processor is not None
assert clip_model is not None and clip_processor is not None

device = torch.device("cuda")
model.eval()
clip_model.eval()

print("GPU:", torch.cuda.get_device_name(0))
print("SigLIP:", model.__class__.__name__)
print("CLIP:", clip_model.__class__.__name__)

# ==================================================
# 2. Load and validate manifest
# ==================================================
trainval = pd.read_csv(
    VISION_DIR / "genimage_train_validation_export_manifest.csv"
).reset_index(drop=True)

assert len(trainval) == 18000
assert trainval["md5"].is_unique

BATCH_SIZE = 32
NUM_BATCHES = (len(trainval) + BATCH_SIZE - 1) // BATCH_SIZE

print("Total images:", len(trainval))
print("Total batches:", NUM_BATCHES)

# ==================================================
# 3. Verify existing checkpoints
# ==================================================
valid_batches = set()

print("\nChecking previous checkpoints...")

for batch_no in range(NUM_BATCHES):
    batch_file = BATCH_DIR / f"batch_{batch_no:04d}.npz"

    if not batch_file.exists():
        continue

    try:
        start = batch_no * BATCH_SIZE
        end = min(start + BATCH_SIZE, len(trainval))
        expected = trainval.iloc[start:end]

        with np.load(batch_file, allow_pickle=True) as saved:
            ids = saved["image_ids"].astype(str)
            paths = saved["image_paths"].astype(str)

            sig = saved["siglip_embeddings"]
            clip = saved["clip_embeddings"]

            valid = (
                np.array_equal(
                    ids, expected["md5"].astype(str).to_numpy()
                )
                and np.array_equal(
                    paths, expected["exported_path"].astype(str).to_numpy()
                )
                and sig.shape == (len(expected), 1152)
                and clip.shape == (len(expected), 768)
                and np.isfinite(sig).all()
                and np.isfinite(clip).all()
            )

        if valid:
            valid_batches.add(batch_no)

    except Exception as e:
        print(f"Checkpoint {batch_no} needs re-extraction: {e}")

print("Verified existing batches:", len(valid_batches))
print("Batches remaining:", NUM_BATCHES - len(valid_batches))

assert len(valid_batches) >= 279, (
    "Expected at least 279 verified checkpoints. Stopping safely."
)

# ==================================================
# 4. Process only missing or invalid batches
# ==================================================
missing_batches = [
    b for b in range(NUM_BATCHES)
    if b not in valid_batches
]

print("\nStarting extraction of remaining batches...")
print("First remaining batch:", missing_batches[0] if missing_batches else "None")

for batch_no in tqdm(missing_batches, desc="Remaining GenImage batches"):

    start = batch_no * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(trainval))
    batch_df = trainval.iloc[start:end]

    batch_file = BATCH_DIR / f"batch_{batch_no:04d}.npz"

    images = []

    for _, row in batch_df.iterrows():
        with Image.open(row["exported_path"]) as img:
            images.append(img.convert("RGB"))

    # SigLIP preprocessing
    sig_inputs = processor(
        images=images,
        return_tensors="pt"
    )
    sig_inputs = {
        k: v.to(device) for k, v in sig_inputs.items()
    }

    # CLIP preprocessing
    clip_inputs = clip_processor(
        images=images,
        return_tensors="pt"
    )
    clip_inputs = {
        k: v.to(device) for k, v in clip_inputs.items()
    }

    with torch.inference_mode():
        sig_output = model.get_image_features(**sig_inputs)
        clip_output = clip_model.get_image_features(**clip_inputs)

        sig_features = torch.nn.functional.normalize(
            sig_output.pooler_output.float(),
            p=2,
            dim=-1
        )

        clip_features = torch.nn.functional.normalize(
            clip_output.pooler_output.float(),
            p=2,
            dim=-1
        )

    sig_array = sig_features.cpu().numpy().astype(np.float32)
    clip_array = clip_features.cpu().numpy().astype(np.float32)

    assert sig_array.shape == (len(batch_df), 1152)
    assert clip_array.shape == (len(batch_df), 768)
    assert np.isfinite(sig_array).all()
    assert np.isfinite(clip_array).all()

    # Save metadata as strings, not object arrays
    np.savez_compressed(
        batch_file,
        image_ids=batch_df["md5"].astype(str).to_numpy(dtype=str),
        image_paths=batch_df["exported_path"].astype(str).to_numpy(dtype=str),
        splits=batch_df["split"].astype(str).to_numpy(dtype=str),
        class_names=batch_df["class_name"].astype(str).to_numpy(dtype=str),
        generators=batch_df["generator"].astype(str).to_numpy(dtype=str),
        siglip_embeddings=sig_array,
        clip_embeddings=clip_array
    )

    del images, sig_inputs, clip_inputs
    del sig_output, clip_output
    del sig_features, clip_features
    del sig_array, clip_array

    gc.collect()

    # Periodically clear unused GPU cache
    if batch_no % 20 == 0:
        torch.cuda.empty_cache()

torch.cuda.empty_cache()

# ==================================================
# 5. Consolidate all 563 batches
# ==================================================
print("\nAll remaining batches processed.")
print("Consolidating all checkpoints...")

all_ids = []
all_paths = []
all_splits = []
all_classes = []
all_generators = []
siglip_arrays = []
clip_arrays = []

for batch_no in range(NUM_BATCHES):

    batch_file = BATCH_DIR / f"batch_{batch_no:04d}.npz"

    assert batch_file.exists(), f"Missing batch: {batch_no}"

    with np.load(batch_file, allow_pickle=True) as saved:
        all_ids.append(saved["image_ids"].astype(str))
        all_paths.append(saved["image_paths"].astype(str))
        all_splits.append(saved["splits"].astype(str))
        all_classes.append(saved["class_names"].astype(str))
        all_generators.append(saved["generators"].astype(str))

        siglip_arrays.append(saved["siglip_embeddings"])
        clip_arrays.append(saved["clip_embeddings"])

image_ids = np.concatenate(all_ids)
image_paths = np.concatenate(all_paths)
splits = np.concatenate(all_splits)
class_names = np.concatenate(all_classes)
generators = np.concatenate(all_generators)

siglip_embeddings = np.concatenate(siglip_arrays, axis=0)
clip_embeddings = np.concatenate(clip_arrays, axis=0)

# ==================================================
# 6. Final validation
# ==================================================
assert len(image_ids) == 18000

assert np.array_equal(
    image_ids, trainval["md5"].astype(str).to_numpy()
)

assert np.array_equal(
    image_paths, trainval["exported_path"].astype(str).to_numpy()
)

assert siglip_embeddings.shape == (18000, 1152)
assert clip_embeddings.shape == (18000, 768)

assert np.isfinite(siglip_embeddings).all()
assert np.isfinite(clip_embeddings).all()

assert np.allclose(
    np.linalg.norm(siglip_embeddings, axis=1),
    1.0, atol=1e-4
)

assert np.allclose(
    np.linalg.norm(clip_embeddings, axis=1),
    1.0, atol=1e-4
)

# ==================================================
# 7. Save final outputs
# ==================================================
siglip_file = FEATURE_DIR / "genimage_siglip2_image_projected_embeddings.npy"
clip_file = FEATURE_DIR / "genimage_clip_vit_l14_image_projected_embeddings.npy"
manifest_file = FEATURE_DIR / "genimage_image_embedding_manifest.csv"
report_file = FEATURE_DIR / "genimage_feature_extraction_report.json"

np.save(siglip_file, siglip_embeddings)
np.save(clip_file, clip_embeddings)

genimage_embedding_manifest = pd.DataFrame({
    "image_id": image_ids,
    "image_path": image_paths,
    "split": splits,
    "class_name": class_names,
    "generator": generators
})

genimage_embedding_manifest.to_csv(manifest_file, index=False)

report = {
    "dataset": "GenImage train and validation",
    "total_images": 18000,
    "batches": NUM_BATCHES,
    "batch_size": BATCH_SIZE,
    "previous_batches_reused": len(valid_batches),
    "new_batches_processed": len(missing_batches),
    "siglip_model": "google/siglip2-so400m-patch14-384",
    "clip_model": "openai/clip-vit-large-patch14",
    "siglip_embedding_shape": list(siglip_embeddings.shape),
    "clip_embedding_shape": list(clip_embeddings.shape),
    "all_embeddings_finite": True,
    "l2_normalized": True,
    "status": "completed"
}

with open(report_file, "w") as f:
    json.dump(report, f, indent=2)

# ==================================================
# 8. Reload verification
# ==================================================
siglip_check = np.load(siglip_file)
clip_check = np.load(clip_file)
manifest_check = pd.read_csv(manifest_file)

assert np.array_equal(siglip_check, siglip_embeddings)
assert np.array_equal(clip_check, clip_embeddings)
assert len(manifest_check) == 18000

print("\n======================================")
print("GENIMAGE EXTRACTION SUCCESSFULLY COMPLETED")
print("======================================")
print("Existing batches reused:", len(valid_batches))
print("New batches processed:", len(missing_batches))
print("Total images:", len(manifest_check))
print("SigLIP shape:", siglip_check.shape)
print("CLIP shape:", clip_check.shape)
print("Split counts:", manifest_check["split"].value_counts().to_dict())
print("\nSaved files:")
print(siglip_file)
print(clip_file)
print(manifest_file)
print(report_file)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json

GEN_DIR = Path(
    "/content/drive/MyDrive/twincheck/features/vision/genimage"
)

files = [
    "genimage_siglip2_image_projected_embeddings.npy",
    "genimage_clip_vit_l14_image_projected_embeddings.npy",
    "genimage_image_embedding_manifest.csv",
    "genimage_feature_extraction_report.json"
]

print("Checking saved GenImage outputs...\n")

for name in files:
    path = GEN_DIR / name
    print(name, "->", "EXISTS" if path.exists() else "MISSING")

siglip = np.load(
    GEN_DIR / "genimage_siglip2_image_projected_embeddings.npy"
)
clip = np.load(
    GEN_DIR / "genimage_clip_vit_l14_image_projected_embeddings.npy"
)
manifest = pd.read_csv(
    GEN_DIR / "genimage_image_embedding_manifest.csv"
)

print("\nSigLIP shape:", siglip.shape)
print("CLIP shape:", clip.shape)
print("Manifest rows:", len(manifest))
print("SigLIP all finite:", np.isfinite(siglip).all())
print("CLIP all finite:", np.isfinite(clip).all())
print("SigLIP normalized:", np.allclose(
    np.linalg.norm(siglip, axis=1), 1.0, atol=1e-4
))
print("CLIP normalized:", np.allclose(
    np.linalg.norm(clip, axis=1), 1.0, atol=1e-4
))

print("\nSplit counts:")
print(manifest["split"].value_counts().to_dict())

In [ ]:
import torch
import importlib.util
from pathlib import Path

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")
print("CUDA available:", torch.cuda.is_available())

print("\nRequired libraries:")
for package in ["diffusers", "transformers", "accelerate", "safetensors"]:
    print(f"{package}:", importlib.util.find_spec(package) is not None)

print("\nTwinCheck directories:")
BASE = Path("/content/drive/MyDrive/twincheck")

for folder in [
    "data/processed/vision/verite",
    "data/processed/vision/m4fc",
    "features/vision/verite/siglip2",
    "features/vision/m4fc/siglip2",
]:
    path = BASE / folder
    print(f"{folder}: {'EXISTS' if path.exists() else 'MISSING'}")

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/twincheck")

print("Contents of data/processed/vision:")
vision_dir = BASE / "data/processed/vision"

for item in sorted(vision_dir.iterdir()):
    print(item.name, "->", "Directory" if item.is_dir() else "File")

print("\nSearching for VERITE image files...")

for root in [
    BASE / "data",
    BASE / "features/vision/verite"
]:
    if root.exists():
        matches = [
            p for p in root.rglob("*")
            if p.is_file()
            and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
            and "verite" in str(p).lower()
        ]
        print(f"\nSearch under {root}: {len(matches)} matching files")
        for p in matches[:5]:
            print(p)

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/twincheck")

manifest_path = (
    BASE / "features/vision/verite/siglip2/"
    "verite_unique_image_manifest.csv"
)

df = pd.read_csv(manifest_path)

print("Manifest columns:", df.columns.tolist())
print("Total unique images:", len(df))

print("\nSplit counts:")
print(df["split"].value_counts().to_dict())

print("\nSample records:")
display(df.head())

# Check that all manifest image paths exist
path_col = "resolved_image_path"

df["file_exists"] = df[path_col].apply(
    lambda p: Path(p).exists()
)

print("\nExisting image paths:", df["file_exists"].sum())
print("Missing image paths:", (~df["file_exists"]).sum())

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/twincheck")

metadata_path = BASE / "data/processed/vision/verite_cleaned.csv"
metadata = pd.read_csv(metadata_path)

print("Metadata shape:", metadata.shape)
print("\nColumns:")
print(metadata.columns.tolist())

print("\nFirst five records:")
display(metadata.head())

print("\nPossible label columns:")
for col in metadata.columns:
    if any(word in col.lower() for word in ["label", "class", "verdict", "category"]):
        print(f"\n{col}:")
        print(metadata[col].value_counts(dropna=False))

In [ ]:
print("True image columns:")
print(true_images.columns.tolist())

print("\nUnique manifest columns:")
print(unique_manifest.columns.tolist())

print("\nTrue image sample:")
display(true_images.head(3))

print("\nUnique manifest sample:")
display(unique_manifest.head(3))

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/twincheck")

OUTPUT_DIR = BASE / "features/vision/verite/image_twins"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Select unique images associated with the true label
true_images = metadata[metadata["label"] == "true"].drop_duplicates(
    subset=["resolved_image_path"]
).copy()

# Merge using the shared resolved_image_path column
true_manifest = true_images.merge(
    unique_manifest[["image_id", "resolved_image_path", "split"]],
    on="resolved_image_path",
    how="left",
    validate="one_to_one"
)

# Verify
true_manifest["file_exists"] = true_manifest[
    "resolved_image_path"
].apply(lambda p: Path(p).exists())

print("Unique true images:", len(true_manifest))
print("Missing split assignments:", true_manifest["split"].isna().sum())
print("Missing image files:", (~true_manifest["file_exists"]).sum())

print("\nSplit distribution:")
print(true_manifest["split"].value_counts(dropna=False))

# Save
output_path = OUTPUT_DIR / "verite_true_image_twin_manifest.csv"
true_manifest.to_csv(output_path, index=False)

print("\nSaved:", output_path)

In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from diffusers import StableDiffusionPipeline
from tqdm.auto import tqdm
import gc
import json

# -----------------------------
# 1. Paths and configuration
# -----------------------------
BASE = Path("/content/drive/MyDrive/twincheck")

MANIFEST = (
    BASE / "features/vision/verite/image_twins/"
    "verite_true_image_twin_manifest.csv"
)

OUTPUT_DIR = BASE / "data/copies/vision/verite"
TWIN_DIR = OUTPUT_DIR / "vae_twins"
REPORT_DIR = BASE / "features/vision/verite/image_twins"

TWIN_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT = REPORT_DIR / "verite_vae_twin_checkpoint.csv"
REPORT = REPORT_DIR / "verite_vae_twin_report.json"

assert torch.cuda.is_available(), "GPU unavailable. Stop here."

device = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

# -----------------------------
# 2. Load manifest and resume
# -----------------------------
df = pd.read_csv(MANIFEST)

if CHECKPOINT.exists():
    completed_df = pd.read_csv(CHECKPOINT)
    completed_ids = set(
        completed_df.loc[
            completed_df["status"] == "success", "image_id"
        ].astype(str)
    )
else:
    completed_df = pd.DataFrame()
    completed_ids = set()

print("Total source images:", len(df))
print("Already completed:", len(completed_ids))

# -----------------------------
# 3. Load Stable Diffusion VAE
# -----------------------------
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None
)

vae = pipe.vae.to(device)
vae.eval()

# Release unused pipeline components
del pipe
gc.collect()
torch.cuda.empty_cache()

print("VAE loaded successfully.")

# -----------------------------
# 4. Generate VAE twins
# -----------------------------
results = completed_df.to_dict("records") if not completed_df.empty else []

for _, row in tqdm(df.iterrows(), total=len(df), desc="VERITE VAE twins"):

    image_id = str(row["image_id"])

    if image_id in completed_ids:
        continue

    source_path = Path(row["resolved_image_path"])
    output_path = TWIN_DIR / f"{Path(image_id).stem}_V.jpg"

    try:
        with Image.open(source_path) as im:
            original = im.convert("RGB")

        # Resize to dimensions divisible by 8
        width = max(64, (original.width // 8) * 8)
        height = max(64, (original.height // 8) * 8)

        resized = original.resize((width, height), Image.Resampling.LANCZOS)

        image_array = np.asarray(resized).astype(np.float32) / 127.5 - 1.0
        image_tensor = torch.from_numpy(
            image_array
        ).permute(2, 0, 1).unsqueeze(0).to(
            device=device, dtype=torch.float16
        )

        with torch.inference_mode():
            latent = vae.encode(image_tensor).latent_dist.mode()
            reconstructed = vae.decode(
                latent / vae.config.scaling_factor
            ).sample

        reconstructed = (
            (reconstructed.float().cpu().squeeze(0)
             .permute(1, 2, 0).numpy() + 1) * 127.5
        ).clip(0, 255).astype(np.uint8)

        twin = Image.fromarray(reconstructed)
        twin.save(output_path, "JPEG", quality=85)

        results.append({
            "image_id": image_id,
            "source_path": str(source_path),
            "twin_path": str(output_path),
            "split": row["split"],
            "status": "success"
        })

    except Exception as e:
        results.append({
            "image_id": image_id,
            "source_path": str(source_path),
            "twin_path": "",
            "split": row["split"],
            "status": "failed",
            "error": str(e)
        })

    # Save progress after every image
    pd.DataFrame(results).to_csv(CHECKPOINT, index=False)

    if len(results) % 10 == 0:
        torch.cuda.empty_cache()

# -----------------------------
# 5. Final report
# -----------------------------
results_df = pd.DataFrame(results)

report = {
    "total_source_images": len(df),
    "successful_twins": int((results_df["status"] == "success").sum()),
    "failed_twins": int((results_df["status"] == "failed").sum()),
    "output_directory": str(TWIN_DIR),
    "checkpoint": str(CHECKPOINT),
    "method": "Stable Diffusion VAE encode-decode",
    "jpeg_quality": 85
}

with open(REPORT, "w") as f:
    json.dump(report, f, indent=4)

print("\nVAE TWIN GENERATION FINISHED")
print(json.dumps(report, indent=4))

# Release GPU memory
del vae
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch
import numpy as np
import pandas as pd
import gc
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, SiglipModel

BASE = Path("/content/drive/MyDrive/twincheck")
DEVICE = "cuda"

# Paths
CHECKPOINT = BASE / "features/vision/verite/image_twins/verite_vae_twin_checkpoint.csv"
ORIGINAL_EMBEDDINGS = BASE / "features/vision/verite/siglip2/verified_image_extraction/verite_siglip2_image_projected_embeddings.npy"
ORIGINAL_MANIFEST = BASE / "features/vision/verite/siglip2/verified_image_extraction/verite_siglip2_image_projected_embeddings_manifest.csv"

OUT_DIR = BASE / "features/vision/verite/image_twins"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load successful twin records
twins = pd.read_csv(CHECKPOINT)
twins = twins[twins["status"] == "success"].copy()

original_embeddings = np.load(ORIGINAL_EMBEDDINGS)
original_manifest = pd.read_csv(ORIGINAL_MANIFEST)

print("Successful twins:", len(twins))
print("Original embeddings:", original_embeddings.shape)

# Match originals by image ID
original_manifest = original_manifest.reset_index(drop=True)
original_manifest["embedding_index"] = np.arange(len(original_manifest))

twins = twins.merge(
    original_manifest[["image_id", "embedding_index"]],
    on="image_id",
    how="left",
    validate="one_to_one"
)

assert twins["embedding_index"].notna().all(), "Some original embeddings are missing."

# Load the same SigLIP2 model
MODEL_ID = "google/siglip2-so400m-patch14-384"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = SiglipModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
).to(DEVICE)

model.eval()

# Extract twin embeddings
all_embeddings = []
batch_size = 16

for start in tqdm(range(0, len(twins), batch_size), desc="Twin embeddings"):
    batch = twins.iloc[start:start + batch_size]

    images = [
        Image.open(path).convert("RGB")
        for path in batch["twin_path"]
    ]

    inputs = processor(
        images=images,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.inference_mode():
        features = model.get_image_features(**inputs).pooler_output
        features = torch.nn.functional.normalize(features.float(), dim=-1)

    all_embeddings.append(features.cpu().numpy())

    del images, inputs, features

twin_embeddings = np.concatenate(all_embeddings, axis=0)

# Compare against original embeddings
original_indices = twins["embedding_index"].astype(int).to_numpy()
matched_originals = original_embeddings[original_indices]

similarities = np.sum(
    matched_originals * twin_embeddings,
    axis=1
)

twins["siglip_cosine_similarity"] = similarities
twins["similarity_pass"] = similarities >= 0.92

# Save embeddings and similarity report
np.save(
    OUT_DIR / "verite_vae_twin_siglip2_embeddings.npy",
    twin_embeddings
)

twins.to_csv(
    OUT_DIR / "verite_vae_twin_similarity_report.csv",
    index=False
)

print("\nSIMILARITY RESULTS")
print("Twins evaluated:", len(twins))
print("Passed (>= 0.92):", int(twins["similarity_pass"].sum()))
print("Below threshold:", int((~twins["similarity_pass"]).sum()))
print("Mean similarity:", round(float(np.mean(similarities)), 4))
print("Minimum similarity:", round(float(np.min(similarities)), 4))
print("Maximum similarity:", round(float(np.max(similarities)), 4))

# Release GPU memory
del model, processor
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

BASE = Path("/content/drive/MyDrive/twincheck")

report_path = (
    BASE / "features/vision/verite/image_twins/"
    "verite_vae_twin_similarity_report.csv"
)

df = pd.read_csv(report_path)

# Inspect examples across similarity levels
df = df.sort_values("siglip_cosine_similarity")

indices = [
    0, 1, 2, 10, 50, 100,
    150, 200, 250, 300, 330, 337
]

selected = df.iloc[indices].reset_index(drop=True)

canvas = Image.new("RGB", (900, 12 * 190), "white")
draw = ImageDraw.Draw(canvas)

for i, row in selected.iterrows():

    original = Image.open(row["source_path"]).convert("RGB")
    twin = Image.open(row["twin_path"]).convert("RGB")

    original.thumbnail((400, 145))
    twin.thumbnail((400, 145))

    y = i * 190

    canvas.paste(original, (0, y + 35))
    canvas.paste(twin, (450, y + 35))

    draw.text(
        (5, y + 5),
        f'{row["image_id"]} | Similarity: {row["siglip_cosine_similarity"]:.4f}',
        fill="black"
    )
    draw.text((150, y + 20), "Original", fill="black")
    draw.text((600, y + 20), "VAE Twin", fill="black")

display(canvas)

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from diffusers import AutoencoderKL
from transformers import AutoProcessor, SiglipModel
from tqdm.auto import tqdm
import gc

BASE = Path("/content/drive/MyDrive/twincheck")
DEVICE = "cuda"

assert torch.cuda.is_available()

# Paths
MANIFEST = BASE / "features/vision/verite/image_twins/verite_true_image_twin_manifest.csv"

ORIGINAL_EMB = BASE / "features/vision/verite/siglip2/verified_image_extraction/verite_siglip2_image_projected_embeddings.npy"

ORIGINAL_MANIFEST = BASE / "features/vision/verite/siglip2/verified_image_extraction/verite_siglip2_image_projected_embeddings_manifest.csv"

OUTPUT = BASE / "data/copies/vision/verite/vae_twins_corrected_pilot"
OUTPUT.mkdir(parents=True, exist_ok=True)

# Select five representative images
df = pd.read_csv(MANIFEST).head(5).copy()

original_embeddings = np.load(ORIGINAL_EMB)
embedding_manifest = pd.read_csv(ORIGINAL_MANIFEST)

embedding_manifest = embedding_manifest.reset_index(drop=True)
embedding_manifest["embedding_index"] = np.arange(len(embedding_manifest))

df = df.merge(
    embedding_manifest[["image_id", "embedding_index"]],
    on="image_id",
    how="left",
    validate="one_to_one"
)

assert df["embedding_index"].notna().all()

print("GPU:", torch.cuda.get_device_name(0))
print("Pilot images:", len(df))

# Load only the VAE, not the full diffusion pipeline
vae = AutoencoderKL.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    subfolder="vae",
    torch_dtype=torch.float16
).to(DEVICE)

vae.eval()

# Correct VAE encode-decode
for _, row in tqdm(df.iterrows(), total=len(df)):

    with Image.open(row["resolved_image_path"]) as im:
        original = im.convert("RGB")

    width = max(64, (original.width // 8) * 8)
    height = max(64, (original.height // 8) * 8)

    resized = original.resize(
        (width, height),
        Image.Resampling.LANCZOS
    )

    array = np.asarray(resized).astype(np.float32) / 127.5 - 1.0

    tensor = torch.from_numpy(array).permute(2, 0, 1)
    tensor = tensor.unsqueeze(0).to(DEVICE, dtype=torch.float16)

    with torch.inference_mode():
        latent = vae.encode(tensor).latent_dist.mode()

        # IMPORTANT: direct unscaled VAE latent is decoded directly
        reconstruction = vae.decode(latent).sample

    image = (
        (reconstruction.float().cpu().squeeze(0)
         .permute(1, 2, 0).numpy() + 1) * 127.5
    ).clip(0, 255).astype(np.uint8)

    Image.fromarray(image).save(
        OUTPUT / f"{Path(row['image_id']).stem}_V_corrected.jpg",
        quality=85
    )

    del tensor, latent, reconstruction
    torch.cuda.empty_cache()

# Release VAE before loading SigLIP
del vae
gc.collect()
torch.cuda.empty_cache()

# Load SigLIP2
processor = AutoProcessor.from_pretrained(
    "google/siglip2-so400m-patch14-384"
)

model = SiglipModel.from_pretrained(
    "google/siglip2-so400m-patch14-384",
    torch_dtype=torch.float16
).to(DEVICE)

model.eval()

corrected_images = []

for _, row in df.iterrows():
    path = OUTPUT / f"{Path(row['image_id']).stem}_V_corrected.jpg"
    corrected_images.append(Image.open(path).convert("RGB"))

inputs = processor(
    images=corrected_images,
    return_tensors="pt"
).to(DEVICE)

with torch.inference_mode():
    features = model.get_image_features(**inputs).pooler_output
    features = torch.nn.functional.normalize(features.float(), dim=-1)

twin_embeddings = features.cpu().numpy()

indices = df["embedding_index"].astype(int).to_numpy()
originals = original_embeddings[indices]

similarities = np.sum(originals * twin_embeddings, axis=1)

df["corrected_similarity"] = similarities
df["similarity_pass"] = similarities >= 0.92

df.to_csv(
    BASE / "features/vision/verite/image_twins/"
    "verite_corrected_vae_pilot_report.csv",
    index=False
)

print("\nCORRECTED PILOT RESULTS")
print(df[["image_id", "corrected_similarity", "similarity_pass"]].to_string(index=False))
print("\nMean similarity:", round(float(similarities.mean()), 4))
print("Passed >= 0.92:", int((similarities >= 0.92).sum()), "/ 5")

del model, processor, inputs, features
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from diffusers import AutoencoderKL
from tqdm.auto import tqdm
import gc

BASE = Path("/content/drive/MyDrive/twincheck")

MANIFEST = BASE / "features/vision/verite/image_twins/verite_true_image_twin_manifest.csv"

OUTPUT = BASE / "data/copies/vision/verite/vae_twins_corrected"
OUTPUT.mkdir(parents=True, exist_ok=True)

PILOT = BASE / "data/copies/vision/verite/vae_twins_corrected_pilot"

REPORT = BASE / "features/vision/verite/image_twins/verite_corrected_vae_generation_checkpoint.csv"

assert torch.cuda.is_available()

df = pd.read_csv(MANIFEST)

results = []

# Resume from existing successful outputs
for _, row in df.iterrows():
    output_path = OUTPUT / f"{Path(row['image_id']).stem}_V.jpg"
    pilot_path = PILOT / f"{Path(row['image_id']).stem}_V_corrected.jpg"

    if output_path.exists():
        results.append({
            "image_id": row["image_id"],
            "twin_path": str(output_path),
            "split": row["split"],
            "status": "success"
        })
    elif pilot_path.exists():
        results.append({
            "image_id": row["image_id"],
            "twin_path": str(pilot_path),
            "split": row["split"],
            "status": "success"
        })

completed = {r["image_id"] for r in results}

print("GPU:", torch.cuda.get_device_name(0))
print("Total images:", len(df))
print("Already generated:", len(completed))

vae = AutoencoderKL.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    subfolder="vae",
    torch_dtype=torch.float16
).to("cuda")

vae.eval()

for _, row in tqdm(df.iterrows(), total=len(df), desc="Corrected VAE twins"):

    image_id = row["image_id"]

    if image_id in completed:
        continue

    output_path = OUTPUT / f"{Path(image_id).stem}_V.jpg"

    try:
        with Image.open(row["resolved_image_path"]) as im:
            original = im.convert("RGB")

        width = max(64, (original.width // 8) * 8)
        height = max(64, (original.height // 8) * 8)

        resized = original.resize(
            (width, height),
            Image.Resampling.LANCZOS
        )

        array = np.asarray(resized).astype(np.float32) / 127.5 - 1

        tensor = torch.from_numpy(array).permute(2, 0, 1)
        tensor = tensor.unsqueeze(0).to("cuda", dtype=torch.float16)

        with torch.inference_mode():
            latent = vae.encode(tensor).latent_dist.mode()
            reconstruction = vae.decode(latent).sample

        image = (
            (reconstruction.float().cpu().squeeze(0)
             .permute(1, 2, 0).numpy() + 1) * 127.5
        ).clip(0, 255).astype(np.uint8)

        Image.fromarray(image).save(output_path, "JPEG", quality=85)

        results.append({
            "image_id": image_id,
            "twin_path": str(output_path),
            "split": row["split"],
            "status": "success"
        })

    except Exception as e:
        results.append({
            "image_id": image_id,
            "twin_path": "",
            "split": row["split"],
            "status": "failed",
            "error": str(e)
        })

    pd.DataFrame(results).to_csv(REPORT, index=False)

    del tensor
    torch.cuda.empty_cache()

print("\nGeneration complete")
print(pd.Series([r["status"] for r in results]).value_counts())
print("Checkpoint:", REPORT)

del vae
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import pandas as pd

checkpoint_path = (
    "/content/drive/MyDrive/twincheck/features/vision/verite/"
    "image_twins/verite_corrected_vae_generation_checkpoint.csv"
)

df = pd.read_csv(checkpoint_path)

print("Checkpoint shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nGeneration status:")
print(df.iloc[:, -1].value_counts(dropna=False))

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModel

# Paths
base = "/content/drive/MyDrive/twincheck"

checkpoint_path = (
    f"{base}/features/vision/verite/image_twins/"
    "verite_corrected_vae_generation_checkpoint.csv"
)

original_manifest_path = (
    f"{base}/features/vision/verite/siglip2/"
    "verified_image_extraction/"
    "verite_siglip2_image_projected_embeddings_manifest.csv"
)

original_embeddings_path = (
    f"{base}/features/vision/verite/siglip2/"
    "verified_image_extraction/"
    "verite_siglip2_image_projected_embeddings.npy"
)

output_dir = f"{base}/features/vision/verite/image_twins"
os.makedirs(output_dir, exist_ok=True)

# Load data
checkpoint = pd.read_csv(checkpoint_path)
checkpoint = checkpoint[checkpoint["status"] == "success"].copy()

original_manifest = pd.read_csv(original_manifest_path)
original_embeddings = np.load(original_embeddings_path)

original_ids = original_manifest["image_id"].astype(str).tolist()
original_index = {image_id: i for i, image_id in enumerate(original_ids)}

assert len(checkpoint) == 338
assert all(
    image_id in original_index
    for image_id in checkpoint["image_id"].astype(str)
)

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_name = "google/siglip2-so400m-patch14-384"
processor = AutoProcessor.from_pretrained(model_name)

model = AutoModel.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

# Extract twin embeddings
twin_embeddings = []
batch_size = 16
twin_paths = checkpoint["twin_path"].tolist()

with torch.inference_mode():
    for start in tqdm(range(0, len(twin_paths), batch_size),
                      desc="Twin SigLIP2 embeddings"):

        paths = twin_paths[start:start + batch_size]
        images = [Image.open(path).convert("RGB") for path in paths]

        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model.get_image_features(**inputs)

        # Handle Transformers output format
        if hasattr(outputs, "pooler_output"):
            features = outputs.pooler_output
        elif isinstance(outputs, torch.Tensor):
            features = outputs
        else:
            raise TypeError(f"Unexpected output type: {type(outputs)}")

        features = F.normalize(features.float(), p=2, dim=-1)
        twin_embeddings.append(features.cpu().numpy())

        for image in images:
            image.close()

twin_embeddings = np.concatenate(twin_embeddings, axis=0)

# Calculate cosine similarity
similarities = []

for row_number, image_id in enumerate(checkpoint["image_id"].astype(str)):
    original_vector = original_embeddings[original_index[image_id]]
    twin_vector = twin_embeddings[row_number]

    original_vector = original_vector / np.linalg.norm(original_vector)
    twin_vector = twin_vector / np.linalg.norm(twin_vector)

    similarities.append(float(np.dot(original_vector, twin_vector)))

checkpoint["siglip2_similarity"] = similarities
checkpoint["accepted"] = checkpoint["siglip2_similarity"] >= 0.92

# Save
report_path = f"{output_dir}/verite_corrected_vae_twin_similarity_report.csv"
embedding_path = f"{output_dir}/verite_corrected_vae_twin_siglip2_embeddings.npy"

checkpoint.to_csv(report_path, index=False)
np.save(embedding_path, twin_embeddings)

# Results
print("\nSIMILARITY EVALUATION COMPLETE")
print("Twins evaluated:", len(checkpoint))
print("Accepted (>= 0.92):", int(checkpoint["accepted"].sum()))
print("Rejected (< 0.92):", int((~checkpoint["accepted"]).sum()))
print("Mean similarity:", round(checkpoint["siglip2_similarity"].mean(), 4))
print("Minimum similarity:", round(checkpoint["siglip2_similarity"].min(), 4))
print("Maximum similarity:", round(checkpoint["siglip2_similarity"].max(), 4))

print("\nReport:", report_path)
print("Embeddings:", embedding_path)

display(checkpoint[
    ["image_id", "split", "siglip2_similarity", "accepted"]
].head(10))

# Release GPU memory
del model, processor
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\nGPU memory released.")

In [ ]:
import pandas as pd
import os

base = "/content/drive/MyDrive/twincheck"

report_path = (
    f"{base}/features/vision/verite/image_twins/"
    "verite_corrected_vae_twin_similarity_report.csv"
)

output_path = (
    f"{base}/features/vision/verite/image_twins/"
    "verite_accepted_vae_twin_manifest.csv"
)

df = pd.read_csv(report_path)

accepted = df[df["accepted"] == True].copy()
accepted.to_csv(output_path, index=False)

print("ACCEPTED TWIN MANIFEST CREATED")
print("--------------------------------")
print("Accepted twins:", len(accepted))
print("\nSplit distribution:")
print(accepted["split"].value_counts())

print("\nSaved to:")
print(output_path)

print("\nFirst 10 accepted twins:")
display(accepted[
    ["image_id", "twin_path", "split", "siglip2_similarity"]
].head(10))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import random
import os

manifest_path = (
    "/content/drive/MyDrive/twincheck/features/vision/verite/"
    "image_twins/verite_accepted_vae_twin_manifest.csv"
)

original_image_dir = (
    "/content/drive/MyDrive/twincheck/data/raw/verite/"
    "images_download/extracted/images"
)

df = pd.read_csv(manifest_path)

random.seed(42)
sample = df.sample(n=30, random_state=42).reset_index(drop=True)

fig, axes = plt.subplots(10, 6, figsize=(18, 28))

for i, row in sample.iterrows():
    image_id = row["image_id"]
    original_path = os.path.join(original_image_dir, image_id)
    twin_path = row["twin_path"]

    original = Image.open(original_path).convert("RGB")
    twin = Image.open(twin_path).convert("RGB")

    r = i // 3
    c = (i % 3) * 2

    axes[r, c].imshow(original)
    axes[r, c].set_title(f"Original: {image_id}", fontsize=8)
    axes[r, c].axis("off")

    axes[r, c + 1].imshow(twin)
    axes[r, c + 1].set_title(
        f"Twin: similarity {row['siglip2_similarity']:.3f}",
        fontsize=8
    )
    axes[r, c + 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd

base = "/content/drive/MyDrive/twincheck"

folders = [
    f"{base}/data/processed/vision/m4fc",
    f"{base}/data/raw/m4fc",
]

print("M4FC METADATA FILE AUDIT")
print("=" * 60)

for folder in folders:
    print(f"\nFolder: {folder}")

    if not os.path.exists(folder):
        print("Folder not found")
        continue

    for root, dirs, files in os.walk(folder):
        dirs[:] = [d for d in dirs if d not in ["images", "__pycache__"]]

        for filename in files:
            if filename.lower().endswith((".csv", ".json", ".jsonl")):
                path = os.path.join(root, filename)
                print(f"\nFile: {path}")

                try:
                    if filename.endswith(".csv"):
                        df = pd.read_csv(path)
                    elif filename.endswith(".jsonl"):
                        df = pd.read_json(path, lines=True)
                    else:
                        df = pd.read_json(path)

                    print("Shape:", df.shape)
                    print("Columns:", df.columns.tolist())
                    display(df.head(3))

                    for col in df.columns:
                        if any(word in col.lower() for word in
                               ["label", "class", "truth", "veracity"]):
                            print(f"\nPossible label column: {col}")
                            print(df[col].value_counts(dropna=False).head(15))

                except Exception as e:
                    print("Could not read:", str(e))

print("\nAudit complete.")

In [ ]:
import os
import pandas as pd

base = "/content/drive/MyDrive/twincheck"

print("SEARCHING FOR M4FC CLAIM/LABEL FILES")
print("=" * 60)

for root, dirs, files in os.walk(base):
    dirs[:] = [d for d in dirs if d not in [
        ".git", "__pycache__", "vae_twins",
        "vae_twins_corrected", "verified_image_extraction"
    ]]

    for filename in files:
        if filename.lower().endswith((".csv", ".json", ".jsonl")):
            if "m4fc" in filename.lower() or "claim" in filename.lower():
                path = os.path.join(root, filename)

                try:
                    if filename.endswith(".csv"):
                        df = pd.read_csv(path)
                    elif filename.endswith(".jsonl"):
                        df = pd.read_json(path, lines=True)
                    else:
                        df = pd.read_json(path)

                    print("\nFILE:", path)
                    print("Shape:", df.shape)
                    print("Columns:", df.columns.tolist())

                    label_cols = [
                        c for c in df.columns
                        if any(x in c.lower() for x in
                               ["label", "veracity", "truth", "class"])
                    ]

                    for col in label_cols:
                        print(f"\n{col} value counts:")
                        print(df[col].value_counts(dropna=False))

                except Exception as e:
                    print("Could not read:", path, str(e))

print("\nSearch complete.")

In [ ]:
import os
import pandas as pd

base = "/content/drive/MyDrive/twincheck"

source = os.path.join(
    base,
    "features/vision/m4fc/siglip2/verified_text_extraction/"
    "m4fc_arabic_claim_manifest.csv"
)

output_dir = os.path.join(
    base, "features/vision/m4fc/image_twins"
)
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(source)

# Select records identified as true captions or true
true_records = df[
    df["orig_label"].isin(["true_caption", "true"])
].copy()

# Keep only records whose image files are available
true_records = true_records[
    true_records["image_available"].astype(str).str.lower().isin(
        ["true", "1"]
    )
].copy()

# Remove duplicate references to the same image
true_images = true_records.drop_duplicates(
    subset=["image_id"]
).copy()

manifest_path = os.path.join(
    output_dir, "m4fc_true_image_twin_manifest.csv"
)

true_images.to_csv(manifest_path, index=False)

print("M4FC TRUE-IMAGE MANIFEST")
print("=" * 55)
print("Selected claim records:", len(true_records))
print("Unique available images:", len(true_images))
print("\nOriginal label counts:")
print(true_records["orig_label"].value_counts())
print("\nSplit distribution:")
print(true_images["split"].value_counts(dropna=False))
print("\nMissing image paths:",
      true_images["resolved_image_path"].isna().sum())
print("\nManifest saved to:")
print(manifest_path)

print("\nSample rows:")
display(
    true_images[
        ["image_id", "orig_label", "split", "resolved_image_path"]
    ].head()
)

In [ ]:
import os
import gc
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from diffusers import AutoencoderKL
from torchvision import transforms

# Paths
base = "/content/drive/MyDrive/twincheck"

manifest_path = os.path.join(
    base, "features/vision/m4fc/image_twins/"
    "m4fc_true_image_twin_manifest.csv"
)

output_dir = os.path.join(
    base, "data/copies/vision/m4fc/vae_twins_corrected"
)
os.makedirs(output_dir, exist_ok=True)

checkpoint_path = os.path.join(
    base, "features/vision/m4fc/image_twins/"
    "m4fc_corrected_vae_generation_checkpoint.csv"
)

# Load manifest
df = pd.read_csv(manifest_path)

# Resume from checkpoint if it exists
if os.path.exists(checkpoint_path):
    checkpoint = pd.read_csv(checkpoint_path)
    completed = set(
        checkpoint.loc[
            checkpoint["status"] == "success", "image_id"
        ].astype(str)
    )
else:
    checkpoint = pd.DataFrame(
        columns=["image_id", "twin_path", "split", "status"]
    )
    completed = set()

print("Images in manifest:", len(df))
print("Already completed:", len(completed))
print("Remaining:", len(df) - len(completed))

# Load VAE
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

vae = AutoencoderKL.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    subfolder="vae",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

vae.eval()

new_rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    image_id = str(row["image_id"])

    if image_id in completed:
        continue

    try:
        image = Image.open(
            row["resolved_image_path"]
        ).convert("RGB")

        # Resize dimensions down to multiples of 8
        width, height = image.size
        new_width = max(8, (width // 8) * 8)
        new_height = max(8, (height // 8) * 8)

        if (new_width, new_height) != image.size:
            image = image.resize(
                (new_width, new_height),
                Image.Resampling.LANCZOS
            )

        tensor = transforms.ToTensor()(image)
        tensor = tensor * 2.0 - 1.0
        tensor = tensor.unsqueeze(0).to(
            device=device,
            dtype=next(vae.parameters()).dtype
        )

        with torch.inference_mode():
            latent = vae.encode(
                tensor
            ).latent_dist.mode()

            # Correct direct VAE reconstruction:
            # Do NOT divide latent by scaling_factor.
            reconstruction = vae.decode(latent).sample

            reconstruction = (
                reconstruction / 2 + 0.5
            ).clamp(0, 1)

        reconstructed_image = transforms.ToPILImage()(
            reconstruction[0].float().cpu()
        )

        twin_path = os.path.join(
            output_dir, f"{image_id}_V.jpg"
        )

        reconstructed_image.save(
            twin_path, "JPEG", quality=85
        )

        new_rows.append({
            "image_id": image_id,
            "twin_path": twin_path,
            "split": row["split"],
            "status": "success"
        })

    except Exception as e:
        new_rows.append({
            "image_id": image_id,
            "twin_path": "",
            "split": row["split"],
            "status": f"failed: {str(e)}"
        })

    # Save checkpoint after every image
    if new_rows:
        checkpoint = pd.concat(
            [checkpoint, pd.DataFrame(new_rows)],
            ignore_index=True
        )
        checkpoint.to_csv(checkpoint_path, index=False)
        completed.update(
            r["image_id"] for r in new_rows
            if r["status"] == "success"
        )
        new_rows = []

# Final summary
checkpoint = pd.read_csv(checkpoint_path)

print("\nVAE GENERATION SUMMARY")
print("=" * 45)
print("Total records:", len(checkpoint))
print("Successful:", (checkpoint["status"] == "success").sum())
print("Failed:", (checkpoint["status"] != "success").sum())
print("Output folder:", output_dir)
print("Checkpoint:", checkpoint_path)

del vae
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
import os
import gc
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModel

base = "/content/drive/MyDrive/twincheck"

manifest_path = os.path.join(
    base, "features/vision/m4fc/image_twins/"
    "m4fc_true_image_twin_manifest.csv"
)

checkpoint_path = os.path.join(
    base, "features/vision/m4fc/image_twins/"
    "m4fc_corrected_vae_generation_checkpoint.csv"
)

original_manifest_path = os.path.join(
    base, "features/vision/m4fc/siglip2/"
    "m4fc_siglip2_image_projected_embeddings_manifest.csv"
)

original_embeddings_path = os.path.join(
    base, "features/vision/m4fc/siglip2/"
    "m4fc_siglip2_image_projected_embeddings.npy"
)

output_dir = os.path.join(
    base, "features/vision/m4fc/image_twins"
)

manifest = pd.read_csv(manifest_path)
checkpoint = pd.read_csv(checkpoint_path)
original_manifest = pd.read_csv(original_manifest_path)
original_embeddings = np.load(original_embeddings_path)

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "google/siglip2-so400m-patch14-384"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

# Map original embeddings by image ID
original_index = {
    str(image_id): i
    for i, image_id in enumerate(original_manifest["id"])
}

results = []
twin_embeddings = []

successful = checkpoint[
    checkpoint["status"] == "success"
]

for _, row in tqdm(successful.iterrows(), total=len(successful)):
    image_id = str(row["image_id"])

    original_row = manifest[
        manifest["image_id"].astype(str) == image_id
    ].iloc[0]

    original_path = original_row["resolved_image_path"]
    twin_path = row["twin_path"]

    original_image = Image.open(original_path).convert("RGB")
    twin_image = Image.open(twin_path).convert("RGB")

    inputs = processor(
        images=[original_image, twin_image],
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        features = model.get_image_features(**inputs)

        # Handle model output consistently
        if hasattr(features, "pooler_output"):
            features = features.pooler_output

        features = torch.nn.functional.normalize(
            features.float(), dim=-1
        )

    features = features.cpu().numpy()

    similarity = float(
        np.dot(features[0], features[1])
    )

    results.append({
        "image_id": image_id,
        "original_path": original_path,
        "twin_path": twin_path,
        "split": row["split"],
        "siglip2_cosine_similarity": similarity,
        "accepted": similarity >= 0.92
    })

    twin_embeddings.append(features[1])

# Save results
results_df = pd.DataFrame(results)

report_path = os.path.join(
    output_dir, "m4fc_corrected_vae_twin_similarity_report.csv"
)
results_df.to_csv(report_path, index=False)

embedding_path = os.path.join(
    output_dir, "m4fc_corrected_vae_twin_siglip2_embeddings.npy"
)
np.save(embedding_path, np.vstack(twin_embeddings))

accepted = results_df[results_df["accepted"]]

accepted_path = os.path.join(
    output_dir, "m4fc_accepted_vae_twin_manifest.csv"
)

accepted.to_csv(accepted_path, index=False)

print("\nM4FC SIMILARITY SUMMARY")
print("=" * 45)
print("Evaluated:", len(results_df))
print("Accepted (>= 0.92):", len(accepted))
print("Rejected:", len(results_df) - len(accepted))
print("Mean similarity:", results_df[
    "siglip2_cosine_similarity"
].mean())
print("Minimum:", results_df[
    "siglip2_cosine_similarity"
].min())
print("Maximum:", results_df[
    "siglip2_cosine_similarity"
].max())

print("\nSaved report:", report_path)
print("Saved embeddings:", embedding_path)
print("Accepted manifest:", accepted_path)

del model, processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import random
import os

base = "/content/drive/MyDrive/twincheck"

manifest_path = os.path.join(
    base, "features/vision/m4fc/image_twins/"
    "m4fc_accepted_vae_twin_manifest.csv"
)

df = pd.read_csv(manifest_path)

sample = df.sample(
    n=min(30, len(df)),
    random_state=42
).reset_index(drop=True)

for i, row in sample.iterrows():
    original = Image.open(row["original_path"]).convert("RGB")
    twin = Image.open(row["twin_path"]).convert("RGB")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(original)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(twin)
    axes[1].set_title("VAE Twin")
    axes[1].axis("off")

    fig.suptitle(
        f'{row["image_id"]} | Similarity: '
        f'{row["siglip2_cosine_similarity"]:.3f}'
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import os
import cv2
import pandas as pd
from tqdm.auto import tqdm

base = "/content/drive/MyDrive/twincheck"

datasets = {
    "VERITE": {
        "accepted": os.path.join(
            base, "features/vision/verite/image_twins/",
            "verite_accepted_vae_twin_manifest.csv"
        ),
        "original": os.path.join(
            base, "features/vision/verite/image_twins/",
            "verite_true_image_twin_manifest.csv"
        )
    },
    "M4FC": {
        "accepted": os.path.join(
            base, "features/vision/m4fc/image_twins/",
            "m4fc_accepted_vae_twin_manifest.csv"
        ),
        "original": os.path.join(
            base, "features/vision/m4fc/image_twins/",
            "m4fc_true_image_twin_manifest.csv"
        )
    }
}

output_dir = os.path.join(
    base, "features/vision/face_exclusion"
)
os.makedirs(output_dir, exist_ok=True)

detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

all_results = []

for dataset, paths in datasets.items():

    accepted = pd.read_csv(paths["accepted"])
    original = pd.read_csv(paths["original"])

    # Get original image paths from the original manifest
    accepted = accepted.merge(
        original[["image_id", "resolved_image_path"]],
        on="image_id",
        how="left"
    )

    print(f"\n{dataset}: {len(accepted)} accepted images")

    for _, row in tqdm(accepted.iterrows(), total=len(accepted)):

        image_path = row["resolved_image_path"]
        image_id = str(row["image_id"])

        image = cv2.imread(str(image_path))

        if image is None:
            all_results.append({
                "dataset": dataset,
                "image_id": image_id,
                "image_path": image_path,
                "faces_detected": -1,
                "largest_face_area_ratio": None,
                "manual_review": "IMAGE_READ_ERROR"
            })
            continue

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        faces = detector.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        image_area = image.shape[0] * image.shape[1]

        ratios = [
            (w * h) / image_area
            for (x, y, w, h) in faces
        ]

        largest_ratio = max(ratios) if ratios else 0

        all_results.append({
            "dataset": dataset,
            "image_id": image_id,
            "image_path": image_path,
            "faces_detected": len(faces),
            "largest_face_area_ratio": largest_ratio,
            "manual_review": (
                "REVIEW_CANDIDATE"
                if largest_ratio >= 0.05
                else "NO_LARGE_FACE_DETECTED"
            )
        })

results = pd.DataFrame(all_results)

report_path = os.path.join(
    output_dir, "face_detection_review_candidates.csv"
)

results.to_csv(report_path, index=False)

print("\nFACE DETECTION SUMMARY")
print("=" * 45)
print(results.groupby(
    ["dataset", "manual_review"]
).size())

print("\nPotential face candidates:")
display(results[
    results["manual_review"] == "REVIEW_CANDIDATE"
])

print("\nReport saved:", report_path)

In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import math

base = "/content/drive/MyDrive/twincheck"

report_path = os.path.join(
    base, "features/vision/face_exclusion/",
    "face_detection_review_candidates.csv"
)

results = pd.read_csv(report_path)

candidates = results[
    results["manual_review"] == "REVIEW_CANDIDATE"
].copy()

for dataset in ["VERITE", "M4FC"]:

    subset = candidates[
        candidates["dataset"] == dataset
    ].reset_index(drop=True)

    cols = 4
    rows = math.ceil(len(subset) / cols)

    fig, axes = plt.subplots(
        rows, cols, figsize=(16, rows * 4)
    )
    axes = axes.flatten()

    for i, row in subset.iterrows():

        image = cv2.imread(row["image_path"])

        if image is not None:
            image = cv2.cvtColor(
                image, cv2.COLOR_BGR2RGB
            )

            gray = cv2.cvtColor(
                image, cv2.COLOR_RGB2GRAY
            )

            detector = cv2.CascadeClassifier(
                cv2.data.haarcascades +
                "haarcascade_frontalface_default.xml"
            )

            faces = detector.detectMultiScale(
                gray, 1.1, 5, minSize=(30, 30)
            )

            for (x, y, w, h) in faces:
                cv2.rectangle(
                    image, (x, y), (x+w, y+h),
                    (255, 0, 0), 3
                )

            axes[i].imshow(image)

        axes[i].set_title(
            f'{row["image_id"]}\n'
            f'Faces: {row["faces_detected"]} | '
            f'Area: {row["largest_face_area_ratio"]:.1%}'
        )
        axes[i].axis("off")

    for j in range(len(subset), len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        f"{dataset}: Face Detection Review Candidates",
        fontsize=16
    )
    plt.tight_layout()
    plt.show()



In [ ]:
import os
import pandas as pd

base = "/content/drive/MyDrive/twincheck"

# Images manually identified for exclusion
exclude_ids = {
    "VERITE": [
        "true_13.jpg",
        "true_24.jpg",
        "true_36.jpg",
        "true_59.jpg",
        "true_99.jpg",
        "true_103.jpg",
        "true_106.jpg",
        "true_117.jpg",
        "true_182.jpg",
        "true_278.jpg",
        "true_292.jpg",
        "true_311.jpg"
    ],
    "M4FC": [
        "m4fc_000088",
        "m4fc_000324",
        "m4fc_000663",
        "m4fc_000667"
    ]
}

configs = {
    "VERITE": {
        "input": os.path.join(
            base, "features/vision/verite/image_twins/",
            "verite_accepted_vae_twin_manifest.csv"
        ),
        "output": os.path.join(
            base, "features/vision/verite/image_twins/",
            "verite_face_filtered_vae_twin_manifest.csv"
        )
    },
    "M4FC": {
        "input": os.path.join(
            base, "features/vision/m4fc/image_twins/",
            "m4fc_accepted_vae_twin_manifest.csv"
        ),
        "output": os.path.join(
            base, "features/vision/m4fc/image_twins/",
            "m4fc_face_filtered_vae_twin_manifest.csv"
        )
    }
}

for dataset, paths in configs.items():

    df = pd.read_csv(paths["input"])

    excluded = set(exclude_ids[dataset])

    if dataset == "VERITE":
        # Match IDs such as true_13.jpg
        ids = df["image_id"].astype(str)
        mask = ids.isin(excluded)
    else:
        ids = df["image_id"].astype(str)
        mask = ids.isin(excluded)

    filtered = df[~mask].copy()

    df.loc[mask].to_csv(
        paths["output"].replace(
            "face_filtered_vae_twin_manifest.csv",
            "face_excluded_review_records.csv"
        ),
        index=False
    )

    filtered.to_csv(paths["output"], index=False)

    print(f"\n{dataset}")
    print("Before:", len(df))
    print("Excluded:", int(mask.sum()))
    print("Remaining:", len(filtered))
    print("Saved:", paths["output"])

In [ ]:

# ============================================================
# TWINCHECK — RUNTIME RECOVERY CELL
# Restores Drive, imports, paths, manifests and saved embeddings
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import json
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------
# 1. Project paths
# -----------------------------
PROJECT = Path("/content/drive/MyDrive/twincheck")

DATA = PROJECT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
COPIES = DATA / "copies" / "vision"
FEATURES = PROJECT / "features" / "vision"

# VERITE
VERITE_FEATURES = FEATURES / "verite"
VERITE_TWINS = VERITE_FEATURES / "image_twins"

VERITE_TRUE_MANIFEST = (
    VERITE_TWINS / "verite_true_image_twin_manifest.csv"
)
VERITE_ACCEPTED_MANIFEST = (
    VERITE_TWINS / "verite_accepted_vae_twin_manifest.csv"
)
VERITE_FACE_FILTERED = (
    VERITE_TWINS / "verite_face_filtered_vae_twin_manifest.csv"
)
VERITE_IMAGE_EMBEDDINGS = (
    VERITE_FEATURES / "siglip2" / "verified_image_extraction" /
    "verite_siglip2_image_projected_embeddings.npy"
)
VERITE_CAPTION_EMBEDDINGS = (
    VERITE_FEATURES / "siglip2" / "verified_caption_extraction" /
    "verite_siglip2_caption_projected_embeddings.npy"
)
VERITE_CLIP_EMBEDDINGS = (
    VERITE_FEATURES / "clip_vit_l14" / "verified_image_extraction" /
    "verite_clip_vit_l14_image_projected_embeddings.npy"
)

# M4FC
M4FC_FEATURES = FEATURES / "m4fc"
M4FC_TWINS = M4FC_FEATURES / "image_twins"

M4FC_ACCEPTED_MANIFEST = (
    M4FC_TWINS / "m4fc_accepted_vae_twin_manifest.csv"
)
M4FC_FACE_FILTERED = (
    M4FC_TWINS / "m4fc_face_filtered_vae_twin_manifest.csv"
)
M4FC_IMAGE_EMBEDDINGS = (
    M4FC_FEATURES / "siglip2" /
    "m4fc_siglip2_image_projected_embeddings.npy"
)
M4FC_CLAIM_EMBEDDINGS = (
    M4FC_FEATURES / "siglip2" / "verified_text_extraction" /
    "m4fc_siglip2_arabic_claim_projected_embeddings.npy"
)

# GenImage
GENIMAGE_FEATURES = FEATURES / "genimage"
GENIMAGE_SIGLIP_EMBEDDINGS = (
    GENIMAGE_FEATURES / "genimage_siglip2_image_projected_embeddings.npy"
)
GENIMAGE_CLIP_EMBEDDINGS = (
    GENIMAGE_FEATURES / "genimage_clip_vit_l14_image_projected_embeddings.npy"
)
GENIMAGE_MANIFEST = (
    GENIMAGE_FEATURES / "genimage_image_embedding_manifest.csv"
)

# Source data for Himasri's SDXL source proposal
M4FC_AR_TEST_JSONL = PROCESSED / "ar" / "m4fc_ar_test.jsonl"
VERITE_JSONL = PROCESSED / "vision" / "verite.jsonl"

# -----------------------------
# 2. Check project directory
# -----------------------------
assert PROJECT.exists(), f"Project folder not found: {PROJECT}"
print("\nProject folder:", PROJECT)

# -----------------------------
# 3. Load saved manifests
# -----------------------------
def load_csv_if_exists(name, path):
    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded {name}: {df.shape}")
        return df
    print(f"NOT FOUND: {name} -> {path}")
    return None

verite_true_manifest = load_csv_if_exists(
    "VERITE true manifest", VERITE_TRUE_MANIFEST
)
verite_accepted_manifest = load_csv_if_exists(
    "VERITE accepted twins", VERITE_ACCEPTED_MANIFEST
)
verite_face_filtered_manifest = load_csv_if_exists(
    "VERITE face-filtered twins", VERITE_FACE_FILTERED
)
m4fc_accepted_manifest = load_csv_if_exists(
    "M4FC accepted twins", M4FC_ACCEPTED_MANIFEST
)
m4fc_face_filtered_manifest = load_csv_if_exists(
    "M4FC face-filtered twins", M4FC_FACE_FILTERED
)
genimage_manifest = load_csv_if_exists(
    "GenImage manifest", GENIMAGE_MANIFEST
)

# -----------------------------
# 4. Load saved feature arrays
# -----------------------------
def load_npy_if_exists(name, path):
    if path.exists():
        arr = np.load(path, mmap_mode="r")
        print(f"Loaded {name}: shape={arr.shape}, dtype={arr.dtype}")
        return arr
    print(f"NOT FOUND: {name} -> {path}")
    return None

verite_siglip_image_embeddings = load_npy_if_exists(
    "VERITE SigLIP2 image embeddings", VERITE_IMAGE_EMBEDDINGS
)
verite_siglip_caption_embeddings = load_npy_if_exists(
    "VERITE SigLIP2 caption embeddings", VERITE_CAPTION_EMBEDDINGS
)
verite_clip_image_embeddings = load_npy_if_exists(
    "VERITE CLIP image embeddings", VERITE_CLIP_EMBEDDINGS
)
m4fc_siglip_image_embeddings = load_npy_if_exists(
    "M4FC SigLIP2 image embeddings", M4FC_IMAGE_EMBEDDINGS
)
m4fc_siglip_claim_embeddings = load_npy_if_exists(
    "M4FC Arabic claim embeddings", M4FC_CLAIM_EMBEDDINGS
)
genimage_siglip_embeddings = load_npy_if_exists(
    "GenImage SigLIP2 embeddings", GENIMAGE_SIGLIP_EMBEDDINGS
)
genimage_clip_embeddings = load_npy_if_exists(
    "GenImage CLIP embeddings", GENIMAGE_CLIP_EMBEDDINGS
)

# -----------------------------
# 5. Check important source files
# -----------------------------
print("\n--- Source file checks ---")
for name, path in [
    ("M4FC Arabic test JSONL", M4FC_AR_TEST_JSONL),
    ("VERITE JSONL", VERITE_JSONL),
]:
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")
    print(path)

# -----------------------------
# 6. GPU check
# -----------------------------
try:
    import torch
    print("\nPyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        torch.cuda.empty_cache()
except Exception as e:
    print("PyTorch/GPU check:", e)

gc.collect()

print("\n========================================")
print("RUNTIME RECOVERY FINISHED")
print("========================================")
print("Loaded variables are available for the next cells.")


In [ ]:

# ============================================================
# TWINCHECK — LOCATE M4FC ARABIC TEST DATA
# Also inspect the VERITE source manifest
# ============================================================

from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/twincheck")

print("1. Searching for M4FC Arabic test JSONL...\n")

matches = list(PROJECT.rglob("m4fc_ar_test.jsonl"))

if matches:
    for i, path in enumerate(matches, 1):
        print(f"{i}. FOUND: {path}")
        with open(path, "r", encoding="utf-8") as f:
            for j, line in enumerate(f):
                if j == 3:
                    break
                record = json.loads(line)
                print(f"   Record {j+1} keys: {list(record.keys())}")
                print(f"   Record {j+1}: {record}")
else:
    print("Exact filename not found.")
    print("\nSearching for other M4FC Arabic JSONL files...")
    for path in PROJECT.rglob("*m4fc*"):
        if path.is_file() and path.suffix.lower() in [".jsonl", ".json", ".csv"]:
            print(path)

print("\n2. Inspecting VERITE true-image manifest...\n")

print("Manifest shape:", verite_true_manifest.shape)
print("Columns:", verite_true_manifest.columns.tolist())

print("\nLabel counts:")
print(verite_true_manifest["label"].value_counts(dropna=False))

print("\nSplit counts:")
print(verite_true_manifest["split"].value_counts(dropna=False))

print("\nImage availability:")
print(verite_true_manifest["image_exists"].value_counts(dropna=False))

print("\n3. Inspecting M4FC image directory...\n")

M4FC_IMAGE_DIR = PROJECT / "data" / "processed" / "vision" / "m4fc_images"

print("Directory exists:", M4FC_IMAGE_DIR.exists())

if M4FC_IMAGE_DIR.exists():
    images = list(M4FC_IMAGE_DIR.glob("*"))
    print("Number of files:", len(images))
    print("First 10 files:")
    for p in images[:10]:
        print(p.name)

print("\nINSPECTION COMPLETE — no files modified.")




In [ ]:

# ============================================================
# M4FC — VERIFY SUPPORTED IMAGE SOURCES
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/twincheck")

mapping_path = (
    PROJECT / "data/processed/vision/m4fc_test_claim_image_mapping.csv"
)
manifest_path = (
    PROJECT / "features/vision/m4fc/image_twins/"
    "m4fc_true_image_twin_manifest.csv"
)
image_dir = PROJECT / "data/processed/vision/m4fc_images"

mapping = pd.read_csv(mapping_path)
true_manifest = pd.read_csv(manifest_path)

print("M4FC mapping shape:", mapping.shape)
print("Mapping columns:", mapping.columns.tolist())

print("\nOriginal label counts:")
print(mapping["orig_label"].value_counts(dropna=False))

print("\nSupported/contradicted counts:")
print(mapping["label"].value_counts(dropna=False))

print("\nImage availability:")
print(mapping["image_exists"].value_counts(dropna=False))

supported = mapping[
    mapping["label"].astype(str).str.strip().str.lower() == "supported"
].copy()

print("\nSupported claim records:", len(supported))
print("Unique supported image paths:", supported["image_path"].nunique())
print("Supported records with existing images:", supported["image_exists"].sum())

print("\nSupported label and path examples:")
print(supported[["claim_id", "image_path", "resolved_path",
                 "label", "orig_label", "image_exists"]].head(10).to_string(index=False))

print("\nExisting true-image twin manifest:")
print("Rows:", len(true_manifest))
print("Columns:", true_manifest.columns.tolist())
print(true_manifest.head(5).to_string(index=False))

print("\nImage directory:", image_dir)
print("Image directory exists:", image_dir.exists())

print("\nNo files modified.")


In [ ]:


# ============================================================
# TWINCHECK — BUILD SDXL SOURCE MANIFEST
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/twincheck")

# Output directory
OUTPUT_DIR = PROJECT / "data" / "copies" / "vision"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. VERITE: test + validation true images only
# ------------------------------------------------------------
verite_sources = verite_true_manifest[
    verite_true_manifest["split"].isin(["test", "validation"])
].copy()

verite_sources = verite_sources[
    verite_sources["image_exists"].astype(bool)
].copy()

verite_sources["id"] = verite_sources["image_id"].astype(str)
verite_sources["dataset"] = "VERITE"
verite_sources["source_image_path"] = (
    verite_sources["resolved_image_path"].astype(str)
)

verite_sources = verite_sources[
    ["id", "dataset", "split", "label", "source_image_path"]
]

# ------------------------------------------------------------
# 2. M4FC: supported images with available files
# ------------------------------------------------------------
m4fc_supported = mapping[
    (mapping["label"].astype(str).str.lower() == "supported") &
    (mapping["image_exists"].astype(bool))
].copy()

# Use the actual image files in the project directory.
m4fc_supported["source_image_path"] = (
    m4fc_supported["image_path"].apply(
        lambda p: str(PROJECT / str(p))
    )
)

m4fc_supported = m4fc_supported[
    m4fc_supported["source_image_path"].apply(
        lambda p: Path(p).is_file()
    )
].copy()

# Deduplicate by image, not claim.
m4fc_supported["id"] = (
    m4fc_supported["image_path"]
    .apply(lambda p: Path(str(p)).stem)
)

m4fc_supported = m4fc_supported.drop_duplicates(
    subset=["id"], keep="first"
)

m4fc_supported["dataset"] = "M4FC"
m4fc_supported["split"] = "test"
m4fc_supported["label"] = "supported"

m4fc_sources = m4fc_supported[
    ["id", "dataset", "split", "label", "source_image_path"]
]

# ------------------------------------------------------------
# 3. Combine and validate
# ------------------------------------------------------------
sources = pd.concat(
    [verite_sources, m4fc_sources],
    ignore_index=True
)

assert sources["source_image_path"].apply(
    lambda p: Path(p).is_file()
).all(), "Some source image files are missing!"

assert not sources.duplicated(
    subset=["dataset", "id"]
).any(), "Duplicate source images found!"

# ------------------------------------------------------------
# 4. Save
# ------------------------------------------------------------
output_path = OUTPUT_DIR / "sources.csv"
sources.to_csv(output_path, index=False)

# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------
print("SOURCE MANIFEST CREATED")
print("Saved to:", output_path)
print("\nTotal sources:", len(sources))

print("\nCounts by dataset and split:")
print(sources.groupby(["dataset", "split"]).size())

print("\nAll source files exist:",
      sources["source_image_path"].apply(
          lambda p: Path(p).is_file()
      ).all())

print("\nDuplicate source IDs:",
      sources.duplicated(["dataset", "id"]).sum())

print("\nFirst 10 rows:")
print(sources.head(10).to_string(index=False))


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/twincheck")
VISION_DIR = PROJECT / "data/copies/vision"

sources = pd.read_csv(VISION_DIR / "sources.csv")

verite_excluded = {
    "true_13.jpg", "true_24.jpg", "true_36.jpg",
    "true_59.jpg", "true_99.jpg", "true_103.jpg",
    "true_106.jpg", "true_117.jpg", "true_182.jpg",
    "true_278.jpg", "true_292.jpg", "true_311.jpg"
}

m4fc_excluded = {
    "m4fc_000088", "m4fc_000324",
    "m4fc_000663", "m4fc_000667"
}

def exclude_face(row):
    if row["dataset"] == "VERITE":
        return row["id"] in verite_excluded
    if row["dataset"] == "M4FC":
        return row["id"] in m4fc_excluded
    return False

sources["face_excluded"] = sources.apply(exclude_face, axis=1)

eligible = sources[~sources["face_excluded"]].copy()
excluded = sources[sources["face_excluded"]].copy()

assert eligible["source_image_path"].apply(
    lambda p: Path(p).is_file()
).all()

eligible.to_csv(VISION_DIR / "sdxl_sources.csv", index=False)
excluded.to_csv(VISION_DIR / "sdxl_face_excluded_sources.csv", index=False)

print("Original candidates:", len(sources))
print("Excluded:", len(excluded))
print("Eligible SDXL sources:", len(eligible))
print("\nEligible by dataset and split:")
print(eligible.groupby(["dataset", "split"]).size())
print("\nSaved:", VISION_DIR / "sdxl_sources.csv")


In [ ]:

# ============================================================
# TWINCHECK — SDXL IMG2IMG PILOT (5 IMAGES)
# Saves outputs and checkpoint to Google Drive
# ============================================================

import os
import gc
import torch
import pandas as pd
from pathlib import Path
from PIL import Image

from diffusers import AutoPipelineForImage2Image

PROJECT = Path("/content/drive/MyDrive/twincheck")
VISION_DIR = PROJECT / "data/copies/vision"

SOURCE_CSV = VISION_DIR / "sdxl_sources.csv"
OUTPUT_DIR = VISION_DIR / "sdxl_twins_pilot"
ORIGINAL_DIR = VISION_DIR / "sdxl_originals_jpeg"
CHECKPOINT = VISION_DIR / "sdxl_pilot_checkpoint.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ORIGINAL_DIR.mkdir(parents=True, exist_ok=True)

sources = pd.read_csv(SOURCE_CSV).head(5).copy()

# -----------------------------
# GPU verification
# -----------------------------
assert torch.cuda.is_available(), "CUDA GPU is unavailable."

print("GPU:", torch.cuda.get_device_name(0))

# -----------------------------
# Load SDXL img2img pipeline
# -----------------------------
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

pipe = AutoPipelineForImage2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
)

pipe.enable_model_cpu_offload()

# -----------------------------
# Resume-safe checkpoint
# -----------------------------
if CHECKPOINT.exists():
    checkpoint = pd.read_csv(CHECKPOINT)
else:
    checkpoint = pd.DataFrame(columns=[
        "id", "dataset", "split",
        "original_jpeg", "twin_path",
        "status", "strength", "seed"
    ])

completed_ids = set(
    checkpoint.loc[
        checkpoint["status"] == "success", "id"
    ].astype(str)
)

results = []

# -----------------------------
# Generate 5 pilot twins
# -----------------------------
for _, row in sources.iterrows():

    source_id = str(row["id"])
    dataset = str(row["dataset"])
    source_path = Path(row["source_image_path"])

    output_name = f"{dataset}_{Path(source_id).stem}_V2.jpg"
    twin_path = OUTPUT_DIR / output_name
    original_path = ORIGINAL_DIR / f"{dataset}_{Path(source_id).stem}.jpg"

    if source_id in completed_ids and twin_path.exists():
        print("Already completed:", source_id)
        continue

    try:
        # Read original
        image = Image.open(source_path).convert("RGB")

        # SDXL dimensions must be compatible with VAE scaling.
        width = (image.width // 8) * 8
        height = (image.height // 8) * 8
        image = image.resize((width, height), Image.Resampling.LANCZOS)

        # Save standardized original
        image.save(original_path, "JPEG", quality=85)

        generator = torch.Generator(
            device="cpu"
        ).manual_seed(42)

        # Empty prompt: avoid introducing an additional textual description.
        result = pipe(
            prompt="",
            image=image,
            strength=0.20,
            guidance_scale=1.0,
            num_inference_steps=30,
            generator=generator
        ).images[0]

        result.save(twin_path, "JPEG", quality=85)

        results.append({
            "id": source_id,
            "dataset": dataset,
            "split": row["split"],
            "original_jpeg": str(original_path),
            "twin_path": str(twin_path),
            "status": "success",
            "strength": 0.20,
            "seed": 42
        })

        print("SUCCESS:", source_id, "->", twin_path.name)

    except Exception as e:
        results.append({
            "id": source_id,
            "dataset": dataset,
            "split": row["split"],
            "original_jpeg": str(original_path),
            "twin_path": str(twin_path),
            "status": f"failed: {str(e)[:250]}",
            "strength": 0.20,
            "seed": 42
        })

        print("FAILED:", source_id, str(e)[:250])

    # Save checkpoint after each image
    if results:
        checkpoint = pd.concat(
            [checkpoint, pd.DataFrame(results)],
            ignore_index=True
        )
        checkpoint = checkpoint.drop_duplicates(
            subset=["id"], keep="last"
        )
        checkpoint.to_csv(CHECKPOINT, index=False)
        results = []

    gc.collect()
    torch.cuda.empty_cache()

# -----------------------------
# Final pilot report
# -----------------------------
print("\nPILOT COMPLETE")
print("Checkpoint:", CHECKPOINT)
print("Successful:", (checkpoint["status"] == "success").sum())
print("Failed:", checkpoint["status"].astype(str).str.startswith("failed").sum())
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


In [ ]:
import pandas as pd
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/twincheck/data/copies/vision/sdxl_pilot_checkpoint.csv"
)

pilot = pd.read_csv(checkpoint_path)

print("Checkpoint columns:")
print(pilot.columns.tolist())

print("\nPilot results:")
display(pilot)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/twincheck/data/copies/vision/sdxl_pilot_checkpoint.csv"
)

pilot = pd.read_csv(checkpoint_path)
pilot = pilot[pilot["status"] == "success"].reset_index(drop=True)

fig, axes = plt.subplots(len(pilot), 2, figsize=(12, 4 * len(pilot)))

if len(pilot) == 1:
    axes = [axes]

for i, row in pilot.iterrows():
    original = Image.open(row["original_jpeg"]).convert("RGB")
    twin = Image.open(row["twin_path"]).convert("RGB")

    axes[i][0].imshow(original)
    axes[i][0].set_title(f"Original: {row['id']}")
    axes[i][0].axis("off")

    axes[i][1].imshow(twin)
    axes[i][1].set_title(f"SDXL Twin: {row['id']}")
    axes[i][1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

PROJECT = Path("/content/drive/MyDrive/twincheck")
checkpoint_path = PROJECT / "data/copies/vision/sdxl_pilot_checkpoint.csv"

pilot = pd.read_csv(checkpoint_path)
pilot = pilot[pilot["status"] == "success"].copy()

device = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.get_image_features(**inputs)

        # Handle SigLIP2 output formats
        if hasattr(outputs, "pooler_output"):
            features = outputs.pooler_output
        elif hasattr(outputs, "last_hidden_state"):
            features = outputs.last_hidden_state[:, 0, :]
        else:
            features = outputs

    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy()[0]

results = []

for _, row in tqdm(pilot.iterrows(), total=len(pilot)):
    original_emb = get_embedding(row["original_jpeg"])
    twin_emb = get_embedding(row["twin_path"])

    similarity = float(np.dot(original_emb, twin_emb))

    results.append({
        "id": row["id"],
        "dataset": row["dataset"],
        "split": row["split"],
        "similarity": similarity,
        "accepted": similarity >= 0.92
    })

results_df = pd.DataFrame(results)

output_path = PROJECT / "features/vision/sdxl_pilot_similarity.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_path, index=False)

display(results_df)

print("\nAccepted:", int(results_df["accepted"].sum()), "/", len(results_df))
print("Mean similarity:", round(results_df["similarity"].mean(), 4))
print("Saved:", output_path)

In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Paths and settings
# --------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/twincheck")

checkpoint_path = (
    PROJECT / "data/copies/vision/sdxl_pilot_checkpoint.csv"
)

output_dir = (
    PROJECT / "data/copies/vision/sdxl_twins_pilot_s015"
)
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_out = (
    PROJECT / "data/copies/vision/sdxl_pilot_s015_checkpoint.csv"
)

similarity_out = (
    PROJECT / "features/vision/sdxl_pilot_s015_similarity.csv"
)
similarity_out.parent.mkdir(parents=True, exist_ok=True)

STRENGTH = 0.15
SEED = 42

# --------------------------------------------------
# 2. Check loaded models
# --------------------------------------------------

if "pipe" not in globals():
    raise RuntimeError("SDXL pipeline is not loaded.")

if "model" not in globals() or "processor" not in globals():
    raise RuntimeError("SigLIP2 model or processor is not loaded.")

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe.set_progress_bar_config(disable=True)
model.eval()

# --------------------------------------------------
# 3. Load the five pilot sources
# --------------------------------------------------

sources = pd.read_csv(checkpoint_path)
sources = sources[sources["status"] == "success"].copy()

print("Pilot sources:", len(sources))

# --------------------------------------------------
# 4. Resume from checkpoint if available
# --------------------------------------------------

if checkpoint_out.exists():
    checkpoint = pd.read_csv(checkpoint_out)
    records = checkpoint.to_dict("records")
else:
    records = []

completed_ids = {
    r["id"] for r in records
    if r["status"] == "success"
}

# --------------------------------------------------
# 5. Generate SDXL twins at strength 0.15
# --------------------------------------------------

for _, row in tqdm(sources.iterrows(), total=len(sources)):

    image_id = row["id"]

    if image_id in completed_ids:
        continue

    try:
        original = Image.open(
            row["original_jpeg"]
        ).convert("RGB")

        generator = torch.Generator(
            device="cpu"
        ).manual_seed(SEED)

        result = pipe(
            prompt="",
            image=original,
            strength=STRENGTH,
            guidance_scale=1.0,
            num_inference_steps=30,
            generator=generator
        ).images[0]

        twin_path = (
            output_dir /
            f"{Path(image_id).stem}_V2_s015.jpg"
        )

        result.save(twin_path, "JPEG", quality=85)

        record = {
            "id": image_id,
            "dataset": row["dataset"],
            "split": row["split"],
            "original_jpeg": row["original_jpeg"],
            "twin_path": str(twin_path),
            "status": "success",
            "strength": STRENGTH,
            "seed": SEED
        }

        print("SUCCESS:", image_id)

    except Exception as e:

        record = {
            "id": image_id,
            "dataset": row["dataset"],
            "split": row["split"],
            "original_jpeg": row["original_jpeg"],
            "twin_path": "",
            "status": "failed",
            "strength": STRENGTH,
            "seed": SEED,
            "error": str(e)
        }

        print("FAILED:", image_id, str(e))

    records = [
        r for r in records
        if r["id"] != image_id
    ]
    records.append(record)

    pd.DataFrame(records).to_csv(
        checkpoint_out, index=False
    )

print("\nGeneration completed.")
print(
    "Successful:",
    sum(r["status"] == "success" for r in records)
)
print(
    "Failed:",
    sum(r["status"] == "failed" for r in records)
)

# --------------------------------------------------
# 6. SigLIP2 embedding function
# --------------------------------------------------

def get_embedding(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.get_image_features(**inputs)

        if hasattr(outputs, "pooler_output"):
            features = outputs.pooler_output

        elif hasattr(outputs, "last_hidden_state"):
            features = outputs.last_hidden_state[:, 0, :]

        else:
            features = outputs

    features = features / features.norm(
        dim=-1, keepdim=True
    )

    return features.cpu().numpy()[0]

# --------------------------------------------------
# 7. Calculate original-twin similarity
# --------------------------------------------------

results = []

successful_records = [
    r for r in records
    if r["status"] == "success"
]

for r in tqdm(successful_records):

    original_emb = get_embedding(
        r["original_jpeg"]
    )

    twin_emb = get_embedding(
        r["twin_path"]
    )

    similarity = float(
        np.dot(original_emb, twin_emb)
    )

    results.append({
        "id": r["id"],
        "dataset": r["dataset"],
        "split": r["split"],
        "similarity": similarity,
        "accepted": similarity >= 0.92
    })

# --------------------------------------------------
# 8. Save and display results
# --------------------------------------------------

comparison = pd.DataFrame(results)

comparison.to_csv(
    similarity_out,
    index=False
)

display(comparison)

if len(comparison) > 0:

    print("\nAccepted:",
          int(comparison["accepted"].sum()),
          "/", len(comparison))

    print(
        "Mean similarity:",
        round(comparison["similarity"].mean(), 4)
    )

print("\nCheckpoint:", checkpoint_out)
print("Similarity report:", similarity_out)

In [ ]:

import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

# ============================================================
# 1. PATHS AND SETTINGS
# ============================================================

PROJECT = Path("/content/drive/MyDrive/twincheck")
COPIES = PROJECT / "data/copies/vision"
FEATURES = PROJECT / "features/vision"

SOURCE_CSV = COPIES / "sdxl_sources.csv"
PILOT_CSV = COPIES / "sdxl_pilot_s015_checkpoint.csv"

ORIGINAL_DIR = COPIES / "sdxl_originals_jpeg"
TWIN_DIR = COPIES / "sdxl_twins_s015"
ORIGINAL_DIR.mkdir(parents=True, exist_ok=True)
TWIN_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT = COPIES / "sdxl_twins_s015_checkpoint.csv"
SIMILARITY_REPORT = FEATURES / "sdxl_s015_similarity_report.csv"
ACCEPTED_MANIFEST = COPIES / "sdxl_accepted_twin_manifest.csv"
REJECTED_MANIFEST = COPIES / "sdxl_rejected_twin_manifest.csv"

STRENGTH = 0.15
SEED = 42
STEPS = 30
THRESHOLD = 0.92

# ============================================================
# 2. CHECK MODELS
# ============================================================

if "pipe" not in globals():
    raise RuntimeError("SDXL pipeline 'pipe' is not loaded.")

if "model" not in globals() or "processor" not in globals():
    raise RuntimeError("SigLIP2 model or processor is not loaded.")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()

# ============================================================
# 3. LOAD SOURCES AND PILOT RESULTS
# ============================================================

sources = pd.read_csv(SOURCE_CSV)
pilot = pd.read_csv(PILOT_CSV)

assert len(sources) == 249, (
    f"Expected 249 sources, found {len(sources)}. "
    "Check sdxl_sources.csv before proceeding."
)

records = []

for _, row in pilot.iterrows():
    records.append(row.to_dict())

if CHECKPOINT.exists():
    previous = pd.read_csv(CHECKPOINT)
    previous_records = previous.to_dict("records")

    # Preserve previous full-run successes/failures.
    # Pilot entries remain authoritative for the five pilot IDs.
    pilot_ids = set(pilot["id"].astype(str))

    for r in previous_records:
        if str(r["id"]) not in pilot_ids:
            records.append(r)

# Keep one latest record per dataset + image ID.
record_map = {}
for r in records:
    key = (str(r["dataset"]), str(r["id"]))
    record_map[key] = r

records = list(record_map.values())

def save_checkpoint():
    pd.DataFrame(records).to_csv(CHECKPOINT, index=False)

# ============================================================
# 4. GENERATE REMAINING SDXL TWINS
# ============================================================

completed = {
    (str(r["dataset"]), str(r["id"]))
    for r in records
    if r.get("status") == "success"
}

print("Total sources:", len(sources))
print("Already successful:", len(completed))
print("Remaining to generate:", len(sources) - len(completed))

for _, row in tqdm(sources.iterrows(), total=len(sources)):

    image_id = str(row["id"])
    dataset = str(row["dataset"])
    key = (dataset, image_id)

    if key in completed:
        continue

    source_path = Path(row["source_image_path"])
    original_path = ORIGINAL_DIR / f"{dataset}_{Path(image_id).stem}.jpg"
    twin_path = TWIN_DIR / f"{dataset}_{Path(image_id).stem}_V2_s015.jpg"

    try:
        if not source_path.exists():
            raise FileNotFoundError(str(source_path))

        # Save source as JPEG quality 85.
        if not original_path.exists():
            with Image.open(source_path) as img:
                img.convert("RGB").save(
                    original_path, "JPEG", quality=85
                )

        original = Image.open(original_path).convert("RGB")

        generator = torch.Generator(device="cpu").manual_seed(SEED)

        generated = pipe(
            prompt="",
            image=original,
            strength=STRENGTH,
            guidance_scale=1.0,
            num_inference_steps=STEPS,
            generator=generator
        ).images[0]

        generated.save(twin_path, "JPEG", quality=85)

        record = {
            "id": image_id,
            "dataset": dataset,
            "split": row["split"],
            "original_jpeg": str(original_path),
            "twin_path": str(twin_path),
            "status": "success",
            "strength": STRENGTH,
            "seed": SEED
        }

        completed.add(key)
        print("SUCCESS:", dataset, image_id)

    except Exception as e:
        record = {
            "id": image_id,
            "dataset": dataset,
            "split": row["split"],
            "original_jpeg": str(original_path),
            "twin_path": "",
            "status": "failed",
            "strength": STRENGTH,
            "seed": SEED,
            "error": str(e)
        }
        print("FAILED:", dataset, image_id, str(e))

    record_map[key] = record
    records = list(record_map.values())
    save_checkpoint()

print("\nGENERATION FINISHED")
print("Successful:", sum(r["status"] == "success" for r in records))
print("Failed:", sum(r["status"] == "failed" for r in records))
print("Checkpoint:", CHECKPOINT)

# ============================================================
# 5. SIGLIP2 EMBEDDING FUNCTION
# ============================================================

def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.get_image_features(**inputs)

        if hasattr(output, "pooler_output"):
            features = output.pooler_output
        elif hasattr(output, "last_hidden_state"):
            features = output.last_hidden_state[:, 0, :]
        else:
            features = output

    features = features / features.norm(dim=-1, keepdim=True)
    return features.float().cpu().numpy()[0]

# ============================================================
# 6. CALCULATE SIMILARITY FOR ALL SUCCESSFUL TWINS
# ============================================================

similarity_records = []

for r in tqdm(records, desc="SigLIP2 similarity"):

    if r.get("status") != "success":
        continue

    try:
        original_embedding = get_embedding(r["original_jpeg"])
        twin_embedding = get_embedding(r["twin_path"])

        similarity = float(
            np.dot(original_embedding, twin_embedding)
        )

        similarity_records.append({
            "id": r["id"],
            "dataset": r["dataset"],
            "split": r["split"],
            "original_jpeg": r["original_jpeg"],
            "twin_path": r["twin_path"],
            "strength": STRENGTH,
            "similarity": similarity,
            "accepted": similarity >= THRESHOLD
        })

    except Exception as e:
        print("SIMILARITY FAILED:", r["id"], str(e))

similarity_df = pd.DataFrame(similarity_records)
similarity_df.to_csv(SIMILARITY_REPORT, index=False)

# ============================================================
# 7. SAVE ACCEPTED AND REJECTED MANIFESTS
# ============================================================

accepted = similarity_df[
    similarity_df["accepted"] == True
].copy()

rejected = similarity_df[
    similarity_df["accepted"] == False
].copy()

accepted.to_csv(ACCEPTED_MANIFEST, index=False)
rejected.to_csv(REJECTED_MANIFEST, index=False)

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n========== FINAL SDXL SUMMARY ==========")
print("Source images:", len(sources))
print("Successful generations:",
      sum(r["status"] == "success" for r in records))
print("Failed generations:",
      sum(r["status"] == "failed" for r in records))
print("Similarity evaluated:", len(similarity_df))
print("Accepted twins:", len(accepted))
print("Rejected twins:", len(rejected))

if len(similarity_df):
    print("Acceptance rate:",
          round(100 * len(accepted) / len(similarity_df), 2), "%")
    print("Mean similarity:",
          round(similarity_df["similarity"].mean(), 4))

print("\nDataset breakdown:")
if len(similarity_df):
    print(
        similarity_df.groupby(["dataset", "split"])["accepted"]
        .agg(["count", "sum", "mean"])
    )

print("\nSaved files:")
print(CHECKPOINT)
print(SIMILARITY_REPORT)
print(ACCEPTED_MANIFEST)
print(REJECTED_MANIFEST)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/twincheck")
manifest = pd.read_csv(
    PROJECT / "data/copies/vision/sdxl_accepted_twin_manifest.csv"
)

sample = (
    manifest.groupby(["dataset", "split"], group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), max(1, round(30 * len(x) / len(manifest)))),
        random_state=42
    ))
    .head(30)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(len(sample), 2, figsize=(12, 3.5 * len(sample)))

for i, row in sample.iterrows():
    original = Image.open(row["original_jpeg"]).convert("RGB")
    twin = Image.open(row["twin_path"]).convert("RGB")

    axes[i, 0].imshow(original)
    axes[i, 0].set_title(f"Original: {row['id']}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(twin)
    axes[i, 1].set_title(f"SDXL Twin | Similarity: {row['similarity']:.3f}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/twincheck")
assert PROJECT.exists(), f"Project folder not found: {PROJECT}"

print("Searching for M4FC CLIP ViT-L/14 embedding files...\n")

matches = [
    p for p in PROJECT.rglob("*")
    if p.is_file()
    and p.suffix.lower() == ".npy"
    and "m4fc" in str(p).lower()
    and ("clip" in str(p).lower() or "vit_l14" in str(p).lower())
]

if not matches:
    print("No M4FC CLIP embedding .npy files found.")
else:
    for path in matches:
        print(f"\nFile: {path.relative_to(PROJECT)}")
        try:
            arr = np.load(path, mmap_mode="r")
            print("Shape:", arr.shape)
            print("Data type:", arr.dtype)
            print("All finite:", bool(np.isfinite(arr).all()))

            if arr.ndim == 2:
                norms = np.linalg.norm(arr, axis=1)
                print("Mean L2 norm:", float(norms.mean()))
                print("Min L2 norm:", float(norms.min()))
                print("Max L2 norm:", float(norms.max()))
                print("Approximately normalized:",
                      bool(np.allclose(norms, 1.0, atol=1e-3)))
        except Exception as e:
            print("Could not inspect:", repr(e))


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/twincheck")
folder = PROJECT / "features/vision/m4fc/siglip2"

print("Files in M4FC SigLIP2 folder:")
for p in sorted(folder.glob("*")):
    print(p.name)

print("\nManifest candidates:")
for p in folder.glob("*.csv"):
    df = pd.read_csv(p)
    print(f"\n{p.name} — shape: {df.shape}")
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))


In [ ]:

!pip install git+https://github.com/openai/CLIP.git


In [ ]:

import torch
import clip
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

PROJECT = Path("/content/drive/MyDrive/twincheck")
SOURCE_MANIFEST = PROJECT / "features/vision/m4fc/siglip2/m4fc_siglip2_image_projected_embeddings_manifest.csv"
OUTPUT_DIR = PROJECT / "features/vision/m4fc/clip_vit_l14"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_FILE = OUTPUT_DIR / "m4fc_clip_vit_l14_image_embeddings.npy"
MANIFEST_FILE = OUTPUT_DIR / "m4fc_clip_vit_l14_image_embeddings_manifest.csv"

# Load and check the existing verified image manifest
df = pd.read_csv(SOURCE_MANIFEST)
df = df[df["image_exists"] == True].copy().reset_index(drop=True)

assert len(df) == 603, f"Expected 603 images, found {len(df)}"
assert df["resolved_image_path"].map(lambda p: Path(p).is_file()).all(), \
    "At least one image path is missing."

# Load CLIP ViT-L/14
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-L/14", device=device)
model.eval()

print("Device:", device)
print("Images:", len(df))

embeddings = []

with torch.no_grad():
    for start in tqdm(range(0, len(df), 32)):
        batch_paths = df["resolved_image_path"].iloc[start:start+32].tolist()
        images = []

        for path in batch_paths:
            with Image.open(path) as img:
                images.append(preprocess(img.convert("RGB")))

        batch = torch.stack(images).to(device)
        features = model.encode_image(batch)
        features = features / features.norm(dim=-1, keepdim=True)
        embeddings.append(features.cpu().numpy().astype(np.float32))

embeddings = np.concatenate(embeddings, axis=0)

# Save embeddings and aligned manifest
np.save(EMBEDDING_FILE, embeddings)

df["embedding_row"] = np.arange(len(df))
df.to_csv(MANIFEST_FILE, index=False)

# Final checks
norms = np.linalg.norm(embeddings, axis=1)

print("\nExtraction complete.")
print("Embedding shape:", embeddings.shape)
print("All finite:", np.isfinite(embeddings).all())
print("Mean L2 norm:", norms.mean())
print("All normalized:", np.allclose(norms, 1.0, atol=1e-3))
print("Embeddings saved:", EMBEDDING_FILE)
print("Manifest saved:", MANIFEST_FILE)


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")

files = {
    "VERITE": ROOT / "features/vision/verite/image_twins/verite_face_filtered_vae_twin_manifest.csv",
    "M4FC": ROOT / "features/vision/m4fc/image_twins/m4fc_face_filtered_vae_twin_manifest.csv",
}

for dataset, path in files.items():
    print(f"\n{'='*55}\n{dataset}")
    if not path.exists():
        print("Expected face-filtered manifest not found:", path)
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")

manifests = {
    "VERITE": (
        ROOT / "features/vision/verite/image_twins/verite_face_filtered_vae_twin_manifest.csv",
        "twin_path",
        "siglip2_similarity",
    ),
    "M4FC": (
        ROOT / "features/vision/m4fc/image_twins/m4fc_face_filtered_vae_twin_manifest.csv",
        "twin_path",
        "siglip2_cosine_similarity",
    ),
}

for name, (manifest_path, path_col, sim_col) in manifests.items():
    df = pd.read_csv(manifest_path)

    exists = df[path_col].map(lambda p: Path(p).is_file())
    similarity = pd.to_numeric(df[sim_col], errors="coerce")

    print(f"\n===== {name} VAE AUDIT =====")
    print("Manifest rows:", len(df))
    print("Twin files existing:", int(exists.sum()))
    print("Twin files missing:", int((~exists).sum()))
    print("Accepted flags:", int(df["accepted"].sum()))
    print("Similarity >= 0.92:", int((similarity >= 0.92).sum()))
    print("Below 0.92:", int((similarity < 0.92).sum()))
    print("Minimum similarity:", round(similarity.min(), 4))
    print("Mean similarity:", round(similarity.mean(), 4))
    print("All rows accepted:", bool(df["accepted"].all()))
    print("All files exist:", bool(exists.all()))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck/data/copies/vision")

files = {
    "checkpoint": ROOT / "sdxl_twins_s015_checkpoint.csv",
    "accepted": ROOT / "sdxl_accepted_twin_manifest.csv",
    "rejected": ROOT / "sdxl_rejected_twin_manifest.csv",
}

for name, path in files.items():
    print(f"\n===== {name.upper()} =====")
    print("Exists:", path.exists())
    if path.exists():
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        print(df.head(2).to_string(index=False))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck/data/copies/vision")

checkpoint = pd.read_csv(ROOT / "sdxl_twins_s015_checkpoint.csv")
accepted = pd.read_csv(ROOT / "sdxl_accepted_twin_manifest.csv")
rejected = pd.read_csv(ROOT / "sdxl_rejected_twin_manifest.csv")

def audit_files(df, path_col):
    return df[path_col].map(lambda p: Path(p).is_file())

checkpoint_exists = audit_files(checkpoint, "twin_path")
accepted_exists = audit_files(accepted, "twin_path")
rejected_exists = audit_files(rejected, "twin_path")

print("===== SDXL FILE AUDIT =====")
print("Checkpoint rows:", len(checkpoint))
print("Successful generations:", (checkpoint["status"] == "success").sum())
print("Checkpoint twin files existing:", checkpoint_exists.sum())
print("Checkpoint twin files missing:", (~checkpoint_exists).sum())
print("Accepted twin files existing:", accepted_exists.sum())
print("Accepted twin files missing:", (~accepted_exists).sum())
print("Rejected twin files existing:", rejected_exists.sum())
print("Rejected twin files missing:", (~rejected_exists).sum())

print("\n===== SIMILARITY AUDIT =====")
print("Accepted rows >= 0.92:", (accepted["similarity"] >= 0.92).sum())
print("Accepted rows below 0.92:", (accepted["similarity"] < 0.92).sum())
print("Rejected rows below 0.92:", (rejected["similarity"] < 0.92).sum())
print("Rejected rows >= 0.92:", (rejected["similarity"] >= 0.92).sum())

print("\n===== OUTPUT FOLDERS =====")
for folder in ["sdxl_twins_pilot_s015", "sdxl_twins_s015"]:
    p = ROOT / folder
    print(folder, "exists:", p.exists(),
          "image count:", len(list(p.glob("*"))) if p.exists() else 0)


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")
VISION = ROOT / "data/copies/vision"

accepted = pd.read_csv(VISION / "sdxl_accepted_twin_manifest.csv")

# Reproducible sample of 30 accepted pairs
sample = accepted.sample(n=30, random_state=42).reset_index(drop=True)

sample["original_exists"] = sample["original_jpeg"].map(
    lambda p: Path(p).is_file()
)
sample["twin_exists"] = sample["twin_path"].map(
    lambda p: Path(p).is_file()
)

output = ROOT / "features/vision/sdxl_visual_inspection_sample_30.csv"
output.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(output, index=False)

print("Sample size:", len(sample))
print("Original files existing:", sample["original_exists"].sum())
print("Twin files existing:", sample["twin_exists"].sum())
print("\nInspection sample:")
print(sample[["id", "dataset", "split", "similarity",
              "original_jpeg", "twin_path"]].to_string(index=False))
print("\nSaved:", output)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")
sample_path = ROOT / "features/vision/sdxl_visual_inspection_sample_30.csv"
df = pd.read_csv(sample_path)

batch = df.iloc[0:6]

fig, axes = plt.subplots(6, 2, figsize=(12, 24))

for i, (_, row) in enumerate(batch.iterrows()):
    original = Image.open(row["original_jpeg"]).convert("RGB")
    twin = Image.open(row["twin_path"]).convert("RGB")

    axes[i, 0].imshow(original)
    axes[i, 0].set_title(f'Original: {row["id"]}', fontsize=10)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(twin)
    axes[i, 1].set_title(
        f'SDXL twin | Similarity: {row["similarity"]:.4f}',
        fontsize=10
    )
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = "/content/drive/MyDrive/twincheck"
sample = pd.read_csv(
    f"{ROOT}/features/vision/sdxl_visual_inspection_sample_30.csv"
)

batch = sample.iloc[6:12]

fig, axes = plt.subplots(6, 2, figsize=(12, 24))

for i, (_, row) in enumerate(batch.iterrows()):
    original = Image.open(row["original_jpeg"]).convert("RGB")
    twin = Image.open(row["twin_path"]).convert("RGB")

    axes[i, 0].imshow(original)
    axes[i, 0].set_title(f'Original: {row["id"]}')
    axes[i, 0].axis("off")

    axes[i, 1].imshow(twin)
    axes[i, 1].set_title(f'SDXL twin | Similarity: {row["similarity"]:.4f}')
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = "/content/drive/MyDrive/twincheck"
sample = pd.read_csv(
    f"{ROOT}/features/vision/sdxl_visual_inspection_sample_30.csv"
)

batch = sample.iloc[12:18]

fig, axes = plt.subplots(6, 2, figsize=(12, 24))

for i, (_, row) in enumerate(batch.iterrows()):
    original = Image.open(row["original_jpeg"]).convert("RGB")
    twin = Image.open(row["twin_path"]).convert("RGB")

    axes[i, 0].imshow(original)
    axes[i, 0].set_title(f'Original: {row["id"]}')
    axes[i, 0].axis("off")

    axes[i, 1].imshow(twin)
    axes[i, 1].set_title(
        f'SDXL twin | Similarity: {row["similarity"]:.4f}'
    )
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# RESTORE COLAB + DISPLAY SDXL PAIRS 19–24

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Restore project paths
ROOT = Path("/content/drive/MyDrive/twincheck")
SAMPLE_PATH = ROOT / "features/vision/sdxl_visual_inspection_sample_30.csv"

# Confirm saved inspection sample exists
assert SAMPLE_PATH.exists(), f"Sample CSV not found: {SAMPLE_PATH}"

sample = pd.read_csv(SAMPLE_PATH)
assert len(sample) == 30, f"Expected 30 pairs, found {len(sample)}"

# Display remaining pairs 19–24
batch = sample.iloc[18:24]

fig, axes = plt.subplots(6, 2, figsize=(12, 24))

for i, (_, row) in enumerate(batch.iterrows()):
    original_path = Path(row["original_jpeg"])
    twin_path = Path(row["twin_path"])

    assert original_path.is_file(), f"Missing original: {original_path}"
    assert twin_path.is_file(), f"Missing twin: {twin_path}"

    with Image.open(original_path) as img:
        axes[i, 0].imshow(img.convert("RGB"))

    with Image.open(twin_path) as img:
        axes[i, 1].imshow(img.convert("RGB"))

    axes[i, 0].set_title(f'Original: {row["id"]}')
    axes[i, 1].set_title(
        f'SDXL twin | Similarity: {row["similarity"]:.4f}'
    )

    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

print("RESTORATION SUCCESSFUL")
print("Displayed pairs: 19–24")
print("Remaining after this batch: 6 pairs (25–30)")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = "/content/drive/MyDrive/twincheck"
sample = pd.read_csv(
    f"{ROOT}/features/vision/sdxl_visual_inspection_sample_30.csv"
)

batch = sample.iloc[24:30]

fig, axes = plt.subplots(6, 2, figsize=(12, 24))

for i, (_, row) in enumerate(batch.iterrows()):
    with Image.open(row["original_jpeg"]) as img:
        axes[i, 0].imshow(img.convert("RGB"))

    with Image.open(row["twin_path"]) as img:
        axes[i, 1].imshow(img.convert("RGB"))

    axes[i, 0].set_title(f'Original: {row["id"]}')
    axes[i, 1].set_title(
        f'SDXL twin | Similarity: {row["similarity"]:.4f}'
    )
    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")
sample_path = ROOT / "features/vision/sdxl_visual_inspection_sample_30.csv"

df = pd.read_csv(sample_path)

observations = [
    ("Traffic-light scene preserved; minor text and appearance differences.", "Review faces"),
    ("Buildings, crowd and smoke preserved.", "Review faces"),
    ("Portrait and background preserved; prominent face.", "Face review"),
    ("Soldier and background preserved; prominent face.", "Face review"),
    ("Indoor scene and people preserved; minor differences.", "Review faces"),
    ("Shoes and their shapes preserved; minor texture differences.", "No specific concern noted"),
    ("Ship and smoke plume preserved; minor text differences.", "No specific concern noted"),
    ("Historical street, vehicles and buildings preserved.", "No specific concern noted"),
    ("Cityscape and waterway preserved.", "No specific concern noted"),
    ("Drone, aircraft and background preserved.", "No specific concern noted"),
    ("Signs and illustrations preserved; fine text differences.", "No specific concern noted"),
    ("Flood scene and vehicles preserved; minor text differences.", "Review faces"),
    ("Wildfire map preserved; no obvious structural change.", "No specific concern noted"),
    ("Bomb and surrounding scene preserved; Arabic text broadly consistent.", "Review faces"),
    ("Crowd and people preserved; no obvious major structural change.", "Review faces"),
    ("Tornado scene and overall structure preserved.", "No specific concern noted"),
    ("Social-media screenshot and people preserved; fine text differences.", "Face and text review"),
    ("Rescue-at-sea scene preserved.", "Review faces"),
    ("People and flags preserved; several visible faces.", "Face review"),
    ("Fire and smoke scene preserved; minor Arabic text differences.", "No specific concern noted"),
    ("Fire scene and surrounding buildings preserved.", "Review faces"),
    ("Person and camp scene preserved; prominent face.", "Face review"),
    ("Crowd and street scene preserved.", "Review faces"),
    ("Three people and their positions preserved; visible faces.", "Face review"),
    ("Man and clothing preserved; prominent visible face.", "Face review"),
    ("Hospital scene and surrounding people preserved; minor text differences.", "Review faces"),
    ("Road accident scene and vehicle positions preserved; visible faces.", "Face review"),
    ("Social-media screenshot and seated person preserved; prominent face.", "Face review"),
    ("Airplane cabin scene and people preserved.", "Review faces"),
    ("Composition preserved; face is blurred in original and twin.", "No additional concern noted"),
]

assert len(df) == len(observations) == 30

df["visual_observation"] = [x[0] for x in observations]
df["manual_review_flag"] = [x[1] for x in observations]
df["visual_inspection_status"] = "Inspected"

out = ROOT / "features/vision/sdxl_visual_inspection_results_30.csv"
df.to_csv(out, index=False)

print("Saved:", out)
print("Inspected pairs:", len(df))
print("\nReview flags:")
print(df["manual_review_flag"].value_counts().to_string())


In [ ]:

from pathlib import Path
from PIL import Image
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")

manifests = {
    "VERITE_VAE": (
        ROOT / "features/vision/verite/image_twins/verite_face_filtered_vae_twin_manifest.csv",
        "twin_path",
    ),
    "M4FC_VAE": (
        ROOT / "features/vision/m4fc/image_twins/m4fc_face_filtered_vae_twin_manifest.csv",
        "twin_path",
    ),
    "SDXL_ACCEPTED": (
        ROOT / "data/copies/vision/sdxl_accepted_twin_manifest.csv",
        "twin_path",
    ),
}

for name, (manifest_path, path_col) in manifests.items():
    df = pd.read_csv(manifest_path)
    valid = 0
    invalid = []

    for path in df[path_col]:
        try:
            with Image.open(path) as img:
                img.verify()
            if Path(path).suffix.lower() in [".jpg", ".jpeg"]:
                valid += 1
            else:
                invalid.append((path, "Not JPEG"))
        except Exception as e:
            invalid.append((path, str(e)))

    print(f"\n===== {name} =====")
    print("Manifest entries:", len(df))
    print("Valid JPEG images:", valid)
    print("Invalid images:", len(invalid))
    if invalid:
        print("Examples:", invalid[:3])

# Check SDXL original JPEG copies
original_dir = ROOT / "data/copies/vision/sdxl_originals_jpeg"
originals = list(original_dir.glob("*.jpg")) + list(original_dir.glob("*.jpeg"))

print("\n===== SDXL ORIGINAL JPEGs =====")
print("JPEG files found:", len(originals))
print("Folder:", original_dir)


In [ ]:

from pathlib import Path
from PIL import Image
from collections import Counter

folder = Path(
    "/content/drive/MyDrive/twincheck/data/copies/vision/sdxl_originals_jpeg"
)

files = sorted(list(folder.glob("*.jpg")) + list(folder.glob("*.jpeg")))

valid = 0
invalid = []

for path in files:
    try:
        with Image.open(path) as img:
            img.verify()
            if img.format == "JPEG":
                valid += 1
            else:
                invalid.append((path.name, img.format))
    except Exception as e:
        invalid.append((path.name, str(e)))

print("SDXL original files:", len(files))
print("Valid JPEG images:", valid)
print("Invalid or non-JPEG images:", len(invalid))
print("All 249 originals valid:", len(files) == 249 and valid == 249)

if invalid:
    print("\nInvalid examples:", invalid[:5])


In [ ]:

from pathlib import Path
import numpy as np

ROOT = Path("/content/drive/MyDrive/twincheck")

checks = {
    "VERITE SigLIP2 image": (
        "features/vision/verite/siglip2/verified_image_extraction/"
        "verite_siglip2_image_projected_embeddings.npy", 1152),
    "VERITE SigLIP2 caption": (
        "features/vision/verite/siglip2/verified_caption_extraction/"
        "verite_siglip2_caption_projected_embeddings.npy", 1152),
    "VERITE CLIP ViT-L/14 image": (
        "features/vision/verite/clip_vit_l14/verified_image_extraction/"
        "verite_clip_vit_l14_image_projected_embeddings.npy", 768),
    "GenImage SigLIP2 image": (
        "features/vision/genimage/"
        "genimage_siglip2_image_projected_embeddings.npy", 1152),
    "GenImage CLIP ViT-L/14 image": (
        "features/vision/genimage/"
        "genimage_clip_vit_l14_image_projected_embeddings.npy", 768),
    "M4FC SigLIP2 image": (
        "features/vision/m4fc/siglip2/"
        "m4fc_siglip2_image_projected_embeddings.npy", 1152),
    "M4FC CLIP ViT-L/14 image": (
        "features/vision/m4fc/clip_vit_l14/"
        "m4fc_clip_vit_l14_image_embeddings.npy", 768),
}

all_passed = True

for name, (relative_path, expected_dim) in checks.items():
    path = ROOT / relative_path
    print(f"\n{name}")

    if not path.exists():
        print("  MISSING")
        all_passed = False
        continue

    arr = np.load(path, mmap_mode="r")
    finite = bool(np.isfinite(arr).all())
    correct_dim = arr.ndim == 2 and arr.shape[1] == expected_dim

    norms = np.linalg.norm(arr, axis=1) if arr.ndim == 2 else np.array([])
    normalized = bool(
        len(norms) > 0 and np.allclose(norms, 1.0, atol=1e-3)
    )

    passed = finite and correct_dim and normalized
    all_passed &= passed

    print("  Shape:", arr.shape)
    print("  Expected dimension:", expected_dim)
    print("  Finite:", finite)
    print("  Normalized:", normalized)
    print("  Result:", "PASS" if passed else "CHECK")

# Locate M4FC caption embedding candidates for verification
print("\nM4FC caption embedding candidates:")
m4fc_folder = ROOT / "features/vision/m4fc/siglip2"

for path in m4fc_folder.rglob("*.npy"):
    if "caption" in path.name.lower() or "text" in str(path.parent).lower():
        arr = np.load(path, mmap_mode="r")
        print(path.relative_to(ROOT), arr.shape)

print("\nOVERALL IMAGE EMBEDDING AUDIT:",
      "PASS" if all_passed else "REQUIRES ATTENTION")


In [ ]:

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/twincheck")
OUT = ROOT / "features/vision/F2_completion_audit.json"

report = {
    "task": "F2 - Vision features and image twins",
    "embedding_audit": {
        "VERITE": {
            "siglip2_images": 663,
            "siglip2_captions": 1001,
            "clip_vit_l14_images": 663
        },
        "GenImage": {
            "siglip2_images": 18000,
            "clip_vit_l14_images": 18000
        },
        "M4FC": {
            "siglip2_images": 603,
            "clip_vit_l14_images": 603,
            "siglip2_arabic_claim_embeddings": 770
        },
        "all_checked_image_embeddings_passed": True
    },
    "vae_twins": {
        "VERITE": {
            "accepted": 311,
            "valid_jpegs": 311,
            "minimum_similarity": 0.9209
        },
        "M4FC": {
            "accepted": 142,
            "valid_jpegs": 142,
            "minimum_similarity": 0.9241
        },
        "total_accepted": 453
    },
    "sdxl_twins": {
        "strength": 0.15,
        "seed": 42,
        "generated": 249,
        "accepted": 157,
        "rejected": 92,
        "missing_files": 0,
        "original_jpegs_valid": 249,
        "visual_pairs_inspected": 30,
        "visual_inspection_file":
            "features/vision/sdxl_visual_inspection_results_30.csv"
    },
    "limitations_and_pending_items": [
        "SDXL evaluation source scope was 249 images, not the original 500-image target.",
        "Identifiable-face concerns were flagged in the 30-pair visual sample and require resolution.",
        "M4FC Arabic claim embeddings should not be described as image-caption embeddings."
    ],
    "status": "Technical audits passed; outstanding scope and face-review items remain."
}

OUT.parent.mkdir(parents=True, exist_ok=True)

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4, ensure_ascii=False)

print("F2 audit report saved:")
print(OUT)
print("\nStatus:", report["status"])


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")
csv_path = ROOT / "features/vision/sdxl_visual_inspection_results_30.csv"

df = pd.read_csv(csv_path)

# Select entries requiring face review
flagged = df[
    df["manual_review_flag"].str.contains("face", case=False, na=False)
].reset_index(drop=True)

print("Pairs requiring face review:", len(flagged))
print("Displaying flagged pairs 1–5")

batch = flagged.iloc[:5]

fig, axes = plt.subplots(len(batch), 2, figsize=(12, 4 * len(batch)))

if len(batch) == 1:
    axes = [axes]

for i, (_, row) in enumerate(batch.iterrows()):
    with Image.open(row["original_jpeg"]) as img:
        axes[i][0].imshow(img.convert("RGB"))

    with Image.open(row["twin_path"]) as img:
        axes[i][1].imshow(img.convert("RGB"))

    axes[i][0].set_title(f'Original: {row["id"]}')
    axes[i][1].set_title(f'SDXL twin: {row["id"]}')
    axes[i][0].axis("off")
    axes[i][1].axis("off")

plt.tight_layout()
plt.show()

print("\nImage IDs:")
print(batch["id"].tolist())


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = "/content/drive/MyDrive/twincheck"

df = pd.read_csv(
    f"{ROOT}/features/vision/sdxl_visual_inspection_results_30.csv"
)

flagged = df[
    df["manual_review_flag"].str.contains("face", case=False, na=False)
].reset_index(drop=True)

batch = flagged.iloc[5:10]

fig, axes = plt.subplots(5, 2, figsize=(12, 20))

for i, (_, row) in enumerate(batch.iterrows()):
    with Image.open(row["original_jpeg"]) as img:
        axes[i, 0].imshow(img.convert("RGB"))

    with Image.open(row["twin_path"]) as img:
        axes[i, 1].imshow(img.convert("RGB"))

    axes[i, 0].set_title(f'Original: {row["id"]}')
    axes[i, 1].set_title(f'SDXL twin: {row["id"]}')
    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

print("Image IDs:", batch["id"].tolist())


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = "/content/drive/MyDrive/twincheck"

df = pd.read_csv(
    f"{ROOT}/features/vision/sdxl_visual_inspection_results_30.csv"
)

flagged = df[
    df["manual_review_flag"].str.contains("face", case=False, na=False)
].reset_index(drop=True)

batch = flagged.iloc[10:15]

print("Image IDs:", batch["id"].tolist())

fig, axes = plt.subplots(len(batch), 2, figsize=(12, 20))

for i, (_, row) in enumerate(batch.iterrows()):
    with Image.open(row["original_jpeg"]) as img:
        axes[i, 0].imshow(img.convert("RGB"))

    with Image.open(row["twin_path"]) as img:
        axes[i, 1].imshow(img.convert("RGB"))

    axes[i, 0].set_title(f'Original: {row["id"]}')
    axes[i, 1].set_title(f'SDXL twin: {row["id"]}')
    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = "/content/drive/MyDrive/twincheck"

df = pd.read_csv(
    f"{ROOT}/features/vision/sdxl_visual_inspection_results_30.csv"
)

flagged = df[
    df["manual_review_flag"].str.contains("face", case=False, na=False)
].reset_index(drop=True)

batch = flagged.iloc[15:20]

print("Image IDs:", batch["id"].tolist())

fig, axes = plt.subplots(len(batch), 2, figsize=(12, 20))

for i, (_, row) in enumerate(batch.iterrows()):
    with Image.open(row["original_jpeg"]) as img:
        axes[i, 0].imshow(img.convert("RGB"))

    with Image.open(row["twin_path"]) as img:
        axes[i, 1].imshow(img.convert("RGB"))

    axes[i, 0].set_title(f'Original: {row["id"]}')
    axes[i, 1].set_title(f'SDXL twin: {row["id"]}')
    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")
VISION = ROOT / "data/copies/vision"

accepted_path = VISION / "sdxl_accepted_twin_manifest.csv"

# Images manually identified for exclusion
exclusion_reasons = {
    "m4fc_000470": "Large identifiable face",
    "m4fc_000518": "Large identifiable face",
    "m4fc_000397": "Large identifiable face",
    "m4fc_000445": "Two prominent identifiable faces",
    "m4fc_000664": "Multiple prominent identifiable faces",
    "true_91.jpg": "Clearly visible identifiable face",
    "true_38.jpg": "Three clearly visible identifiable faces",
    "true_146.jpg": "Large identifiable face",
    "m4fc_000449": "Large identifiable face",
}

# Load original accepted manifest
accepted = pd.read_csv(accepted_path)

assert "id" in accepted.columns, "ID column not found"

# Check that every exclusion candidate exists
available_ids = set(accepted["id"].astype(str))
missing_ids = set(exclusion_reasons) - available_ids

print("Original accepted twins:", len(accepted))
print("Exclusion candidates:", len(exclusion_reasons))

if missing_ids:
    print("STOP: These IDs were not found:", sorted(missing_ids))
    raise ValueError("Review the IDs before filtering.")

# Create exclusion manifest
excluded = accepted[
    accepted["id"].astype(str).isin(exclusion_reasons)
].copy()

excluded["exclusion_reason"] = excluded["id"].astype(str).map(
    exclusion_reasons
)

# Create filtered manifest
filtered = accepted[
    ~accepted["id"].astype(str).isin(exclusion_reasons)
].copy()

# Save separate outputs; original manifest remains unchanged
excluded_path = VISION / "sdxl_face_excluded_manifest.csv"
filtered_path = VISION / "sdxl_face_filtered_accepted_manifest.csv"

excluded.to_csv(excluded_path, index=False)
filtered.to_csv(filtered_path, index=False)

print("\n===== FACE FILTER RESULTS =====")
print("Original accepted:", len(accepted))
print("Excluded:", len(excluded))
print("Remaining accepted:", len(filtered))
print("Excluded manifest:", excluded_path)
print("Filtered manifest:", filtered_path)

assert len(accepted) == len(excluded) + len(filtered)
print("\nFiltering completed successfully.")


In [ ]:

import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")
VISION = ROOT / "data/copies/vision"

path = VISION / "sdxl_face_filtered_accepted_manifest.csv"
df = pd.read_csv(path)

file_exists = df["twin_path"].map(lambda p: Path(p).is_file())
similarity_ok = df["similarity"] >= 0.92

print("Filtered SDXL manifest audit")
print("Rows:", len(df))
print("Existing twin files:", int(file_exists.sum()))
print("Missing twin files:", int((~file_exists).sum()))
print("Similarity >= 0.92:", int(similarity_ok.sum()))
print("Similarity below 0.92:", int((~similarity_ok).sum()))
print("All accepted flags:", bool(df["accepted"].all()))

assert len(df) == 148
assert file_exists.all()
assert similarity_ok.all()
assert df["accepted"].all()

print("\nPASS: All 148 filtered twins passed the checks.")


In [ ]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/twincheck")
report_path = ROOT / "features/vision/F2_completion_audit.json"

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

report["sdxl_twins"].update({
    "generated": 249,
    "similarity_accepted_before_face_review": 157,
    "similarity_rejected": 92,
    "manually_excluded_for_faces": 9,
    "final_accepted_after_face_review": 148,
    "final_filtered_manifest":
        "data/copies/vision/sdxl_face_filtered_accepted_manifest.csv",
    "face_exclusion_manifest":
        "data/copies/vision/sdxl_face_excluded_manifest.csv",
    "final_file_audit_passed": True
})

report["limitations_and_pending_items"] = [
    "SDXL evaluation source scope was 249 images rather than the original 500-image target; team agreement and documentation are required.",
    "Face exclusions were based on manual review of the 30-pair sample, not automated face detection or exhaustive review of all generated images.",
    "M4FC Arabic claim embeddings are claim embeddings, not image-caption embeddings."
]

report["status"] = (
    "Technical audits passed; SDXL face filtering completed. "
    "Original SDXL scope limitation and remaining F2 requirements "
    "must be resolved before final completion."
)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4, ensure_ascii=False)

print("Updated report:", report_path)
print("Final SDXL accepted twins:",
      report["sdxl_twins"]["final_accepted_after_face_review"])
print("Status:", report["status"])


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")
DATA = ROOT / "data"

print("GENIMAGE METADATA INSPECTION")
print("=" * 55)

# Find GenImage-related metadata files
for base in [DATA / "raw", DATA / "processed"]:
    if not base.exists():
        continue

    print(f"\nSearching: {base}")
    matches = [
        p for p in base.rglob("*")
        if p.is_file()
        and "genimage" in str(p).lower()
        and p.suffix.lower() in {".csv", ".json", ".jsonl", ".parquet", ".txt"}
    ]

    for p in matches[:40]:
        print(p.relative_to(ROOT))

    if len(matches) > 40:
        print(f"... and {len(matches) - 40} more files")

# Inspect the existing GenImage embedding manifest
manifest = ROOT / "features/vision/genimage/genimage_image_embedding_manifest.csv"

if manifest.exists():
    df = pd.read_csv(manifest, nrows=5)
    print("\nGENIMAGE EMBEDDING MANIFEST")
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))
else:
    print("\nEmbedding manifest not found:", manifest)

# Inspect a few JSON/JSONL metadata records, if available
metadata_files = []
for base in [DATA / "raw", DATA / "processed"]:
    if base.exists():
        metadata_files.extend([
            p for p in base.rglob("*")
            if p.is_file()
            and "genimage" in str(p).lower()
            and p.suffix.lower() in {".json", ".jsonl"}
        ])

for p in metadata_files[:3]:
    print(f"\nSAMPLE CONTENT: {p.relative_to(ROOT)}")
    try:
        with open(p, "r", encoding="utf-8") as f:
            if p.suffix.lower() == ".jsonl":
                for _ in range(2):
                    line = f.readline()
                    if line:
                        print(line[:1000])
            else:
                obj = json.load(f)
                if isinstance(obj, list):
                    print(obj[:2])
                elif isinstance(obj, dict):
                    print("Top-level keys:", list(obj.keys())[:30])
                    print(str(obj)[:1500])
    except Exception as e:
        print("Could not inspect:", repr(e))

print("\nInspection complete.")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")

manifests = {
    "VERITE": ROOT / "features/vision/verite/image_twins/verite_face_filtered_vae_twin_manifest.csv",
    "M4FC": ROOT / "features/vision/m4fc/image_twins/m4fc_face_filtered_vae_twin_manifest.csv",
}

for dataset, path in manifests.items():
    print(f"\n{'='*60}")
    print(dataset)
    print(f"Manifest: {path}")
    print("Exists:", path.exists())

    if not path.exists():
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    # Display path-related columns and a few actual records
    path_cols = [
        c for c in df.columns
        if "path" in c.lower() or "file" in c.lower()
    ]

    if path_cols:
        print("\nPath examples:")
        print(df[path_cols].head(3).to_string(index=False))
    else:
        print("\nNo path-related columns found.")

print("\nManifest inspection complete. No files modified.")


In [ ]:

from pathlib import Path
from PIL import Image
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")

manifest_paths = {
    "VERITE twins": ROOT / "features/vision/verite/image_twins/verite_face_filtered_vae_twin_manifest.csv",
    "M4FC twins and originals": ROOT / "features/vision/m4fc/image_twins/m4fc_face_filtered_vae_twin_manifest.csv",
}

records = []

for name, manifest_path in manifest_paths.items():
    df = pd.read_csv(manifest_path)

    columns = ["twin_path"]
    if "original_path" in df.columns:
        columns.insert(0, "original_path")

    for _, row in df.iterrows():
        for column in columns:
            path = Path(str(row[column]))
            record = {
                "dataset": name,
                "role": column,
                "path": str(path),
                "exists": path.is_file(),
                "format": None,
                "width": None,
                "height": None,
                "size_kb": None,
                "has_jpeg_quantization_tables": None,
            }

            if path.is_file():
                record["size_kb"] = round(path.stat().st_size / 1024, 2)
                try:
                    with Image.open(path) as img:
                        record["format"] = img.format
                        record["width"], record["height"] = img.size
                        record["has_jpeg_quantization_tables"] = bool(
                            getattr(img, "quantization", None)
                        )
                except Exception as e:
                    record["format"] = f"READ_ERROR: {e}"

            records.append(record)

audit = pd.DataFrame(records)

print("FILE AUDIT SUMMARY")
print("=" * 65)

print(audit.groupby(["dataset", "role", "format"], dropna=False)
      .agg(files=("path", "count"),
           existing=("exists", "sum"))
      .to_string())

print("\nMissing files:", int((~audit["exists"]).sum()))
print("Non-JPEG files:", int((audit["format"] != "JPEG").sum()))
print("Unreadable files:", int(audit["format"].fillna("").str.startswith("READ_ERROR").sum()))

print("\nSample records:")
print(audit.head(10).to_string(index=False))

output = ROOT / "features/vision/vae_jpeg_file_audit.csv"
audit.to_csv(output, index=False)
print("\nAudit saved to:", output)

print("\nNote: JPEG format and quantization tables do not prove the original quality=85 setting.")


In [ ]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/twincheck")
extensions = {".py", ".ipynb"}

terms = [
    "quality=85",
    'quality = 85',
    "quality=",
    "format='JPEG'",
    'format="JPEG"',
    "Image.save",
    ".save("
]

matches = 0

print("SEARCHING VAE GENERATION CODE FOR JPEG SETTINGS")
print("=" * 65)

for path in ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue

    try:
        if path.stat().st_size > 10_000_000:
            continue

        content = path.read_text(encoding="utf-8", errors="ignore")

        if path.suffix.lower() == ".ipynb":
            notebook = json.loads(content)
            source = "\n".join(
                "".join(cell.get("source", []))
                for cell in notebook.get("cells", [])
                if cell.get("cell_type") == "code"
            )
        else:
            source = content

        lines = source.splitlines()
        relevant = [
            (i, line.strip())
            for i, line in enumerate(lines)
            if any(term.lower() in line.lower() for term in terms)
        ]

        if relevant:
            print(f"\nFILE: {path.relative_to(ROOT)}")
            for i, line in relevant:
                print(f"Line {i+1}: {line[:250]}")
                matches += 1

    except Exception:
        continue

print("\n" + "=" * 65)
print("Matching lines found:", matches)
print("Search complete. No files modified.")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")

checkpoints = [
    ROOT / "features/vision/verite/image_twins/verite_corrected_vae_generation_checkpoint.csv",
    ROOT / "features/vision/m4fc/image_twins/m4fc_corrected_vae_generation_checkpoint.csv",
]

for path in checkpoints:
    print("\n" + "=" * 65)
    print(path.name)
    print("Exists:", path.exists())

    if path.exists():
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))

print("\nCheckpoint inspection complete. No files modified.")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")

files = {
    "VERITE VAE accepted": ROOT / "features/vision/verite/image_twins/verite_face_filtered_vae_twin_manifest.csv",
    "M4FC VAE accepted": ROOT / "features/vision/m4fc/image_twins/m4fc_face_filtered_vae_twin_manifest.csv",
    "SDXL generated": ROOT / "data/copies/vision/sdxl_sources.csv",
    "SDXL similarity accepted": ROOT / "data/copies/vision/sdxl_accepted_twin_manifest.csv",
    "SDXL face-filtered accepted": ROOT / "data/copies/vision/sdxl_face_filtered_accepted_manifest.csv",
    "SDXL rejected": ROOT / "data/copies/vision/sdxl_rejected_twin_manifest.csv",
    "SDXL face excluded": ROOT / "data/copies/vision/sdxl_face_excluded_manifest.csv",
}

print("F2 FINAL TWIN COUNT AUDIT")
print("=" * 65)

for name, path in files.items():
    print(f"\n{name}")
    print("File exists:", path.exists())

    if path.exists():
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())

print("\nAudit complete. No files modified.")


In [ ]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/twincheck")

audit_path = ROOT / "features/vision/F2_completion_audit.json"

print("F2 COMPLETION AUDIT")
print("=" * 60)

if audit_path.exists():
    with open(audit_path, "r", encoding="utf-8") as f:
        audit = json.load(f)
    print(json.dumps(audit, indent=2, ensure_ascii=False))
else:
    print("Audit JSON not found:", audit_path)

print("\nPOSSIBLE F2 REPORT FILES")
print("=" * 60)

for folder in [ROOT / "features/vision", ROOT / "reports"]:
    if folder.exists():
        for p in sorted(folder.glob("*")):
            if p.is_file() and ("F2" in p.name.upper() or "VISION" in p.name.upper()):
                print(p)

print("\nInspection complete. No files modified.")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")

manifest_files = {
    "VERITE_VAE": ROOT / "features/vision/verite/image_twins/verite_face_filtered_vae_twin_manifest.csv",
    "M4FC_VAE": ROOT / "features/vision/m4fc/image_twins/m4fc_face_filtered_vae_twin_manifest.csv",
    "SDXL": ROOT / "data/copies/vision/sdxl_face_filtered_accepted_manifest.csv",
}

for name, path in manifest_files.items():
    print("\n" + "=" * 75)
    print(name)
    print("Manifest exists:", path.exists())

    if not path.exists():
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    for col in ["original_path", "source_image_path", "twin_path"]:
        if col in df.columns:
            exists = df[col].dropna().apply(
                lambda p: Path(str(p)).exists()
            )
            print(f"{col}: {exists.sum()}/{len(exists)} paths exist")

    if "split" in df.columns:
        print("Split counts:")
        print(df["split"].value_counts(dropna=False).to_dict())

    if name == "SDXL" and "dataset" in df.columns:
        print("Dataset counts:")
        print(df["dataset"].value_counts(dropna=False).to_dict())

        verite = df[df["dataset"].astype(str).str.lower() == "verite"]
        if len(verite) and "split" in verite.columns:
            train_count = (
                verite["split"].astype(str).str.lower() == "train"
            ).sum()
            print("VERITE train sources:", train_count)

    print("\nFirst 10 records:")
    display(df.head(10))

print("\nInspection complete. No files modified.")


In [ ]:

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

print("GOOGLE DRIVE MOUNT:", DRIVE.exists())

print("\nPOSSIBLE PROJECT FOLDERS")
for name in ["TwinCheck", "twincheck"]:
    p = DRIVE / name
    print(f"{p}: exists={p.exists()}, directory={p.is_dir()}")

print("\nSEARCHING FOR F2 MANIFESTS")
target_names = {
    "verite_face_filtered_vae_twin_manifest.csv",
    "m4fc_face_filtered_vae_twin_manifest.csv",
    "sdxl_face_filtered_accepted_manifest.csv",
}

found = {name: [] for name in target_names}

for project_name in ["TwinCheck", "twincheck"]:
    root = DRIVE / project_name
    if root.exists():
        for p in root.rglob("*.csv"):
            if p.name in target_names:
                found[p.name].append(str(p))

for name, paths in found.items():
    print(f"\n{name}")
    if paths:
        for p in paths:
            print(" ", p)
    else:
        print("  Not found")

print("\nSearch complete. No files modified.")


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

print("\nGoogle Drive mounted:", DRIVE.exists())

for name in ["TwinCheck", "twincheck"]:
    path = DRIVE / name
    print(f"{name}: exists={path.exists()}, directory={path.is_dir()}")


In [ ]:

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")

targets = [
    "verite_face_filtered_vae_twin_manifest.csv",
    "m4fc_face_filtered_vae_twin_manifest.csv",
    "sdxl_face_filtered_accepted_manifest.csv",
]

print("PROJECT FOLDER:", ROOT)
print("Exists:", ROOT.exists())

for filename in targets:
    print("\n" + filename)
    matches = list(ROOT.rglob(filename))

    if matches:
        for path in matches:
            print("FOUND:", path)
    else:
        print("NOT FOUND")

print("\nSearch complete. No files modified.")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")

manifest_paths = {
    "VERITE_VAE": ROOT / "features/vision/verite/image_twins/verite_face_filtered_vae_twin_manifest.csv",
    "M4FC_VAE": ROOT / "features/vision/m4fc/image_twins/m4fc_face_filtered_vae_twin_manifest.csv",
    "SDXL": ROOT / "data/copies/vision/sdxl_face_filtered_accepted_manifest.csv",
}

for name, path in manifest_paths.items():
    print("\n" + "=" * 70)
    print(name)

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    for col in ["image_id", "original_path", "source_image_path",
                "original_jpeg", "twin_path", "split", "dataset"]:
        if col in df.columns:
            print(f"\n{col} examples:")
            print(df[col].head(5).to_list())

    for col in ["original_path", "source_image_path",
                "original_jpeg", "twin_path"]:
        if col in df.columns:
            paths = df[col].dropna().astype(str)
            existing = paths.apply(lambda x: Path(x).is_file()).sum()
            print(f"{col}: {existing}/{len(paths)} files exist")

    if name == "SDXL" and "split" in df.columns:
        verite = df[df["dataset"].astype(str).str.lower() == "verite"]
        print("\nVERITE split counts:")
        print(verite["split"].value_counts().to_dict())

print("\nInspection complete. No files modified.")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")

manifest = pd.read_csv(
    ROOT / "features/vision/verite/image_twins/"
    "verite_face_filtered_vae_twin_manifest.csv"
)

search_dirs = [
    ROOT / "data/processed/vision",
    ROOT / "data/raw/vision",
    ROOT / "data/raw/verite",
]

print("VERITE ORIGINAL IMAGE SEARCH")
print("=" * 65)

for _, row in manifest.head(10).iterrows():
    image_id = str(row["image_id"])
    matches = []

    for folder in search_dirs:
        if folder.exists():
            matches.extend(folder.rglob(image_id))

    matches = sorted(set(p for p in matches if p.is_file()))

    print(f"\nImage ID: {image_id}")
    print("Matches:", len(matches))
    for path in matches:
        print(" ", path)

print("\nSearch complete. No files modified.")


In [ ]:

from pathlib import Path
import pandas as pd
from PIL import Image
import json

ROOT = Path("/content/drive/MyDrive/twincheck")
OUT = ROOT / "data/copies/vision/jpeg85_pilot"

verite_manifest = pd.read_csv(
    ROOT / "features/vision/verite/image_twins/"
    "verite_face_filtered_vae_twin_manifest.csv"
)

m4fc_manifest = pd.read_csv(
    ROOT / "features/vision/m4fc/image_twins/"
    "m4fc_face_filtered_vae_twin_manifest.csv"
)

sdxl_manifest = pd.read_csv(
    ROOT / "data/copies/vision/sdxl_face_filtered_accepted_manifest.csv"
)

# Build a pilot of 10 original-twin pairs.
pairs = []

for _, row in verite_manifest.head(4).iterrows():
    original = (
        ROOT / "data/raw/verite/images_download/"
        "extracted/images" / str(row["image_id"])
    )
    pairs.append(("VERITE_VAE", str(row["image_id"]),
                  original, Path(row["twin_path"])))

for _, row in m4fc_manifest.head(3).iterrows():
    pairs.append(("M4FC_VAE", str(row["image_id"]),
                  Path(row["original_path"]),
                  Path(row["twin_path"])))

for _, row in sdxl_manifest.head(3).iterrows():
    pairs.append(("SDXL", str(row["id"]),
                  Path(row["original_jpeg"]),
                  Path(row["twin_path"])))

if len(pairs) != 10:
    raise RuntimeError(f"Expected 10 pairs, found {len(pairs)}")

# Validate every source before creating outputs.
for dataset, image_id, original, twin in pairs:
    if not original.is_file():
        raise FileNotFoundError(f"Original missing: {original}")
    if not twin.is_file():
        raise FileNotFoundError(f"Twin missing: {twin}")

records = []

for dataset, image_id, original, twin in pairs:
    for role, source in [("original", original), ("twin", twin)]:
        destination = OUT / dataset / role / f"{image_id}.jpg"
        destination.parent.mkdir(parents=True, exist_ok=True)

        with Image.open(source) as img:
            rgb = img.convert("RGB")
            original_size = rgb.size
            rgb.save(destination, format="JPEG", quality=85)

        # Verify saved output.
        with Image.open(destination) as check:
            check.verify()

        with Image.open(destination) as check:
            records.append({
                "dataset": dataset,
                "image_id": image_id,
                "role": role,
                "source_path": str(source),
                "output_path": str(destination),
                "format": check.format,
                "dimensions": str(check.size),
                "quality_setting": 85,
                "valid": check.format == "JPEG",
                "dimensions_preserved": check.size == original_size
            })

audit = pd.DataFrame(records)
audit_path = OUT / "jpeg85_pilot_audit.csv"
audit.to_csv(audit_path, index=False)

print("JPEG-85 PILOT")
print("=" * 60)
print("Pairs:", len(pairs))
print("Images saved:", len(audit))
print("Valid JPEGs:", int(audit["valid"].sum()))
print("Dimensions preserved:",
      int(audit["dimensions_preserved"].sum()))
print("Output folder:", OUT)
print("Audit:", audit_path)

display(audit)

if len(audit) != 20 or not audit["valid"].all():
    raise RuntimeError("Pilot validation failed.")

print("\nPilot completed. Original files were not modified.")


In [ ]:

from pathlib import Path
import pandas as pd
from PIL import Image
import re

ROOT = Path("/content/drive/MyDrive/twincheck")
OUT = ROOT / "data/copies/vision/jpeg85_final"

manifests = {
    "VERITE_VAE": ROOT / "features/vision/verite/image_twins/verite_face_filtered_vae_twin_manifest.csv",
    "M4FC_VAE": ROOT / "features/vision/m4fc/image_twins/m4fc_face_filtered_vae_twin_manifest.csv",
    "SDXL": ROOT / "data/copies/vision/sdxl_face_filtered_accepted_manifest.csv",
}

frames = {name: pd.read_csv(path) for name, path in manifests.items()}

pairs = []

for _, row in frames["VERITE_VAE"].iterrows():
    image_id = str(row["image_id"])
    original = ROOT / "data/raw/verite/images_download/extracted/images" / image_id
    pairs.append(("VERITE_VAE", image_id, original, Path(row["twin_path"])))

for _, row in frames["M4FC_VAE"].iterrows():
    pairs.append((
        "M4FC_VAE",
        str(row["image_id"]),
        Path(row["original_path"]),
        Path(row["twin_path"])
    ))

for _, row in frames["SDXL"].iterrows():
    pairs.append((
        "SDXL",
        str(row["id"]),
        Path(row["original_jpeg"]),
        Path(row["twin_path"])
    ))

expected = 311 + 142 + 148
if len(pairs) != expected:
    raise RuntimeError(f"Expected {expected} pairs, found {len(pairs)}")

# Preflight: validate every source before writing any output.
missing = []
for dataset, image_id, original, twin in pairs:
    for role, path in [("original", original), ("twin", twin)]:
        if not path.is_file():
            missing.append((dataset, image_id, role, str(path)))

if missing:
    print("Missing source files:", len(missing))
    for item in missing[:20]:
        print(item)
    raise RuntimeError("Preflight failed. No output files were written.")

print("Preflight passed:", len(pairs), "complete pairs")
print("Creating JPEG quality-85 copies...")

records = []

for dataset, image_id, original, twin in pairs:
    safe_id = re.sub(r"[^A-Za-z0-9_-]", "_", Path(image_id).stem)

    for role, source in [("original", original), ("twin", twin)]:
        filename = f"{safe_id}.jpg" if role == "original" else f"{safe_id}_V.jpg"
        destination = OUT / dataset / role / filename
        destination.parent.mkdir(parents=True, exist_ok=True)

        with Image.open(source) as img:
            rgb = img.convert("RGB")
            source_dimensions = rgb.size
            rgb.save(destination, format="JPEG", quality=85)

        with Image.open(destination) as check:
            check.verify()

        with Image.open(destination) as check:
            valid = check.format == "JPEG"
            dimensions = check.size

        records.append({
            "dataset": dataset,
            "image_id": image_id,
            "role": role,
            "source_path": str(source),
            "output_path": str(destination),
            "format": "JPEG",
            "quality_setting": 85,
            "width": dimensions[0],
            "height": dimensions[1],
            "valid": valid,
            "dimensions_preserved": dimensions == source_dimensions
        })

audit = pd.DataFrame(records)
audit_path = OUT / "jpeg85_conversion_manifest.csv"
audit.to_csv(audit_path, index=False)

print("\nJPEG-85 FULL CONVERSION")
print("=" * 60)
print("Pairs processed:", len(pairs))
print("Images saved:", len(audit))
print("Valid JPEGs:", int(audit["valid"].sum()))
print("Dimensions preserved:", int(audit["dimensions_preserved"].sum()))
print("\nCounts by dataset and role:")
print(audit.groupby(["dataset", "role"]).size())
print("\nOutput directory:", OUT)
print("Conversion manifest:", audit_path)

if len(audit) != expected * 2 or not audit["valid"].all():
    raise RuntimeError("Full conversion validation failed.")

print("\nConversion completed. Original files were not modified.")


In [ ]:

import cv2

print("OpenCV version:", getattr(cv2, "__version__", "Unknown"))
print("OpenCV module path:", getattr(cv2, "__file__", "Unknown"))
print("CascadeClassifier available:",
      hasattr(cv2, "CascadeClassifier"))
print("OpenCV build information available:",
      hasattr(cv2, "getBuildInformation"))

print("\nInstalled OpenCV packages:")
import subprocess
result = subprocess.run(
    ["pip", "list"],
    capture_output=True,
    text=True
)

for line in result.stdout.splitlines():
    if "opencv" in line.lower():
        print(line)


In [ ]:

%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
%pip install opencv-contrib-python==4.10.0.84


In [ ]:
import cv2

print("OpenCV version:", cv2.__version__)
print("CascadeClassifier available:", hasattr(cv2, "CascadeClassifier"))

assert hasattr(cv2, "CascadeClassifier"), "OpenCV installation is still incorrect."

print("OpenCV is ready for face detection.")

In [ ]:

from pathlib import Path
import pandas as pd
import cv2

ROOT = Path("/content/drive/MyDrive/twincheck")
OUT = ROOT / "data/copies/vision/jpeg85_final"

assert ROOT.exists(), "Google Drive is not mounted or project folder is missing."

manifest = pd.read_csv(OUT / "jpeg85_conversion_manifest.csv")
twins = manifest[manifest["role"] == "twin"].copy()

samples = pd.concat([
    twins[twins["dataset"] == "VERITE_VAE"].head(4),
    twins[twins["dataset"] == "M4FC_VAE"].head(3),
    twins[twins["dataset"] == "SDXL"].head(3),
])

cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(cascade_path)

assert not face_cascade.empty(), "Haar cascade could not be loaded."

results = []

for _, row in samples.iterrows():
    image = cv2.imread(str(row["output_path"]))
    assert image is not None, f"Could not read {row['output_path']}"

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    areas = [
        (int(w) * int(h)) / (width * height)
        for x, y, w, h in faces
    ]

    largest_ratio = max(areas, default=0.0)

    results.append({
        "dataset": row["dataset"],
        "image_id": row["image_id"],
        "twin_path": row["output_path"],
        "detected_faces": len(faces),
        "largest_face_area_ratio": largest_ratio,
        "exclude_over_10_percent": largest_ratio > 0.10
    })

pilot = pd.DataFrame(results)
pilot.to_csv(OUT / "face_detection_pilot_10.csv", index=False)

print(pilot.to_string(index=False))
print("\nPilot report saved.")


In [ ]:

from pathlib import Path
import pandas as pd
import cv2

ROOT = Path("/content/drive/MyDrive/twincheck")
OUT = ROOT / "data/copies/vision/jpeg85_final"

manifest = pd.read_csv(OUT / "jpeg85_conversion_manifest.csv")
twins = manifest[manifest["role"] == "twin"].copy()

cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(cascade_path)

assert not face_cascade.empty(), "Haar cascade could not be loaded."

results = []

for index, row in twins.iterrows():
    image_path = Path(row["output_path"])
    image = cv2.imread(str(image_path))

    if image is None:
        raise RuntimeError(f"Could not read image: {image_path}")

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    ratios = [
        (int(w) * int(h)) / (width * height)
        for x, y, w, h in faces
    ]

    largest_ratio = max(ratios, default=0.0)

    results.append({
        "dataset": row["dataset"],
        "image_id": row["image_id"],
        "twin_path": str(image_path),
        "detected_faces": len(faces),
        "largest_face_area_ratio": largest_ratio,
        "exclude_over_10_percent": largest_ratio > 0.10
    })

face_report = pd.DataFrame(results)
report_path = OUT / "face_detection_all_twins.csv"
face_report.to_csv(report_path, index=False)

print("Total twins screened:", len(face_report))
print("Twins flagged for exclusion:",
      int(face_report["exclude_over_10_percent"].sum()))
print("Twins not flagged:",
      int((~face_report["exclude_over_10_percent"]).sum()))

print("\nFlagged images by dataset:")
print(
    face_report[face_report["exclude_over_10_percent"]]
    .groupby("dataset")
    .size()
    .to_string()
)

print("\nFull report saved to:", report_path)


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")
OUT = ROOT / "data/copies/vision/jpeg85_final"

report = pd.read_csv(OUT / "face_detection_all_twins.csv")
flagged = report[report["exclude_over_10_percent"]].copy()

print(
    flagged[
        ["dataset", "image_id", "detected_faces",
         "largest_face_area_ratio", "twin_path"]
    ].to_string(index=False)
)

In [ ]:

from pathlib import Path
import pandas as pd
import cv2
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/twincheck")
OUT = ROOT / "data/copies/vision/jpeg85_final"

report = pd.read_csv(OUT / "face_detection_all_twins.csv")
flagged = report[report["exclude_over_10_percent"]]

cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (_, row) in zip(axes, flagged.iterrows()):
    image = cv2.imread(row["twin_path"])
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    faces = cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
    )

    for x, y, w, h in faces:
        cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 3)

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    ax.imshow(image)
    ax.set_title(
        f"{row['dataset']} | {row['image_id']}\n"
        f"Face area: {row['largest_face_area_ratio']:.1%}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")
OUT = ROOT / "data/copies/vision/jpeg85_final"

report = pd.read_csv(OUT / "face_detection_all_twins.csv")

manual_exclusions = {
    ("M4FC_VAE", "m4fc_000478")
}

report["manual_review"] = "keep"
report["manual_exclusion"] = False

for idx, row in report.iterrows():
    key = (row["dataset"], row["image_id"])
    if key in manual_exclusions:
        report.loc[idx, "manual_review"] = "exclude_large_identifiable_face"
        report.loc[idx, "manual_exclusion"] = True
    elif row["exclude_over_10_percent"]:
        report.loc[idx, "manual_review"] = "keep_false_positive"

report.to_csv(OUT / "face_detection_manual_review.csv", index=False)

print("Total twins reviewed:", len(report))
print("Manually confirmed exclusions:", int(report["manual_exclusion"].sum()))
print("Automatic false positives retained:",
      int(((report["exclude_over_10_percent"]) &
           (~report["manual_exclusion"])).sum()))
print("Saved:", OUT / "face_detection_manual_review.csv")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/twincheck")
OUT = ROOT / "data/copies/vision/jpeg85_final"

conversion = pd.read_csv(OUT / "jpeg85_conversion_manifest.csv")
review = pd.read_csv(OUT / "face_detection_manual_review.csv")

exclusions = set(
    zip(
        review.loc[review["manual_exclusion"], "dataset"],
        review.loc[review["manual_exclusion"], "image_id"]
    )
)

conversion["manual_exclusion"] = conversion.apply(
    lambda r: (r["dataset"], r["image_id"]) in exclusions,
    axis=1
)

final_manifest = conversion[~conversion["manual_exclusion"]].copy()
excluded_manifest = conversion[conversion["manual_exclusion"]].copy()

final_path = OUT / "jpeg85_final_manifest.csv"
excluded_path = OUT / "jpeg85_excluded_manifest.csv"

final_manifest.to_csv(final_path, index=False)
excluded_manifest.to_csv(excluded_path, index=False)

# Validate that every retained pair has both files.
pair_counts = final_manifest.groupby(
    ["dataset", "image_id"]
)["role"].nunique()

assert (pair_counts == 2).all(), "Some retained pairs are incomplete."

missing = [
    p for p in final_manifest["output_path"]
    if not Path(p).is_file()
]

assert not missing, f"Missing files: {missing[:5]}"

print("Retained pairs:", final_manifest.groupby(
    ["dataset", "image_id"]
).ngroups)
print("Retained files:", len(final_manifest))
print("Excluded pairs:", excluded_manifest.groupby(
    ["dataset", "image_id"]
).ngroups)
print("Missing retained files:", len(missing))
print("\nRetained pairs by dataset:")
print(
    final_manifest.groupby(["dataset", "role"]).size().to_string()
)
print("\nFinal manifest:", final_path)
print("Exclusion manifest:", excluded_path)


In [ ]:

# ============================================================
# TWINCHECK F2 — ENVIRONMENT RECOVERY AND VALIDATION
# ============================================================

from pathlib import Path
import os
import sys
import subprocess
import importlib.util
import pandas as pd
import numpy as np
import torch

print("=" * 65)
print("1. PYTHON AND GPU")
print("=" * 65)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1e9, 2
    ))
else:
    raise RuntimeError("GPU is not active. Stop here.")

print("\n" + "=" * 65)
print("2. GOOGLE DRIVE")
print("=" * 65)

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE = Path("/content/drive/MyDrive")
assert DRIVE.exists(), "Google Drive is not accessible."

print("Drive mounted:", DRIVE.exists())

print("\n" + "=" * 65)
print("3. LOCATE TWINCHECK PROJECT")
print("=" * 65)

# Check the known lowercase path first.
candidates = [
    DRIVE / "twincheck",
    DRIVE / "TwinCheck"
]

ROOT = next((p for p in candidates if p.is_dir()), None)

if ROOT is None:
    print("Known project paths not found.")
    print("MyDrive folders:", [p.name for p in DRIVE.iterdir() if p.is_dir()])
    raise FileNotFoundError(
        "TwinCheck folder was not found. Check the Drive folder location."
    )

print("Project root:", ROOT)

OUT = ROOT / "data/copies/vision/jpeg85_final"

assert OUT.is_dir(), f"JPEG-85 output folder not found: {OUT}"

print("JPEG-85 folder:", OUT)

print("\n" + "=" * 65)
print("4. CHECK REQUIRED MANIFESTS")
print("=" * 65)

required_files = [
    "jpeg85_conversion_manifest.csv",
    "face_detection_all_twins.csv",
    "face_detection_manual_review.csv",
    "jpeg85_final_manifest.csv",
    "jpeg85_excluded_manifest.csv"
]

for filename in required_files:
    path = OUT / filename
    print(f"{filename}: {'FOUND' if path.is_file() else 'MISSING'}")

missing = [f for f in required_files if not (OUT / f).is_file()]

if missing:
    raise FileNotFoundError(
        "Required files are missing: " + ", ".join(missing)
    )

print("\n" + "=" * 65)
print("5. VERIFY FINAL MANIFEST AND IMAGE FILES")
print("=" * 65)

manifest = pd.read_csv(OUT / "jpeg85_final_manifest.csv")
excluded = pd.read_csv(OUT / "jpeg85_excluded_manifest.csv")

print("Final image rows:", len(manifest))
print("Excluded image rows:", len(excluded))

pairs = manifest.groupby(["dataset", "image_id"])["role"].nunique()

print("Unique retained pairs:", len(pairs))
print("Incomplete pairs:", int((pairs != 2).sum()))

missing_images = [
    p for p in manifest["output_path"]
    if not Path(p).is_file()
]

print("Missing retained image files:", len(missing_images))

print("\nRetained dataset counts:")
print(manifest.groupby(["dataset", "role"]).size().to_string())

assert len(manifest) == 1200, "Unexpected retained image count."
assert len(pairs) == 600, "Unexpected retained pair count."
assert (pairs == 2).all(), "Some image pairs are incomplete."
assert len(missing_images) == 0, "Some image files are missing."

print("\n" + "=" * 65)
print("6. VERIFY FACE REVIEW")
print("=" * 65)

review = pd.read_csv(OUT / "face_detection_manual_review.csv")

print("Twins reviewed:", len(review))
print("Confirmed exclusions:", int(review["manual_exclusion"].sum()))
print("Automatic false positives retained:",
      int(((review["exclude_over_10_percent"]) &
           (~review["manual_exclusion"])).sum()))

print("\n" + "=" * 65)
print("7. CHECK EMBEDDING LIBRARIES")
print("=" * 65)

for package in ["transformers", "PIL", "open_clip", "timm"]:
    print(package, ":", importlib.util.find_spec(package) is not None)

print("\n" + "=" * 65)
print("RECOVERY AND VALIDATION COMPLETED")
print("=" * 65)
print("GPU:", torch.cuda.get_device_name(0))
print("Project:", ROOT)
print("Final retained pairs:", len(pairs))
print("Final retained image files:", len(manifest))
print("No project files were modified by this verification.")


In [ ]:

import torch
import numpy as np
from PIL import Image

assert torch.cuda.is_available()
assert "siglip_model" in globals()
assert "siglip_processor" in globals()
assert "images" in globals()

siglip_model.eval()

inputs = siglip_processor(
    images=images,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    outputs = siglip_model.get_image_features(
        pixel_values=inputs["pixel_values"]
    )

    # Handle Transformers versions returning either
    # a tensor or an output object.
    if isinstance(outputs, torch.Tensor):
        features = outputs
    elif hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        features = outputs.pooler_output
    elif hasattr(outputs, "last_hidden_state"):
        features = outputs.last_hidden_state[:, 0, :]
    else:
        raise TypeError(
            f"Unexpected output type: {type(outputs)}"
        )

    features = torch.nn.functional.normalize(
        features.float(), p=2, dim=-1
    )

features = features.cpu().numpy().astype(np.float32)

print("Embedding shape:", features.shape)
print("All finite:", np.isfinite(features).all())
print("Embedding dimensions:", features.shape[1])
print("L2 norms:", np.linalg.norm(features, axis=1))

assert features.shape == (4, 1152)
assert np.isfinite(features).all()
assert np.allclose(
    np.linalg.norm(features, axis=1), 1.0, atol=1e-4
)

print("\nSigLIP 2 pilot passed.")


In [ ]:
import os
import pandas as pd
from PIL import Image
import torch

ROOT = "/content/drive/MyDrive/twincheck"
JPEG_DIR = os.path.join(ROOT, "data/copies/vision/jpeg85_final")
MANIFEST = os.path.join(JPEG_DIR, "jpeg85_final_manifest.csv")

df = pd.read_csv(MANIFEST)

assert len(df) == 1200
assert set(df["role"].unique()) == {"original", "twin"}
assert df["output_path"].notna().all()
assert df["valid"].astype(bool).all()

# Check all final JPEG output paths
paths = df["output_path"].astype(str).tolist()
missing = [p for p in paths if not os.path.isfile(p)]

print("Total manifest rows:", len(df))
print("Originals:", (df["role"] == "original").sum())
print("Twins:", (df["role"] == "twin").sum())
print("Missing files:", len(missing))

if missing:
    print("First missing paths:", missing[:5])

# Check a sample of actual JPEG files and their dimensions
for _, row in df.head(6).iterrows():
    with Image.open(row["output_path"]) as im:
        print(
            row["dataset"],
            row["image_id"],
            row["role"],
            im.format,
            im.size
        )

assert not missing, "Some JPEG files are missing."
assert torch.cuda.is_available()
assert "siglip_model" in globals()
assert "siglip_processor" in globals()

print("\nGPU:", torch.cuda.get_device_name(0))
print("SigLIP device:", next(siglip_model.parameters()).device)
print("SigLIP training mode:", siglip_model.training)
print("\nValidation passed. Ready for full extraction.")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Paths and configuration
# --------------------------------------------------
ROOT = "/content/drive/MyDrive/twincheck"
JPEG_DIR = os.path.join(ROOT, "data/copies/vision/jpeg85_final")
INPUT_MANIFEST = os.path.join(JPEG_DIR, "jpeg85_final_manifest.csv")

OUTPUT_DIR = os.path.join(
    ROOT, "features/vision/jpeg85_final/siglip2"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMBEDDING_PATH = os.path.join(
    OUTPUT_DIR, "jpeg85_siglip2_embeddings.npy"
)
CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR, "jpeg85_siglip2_checkpoint.json"
)
OUTPUT_MANIFEST = os.path.join(
    OUTPUT_DIR, "jpeg85_siglip2_embedding_manifest.csv"
)

BATCH_SIZE = 16
EXPECTED_DIM = 1152

# --------------------------------------------------
# 2. Validate input and model
# --------------------------------------------------
assert torch.cuda.is_available(), "CUDA GPU is not available."
assert "siglip_model" in globals()
assert "siglip_processor" in globals()

df = pd.read_csv(INPUT_MANIFEST)

assert len(df) == 1200
assert df["output_path"].notna().all()
assert all(os.path.isfile(p) for p in df["output_path"])

siglip_model.eval()
assert next(siglip_model.parameters()).device.type == "cuda"

print("Images to process:", len(df))
print("Batch size:", BATCH_SIZE)
print("Embedding dimension:", EXPECTED_DIM)
print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# 3. Create or resume embedding storage
# --------------------------------------------------
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "r") as f:
        checkpoint = json.load(f)

    assert checkpoint["total_images"] == len(df)
    assert checkpoint["embedding_dim"] == EXPECTED_DIM
    assert os.path.exists(EMBEDDING_PATH)

    start_index = int(checkpoint["completed_images"])
    embeddings = np.load(EMBEDDING_PATH, mmap_mode="r+")

    assert embeddings.shape == (len(df), EXPECTED_DIM)
    print("Resuming from image:", start_index)

else:
    assert not os.path.exists(EMBEDDING_PATH), (
        "An embedding file already exists without a checkpoint. "
        "Please inspect it before starting to avoid overwriting."
    )

    embeddings = np.lib.format.open_memmap(
        EMBEDDING_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(len(df), EXPECTED_DIM)
    )
    start_index = 0
    print("Starting a new extraction.")

# --------------------------------------------------
# 4. Extract embeddings in batches
# --------------------------------------------------
for batch_start in tqdm(
    range(start_index, len(df), BATCH_SIZE),
    desc="SigLIP 2 extraction"
):
    batch_end = min(batch_start + BATCH_SIZE, len(df))
    batch_paths = df["output_path"].iloc[batch_start:batch_end].tolist()

    batch_images = []
    for path in batch_paths:
        with Image.open(path) as im:
            batch_images.append(im.convert("RGB"))

    inputs = siglip_processor(
        images=batch_images,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        outputs = siglip_model.get_image_features(
            pixel_values=inputs["pixel_values"]
        )

        # Use the projected pooled image features.
        if isinstance(outputs, torch.Tensor):
            features = outputs
        elif (
            hasattr(outputs, "pooler_output")
            and outputs.pooler_output is not None
        ):
            features = outputs.pooler_output
        else:
            raise TypeError(
                f"Unexpected SigLIP output: {type(outputs)}. "
                "Expected a tensor or an output with pooler_output."
            )

        features = torch.nn.functional.normalize(
            features.float(), p=2, dim=-1
        )

    batch_embeddings = features.cpu().numpy()

    assert batch_embeddings.shape == (
        len(batch_paths), EXPECTED_DIM
    ), f"Unexpected batch shape: {batch_embeddings.shape}"

    assert np.isfinite(batch_embeddings).all()

    embeddings[batch_start:batch_end] = batch_embeddings
    embeddings.flush()

    # Save progress only after the batch is safely written.
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump({
            "total_images": len(df),
            "completed_images": batch_end,
            "embedding_dim": EXPECTED_DIM,
            "model": "google/siglip2-so400m-patch14-384",
            "batch_size": BATCH_SIZE
        }, f, indent=2)

    del inputs, outputs, features, batch_embeddings, batch_images
    torch.cuda.empty_cache()

# --------------------------------------------------
# 5. Final validation and manifest
# --------------------------------------------------
final_embeddings = np.load(EMBEDDING_PATH, mmap_mode="r")

assert final_embeddings.shape == (1200, 1152)
assert np.isfinite(final_embeddings).all()

norms = np.linalg.norm(final_embeddings, axis=1)

assert np.allclose(norms, 1.0, atol=1e-4), (
    "Some embeddings are not L2-normalized."
)

result_manifest = df.copy()
result_manifest.insert(
    0, "embedding_row", np.arange(len(result_manifest))
)
result_manifest["embedding_file"] = EMBEDDING_PATH
result_manifest.to_csv(OUTPUT_MANIFEST, index=False)

print("\n========== EXTRACTION COMPLETE ==========")
print("Embedding shape:", final_embeddings.shape)
print("All finite:", np.isfinite(final_embeddings).all())
print("Mean L2 norm:", float(norms.mean()))
print("Minimum L2 norm:", float(norms.min()))
print("Maximum L2 norm:", float(norms.max()))
print("Embedding file:", EMBEDDING_PATH)
print("Manifest file:", OUTPUT_MANIFEST)
print("File size (MB):", round(os.path.getsize(EMBEDDING_PATH) / 1e6, 2))
print("Completed images:", len(result_manifest))

In [ ]:
import sys
import subprocess
import importlib.util
import torch
import numpy as np
import pandas as pd
from PIL import Image

# 1. Check GPU
assert torch.cuda.is_available(), "CUDA GPU is not available."
print("GPU:", torch.cuda.get_device_name(0))

# 2. Install official OpenAI CLIP package if needed
if importlib.util.find_spec("clip") is None:
    print("Installing OpenAI CLIP...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/openai/CLIP.git"
    ])

import clip

# 3. Load CLIP ViT-L/14
clip_model, clip_preprocess = clip.load(
    "ViT-L/14",
    device="cuda",
    jit=False
)
clip_model.eval()

print("CLIP model loaded:", "ViT-L/14")

# 4. Load four final JPEG85 images
ROOT = "/content/drive/MyDrive/twincheck"
manifest_path = (
    ROOT + "/data/copies/vision/jpeg85_final/"
    "jpeg85_final_manifest.csv"
)

df = pd.read_csv(manifest_path)
sample_paths = df["output_path"].iloc[:4].tolist()

images = []
for path in sample_paths:
    with Image.open(path) as im:
        images.append(clip_preprocess(im.convert("RGB")))

image_tensor = torch.stack(images).to("cuda")

# 5. Extract projected CLIP image features
with torch.inference_mode():
    features = clip_model.encode_image(image_tensor)
    features = torch.nn.functional.normalize(
        features.float(), p=2, dim=-1
    )

features = features.cpu().numpy()

# 6. Validate
print("\nEmbedding shape:", features.shape)
print("All finite:", np.isfinite(features).all())
print("Embedding dimensions:", features.shape[1])
print("L2 norms:", np.linalg.norm(features, axis=1))

assert features.shape == (4, 768)
assert np.isfinite(features).all()
assert np.allclose(
    np.linalg.norm(features, axis=1), 1.0, atol=1e-5
)

print("\nCLIP ViT-L/14 pilot passed.")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import clip
from PIL import Image
from tqdm.auto import tqdm

# 1. Paths
ROOT = "/content/drive/MyDrive/twincheck"
JPEG_DIR = os.path.join(ROOT, "data/copies/vision/jpeg85_final")
INPUT_MANIFEST = os.path.join(
    JPEG_DIR, "jpeg85_final_manifest.csv"
)

OUTPUT_DIR = os.path.join(
    ROOT, "features/vision/jpeg85_final/clip_vit_l14"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMBEDDING_PATH = os.path.join(
    OUTPUT_DIR, "jpeg85_clip_vit_l14_embeddings.npy"
)
CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR, "jpeg85_clip_vit_l14_checkpoint.json"
)
OUTPUT_MANIFEST = os.path.join(
    OUTPUT_DIR, "jpeg85_clip_vit_l14_embedding_manifest.csv"
)

BATCH_SIZE = 16
EXPECTED_DIM = 768

# 2. Validate inputs
assert torch.cuda.is_available()
assert "clip_model" in globals()
assert "clip_preprocess" in globals()

df = pd.read_csv(INPUT_MANIFEST)
assert len(df) == 1200
assert df["output_path"].notna().all()
assert all(os.path.isfile(p) for p in df["output_path"])

clip_model.eval()

print("Images:", len(df))
print("Batch size:", BATCH_SIZE)
print("GPU:", torch.cuda.get_device_name(0))

# 3. Create or resume extraction
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "r") as f:
        checkpoint = json.load(f)

    assert checkpoint["total_images"] == len(df)
    assert checkpoint["embedding_dim"] == EXPECTED_DIM
    assert os.path.exists(EMBEDDING_PATH)

    start_index = int(checkpoint["completed_images"])
    embeddings = np.load(EMBEDDING_PATH, mmap_mode="r+")

    assert embeddings.shape == (1200, EXPECTED_DIM)
    print("Resuming from image:", start_index)

else:
    assert not os.path.exists(EMBEDDING_PATH), (
        "Embedding file exists without a checkpoint. "
        "Inspect before proceeding."
    )

    embeddings = np.lib.format.open_memmap(
        EMBEDDING_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(1200, EXPECTED_DIM)
    )
    start_index = 0
    print("Starting new extraction.")

# 4. Extract embeddings in batches
for batch_start in tqdm(
    range(start_index, len(df), BATCH_SIZE),
    desc="CLIP ViT-L/14 extraction"
):
    batch_end = min(batch_start + BATCH_SIZE, len(df))
    batch_paths = df["output_path"].iloc[
        batch_start:batch_end
    ].tolist()

    batch_images = []

    for path in batch_paths:
        with Image.open(path) as im:
            batch_images.append(
                clip_preprocess(im.convert("RGB"))
            )

    image_tensor = torch.stack(batch_images).to("cuda")

    with torch.inference_mode():
        features = clip_model.encode_image(image_tensor)
        features = torch.nn.functional.normalize(
            features.float(), p=2, dim=-1
        )

    batch_embeddings = features.cpu().numpy()

    assert batch_embeddings.shape == (
        len(batch_paths), EXPECTED_DIM
    )
    assert np.isfinite(batch_embeddings).all()

    embeddings[batch_start:batch_end] = batch_embeddings
    embeddings.flush()

    with open(CHECKPOINT_PATH, "w") as f:
        json.dump({
            "total_images": 1200,
            "completed_images": batch_end,
            "embedding_dim": EXPECTED_DIM,
            "model": "CLIP ViT-L/14",
            "batch_size": BATCH_SIZE
        }, f, indent=2)

    del image_tensor, features, batch_embeddings, batch_images
    torch.cuda.empty_cache()

# 5. Final validation
final_embeddings = np.load(EMBEDDING_PATH, mmap_mode="r")

assert final_embeddings.shape == (1200, 768)
assert np.isfinite(final_embeddings).all()

norms = np.linalg.norm(final_embeddings, axis=1)
assert np.allclose(norms, 1.0, atol=1e-4)

result_manifest = df.copy()
result_manifest.insert(
    0, "embedding_row", np.arange(len(result_manifest))
)
result_manifest["embedding_file"] = EMBEDDING_PATH
result_manifest.to_csv(OUTPUT_MANIFEST, index=False)

print("\n========== CLIP EXTRACTION COMPLETE ==========")
print("Embedding shape:", final_embeddings.shape)
print("All finite:", np.isfinite(final_embeddings).all())
print("Mean L2 norm:", float(norms.mean()))
print("Minimum L2 norm:", float(norms.min()))
print("Maximum L2 norm:", float(norms.max()))
print("Embedding file:", EMBEDDING_PATH)
print("Manifest:", OUTPUT_MANIFEST)
print("File size (MB):", round(
    os.path.getsize(EMBEDDING_PATH) / 1e6, 2
))
print("Completed images:", len(result_manifest))

In [ ]:
import os
import numpy as np

ROOT = "/content/drive/MyDrive/twincheck"
FEATURE_DIR = os.path.join(ROOT, "features/vision/genimage")

FILES = {
    "SigLIP2": os.path.join(
        FEATURE_DIR,
        "genimage_siglip2_image_projected_embeddings.npy"
    ),
    "CLIP": os.path.join(
        FEATURE_DIR,
        "genimage_clip_vit_l14_image_projected_embeddings.npy"
    )
}

print("Checking original GenImage embeddings...\n")

for name, path in FILES.items():
    assert os.path.isfile(path), f"Missing {name} file: {path}"

    arr = np.load(path, mmap_mode="r")

    print(
        name,
        "| Shape:", arr.shape,
        "| Dtype:", arr.dtype,
        "| Size (MB):", round(os.path.getsize(path) / 1e6, 2)
    )

    assert arr.shape[0] == 18000
    assert arr.dtype == np.float32

print("\nInspection passed. Original files will remain unchanged.")
print("Next, we will create separate float16 copies.")

In [ ]:
import os
import numpy as np

ROOT = "/content/drive/MyDrive/twincheck"
FEATURE_DIR = os.path.join(ROOT, "features/vision/genimage")

FILES = {
    "SigLIP2": (
        "genimage_siglip2_image_projected_embeddings.npy",
        "genimage_siglip2_image_projected_embeddings_float16.npy"
    ),
    "CLIP": (
        "genimage_clip_vit_l14_image_projected_embeddings.npy",
        "genimage_clip_vit_l14_image_projected_embeddings_float16.npy"
    )
}

for name, (source_name, output_name) in FILES.items():
    source_path = os.path.join(FEATURE_DIR, source_name)
    output_path = os.path.join(FEATURE_DIR, output_name)

    assert os.path.isfile(source_path), f"Missing: {source_path}"
    assert not os.path.exists(output_path), (
        f"Output already exists: {output_path}. "
        "Inspect it before overwriting."
    )

    original = np.load(source_path, mmap_mode="r")

    # Convert in batches to limit memory usage.
    converted = np.lib.format.open_memmap(
        output_path,
        mode="w+",
        dtype=np.float16,
        shape=original.shape
    )

    batch_size = 1000

    for start in range(0, len(original), batch_size):
        end = min(start + batch_size, len(original))
        converted[start:end] = original[start:end].astype(np.float16)

    converted.flush()
    del converted

    # Validate converted copy
    result = np.load(output_path, mmap_mode="r")

    assert result.shape == original.shape
    assert result.dtype == np.float16
    assert np.isfinite(result).all()

    # Float16 introduces small rounding differences.
    norms = np.linalg.norm(result.astype(np.float32), axis=1)

    print(f"\n{name}")
    print("Shape:", result.shape)
    print("Dtype:", result.dtype)
    print("All finite:", np.isfinite(result).all())
    print("Mean L2 norm:", float(norms.mean()))
    print("File size (MB):", round(os.path.getsize(output_path) / 1e6, 2))
    print("Saved:", output_path)

print("\nBoth float16 conversions completed.")

In [ ]:
import os
import json
import pandas as pd

ROOT = "/content/drive/MyDrive/twincheck"

targets = [
    "data/copies/vision",
    "features/vision/jpeg85_final",
]

for folder in targets:
    path = os.path.join(ROOT, folder)
    print(f"\n{'='*65}\n{folder}")
    if os.path.isdir(path):
        for name in sorted(os.listdir(path)):
            full = os.path.join(path, name)
            if os.path.isfile(full):
                print(f"FILE  {name} ({os.path.getsize(full)/1e6:.2f} MB)")
            else:
                print(f"DIR   {name}")
    else:
        print("Folder not found")

# Inspect existing PR deliverables, if present
print("\n\nEXISTING PR OUTPUTS")
for name in ["vae.jsonl", "sdxl.jsonl", "sources.csv"]:
    path = os.path.join(ROOT, "data/copies/vision", name)
    print(f"\n{name}:")
    if not os.path.isfile(path):
        print("Not present")
        continue

    if name.endswith(".csv"):
        sample = pd.read_csv(path)
        print("Shape:", sample.shape)
        print("Columns:", sample.columns.tolist())
        display(sample.head(3))
    else:
        with open(path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i == 3:
                    break
                print(line[:1500].rstrip())

print("\nInspection complete. No files modified.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/drive/MyDrive/twincheck"

# 1. Inspect project instruction and documentation files
print("=" * 70)
print("PROJECT DOCUMENTATION")
print("=" * 70)

for name in ["README.md", "CLAUDE.md"]:
    path = os.path.join(ROOT, name)
    if os.path.isfile(path):
        print(f"\n--- {name} ---")
        with open(path, "r", encoding="utf-8") as f:
            print(f.read()[:10000])

# 2. Inspect VAE and SDXL manifest files
folders = [
    "data/copies/vision/verite",
    "data/copies/vision/m4fc",
    "data/copies/vision"
]

print("\n" + "=" * 70)
print("AVAILABLE MANIFESTS")
print("=" * 70)

for folder in folders:
    path = os.path.join(ROOT, folder)
    if not os.path.isdir(path):
        continue

    print(f"\n--- {folder} ---")

    for name in sorted(os.listdir(path)):
        full = os.path.join(path, name)

        if os.path.isfile(full) and name.endswith(".csv"):
            print(f"\n{name}")

            try:
                data = pd.read_csv(full)
                print("Shape:", data.shape)
                print("Columns:", data.columns.tolist())
                display(data.head(2))
            except Exception as e:
                print("Could not inspect:", str(e))

# 3. Search project code for JSONL schema references
print("\n" + "=" * 70)
print("JSONL SCHEMA REFERENCES IN PROJECT CODE")
print("=" * 70)

terms = ["vae.jsonl", "sdxl.jsonl", "source_image_path"]
extensions = (".py", ".md", ".yaml", ".yml", ".json")

for current, dirs, files in os.walk(ROOT):
    dirs[:] = [
        d for d in dirs
        if d not in [".git", "__pycache__", ".ipynb_checkpoints"]
    ]

    for name in files:
        if not name.endswith(extensions):
            continue

        path = os.path.join(current, name)

        try:
            with open(path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            matches = [
                (i + 1, line.rstrip())
                for i, line in enumerate(lines)
                if any(term in line.lower() for term in terms)
            ]

            if matches:
                print(f"\n{os.path.relpath(path, ROOT)}")
                for line_no, line in matches[:15]:
                    print(f"  L{line_no}: {line[:250]}")

        except Exception:
            pass

print("\nInspection complete. No files modified.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/drive/MyDrive/twincheck"

files_to_check = [
    "features/vision/verite/image_twins/verite_face_filtered_vae_twin_manifest.csv",
    "features/vision/m4fc/image_twins/m4fc_face_filtered_vae_twin_manifest.csv",
    "data/copies/vision/sdxl_face_filtered_accepted_manifest.csv",
    "data/copies/vision/jpeg85_final/jpeg85_final_manifest.csv",
    "features/vision/F2_completion_audit.json",
    "features/vision/F2_completion_report.md",
]

for relative_path in files_to_check:
    path = os.path.join(ROOT, relative_path)

    print("\n" + "=" * 70)
    print(relative_path)

    if not os.path.isfile(path):
        print("NOT FOUND")
        continue

    if path.endswith(".csv"):
        data = pd.read_csv(path)
        print("Rows:", len(data))
        print("Columns:", data.columns.tolist())
        display(data.head(3))

    elif path.endswith(".json"):
        import json
        with open(path, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2)[:6000])

    else:
        with open(path, "r", encoding="utf-8") as f:
            print(f.read()[:6000])

print("\nInspection complete. No files modified.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/drive/MyDrive/twincheck"

path = os.path.join(
    ROOT,
    "data/copies/vision/jpeg85_final/jpeg85_final_manifest.csv"
)

df = pd.read_csv(path)

originals = df[df["role"] == "original"]
twins = df[df["role"] == "twin"]

pairs = originals.merge(
    twins,
    on=["dataset", "image_id"],
    suffixes=("_original", "_twin"),
    validate="one_to_one"
)

pairs["final_id"] = (
    pairs["image_id"].str.replace(r"\.[^.]+$", "", regex=True)
    + "_V"
)

# Check ID uniqueness separately by generation method
print("ID uniqueness by dataset:")
for dataset, group in pairs.groupby("dataset"):
    print(
        dataset,
        "| pairs:", len(group),
        "| unique IDs:", group["final_id"].nunique(),
        "| duplicates:", group["final_id"].duplicated().sum()
    )

# Compare M4FC VAE accepted manifest with JPEG85 retained pairs
m4fc_path = os.path.join(
    ROOT,
    "features/vision/m4fc/image_twins/"
    "m4fc_face_filtered_vae_twin_manifest.csv"
)

m4fc = pd.read_csv(m4fc_path)

jpeg_m4fc = set(
    pairs.loc[pairs["dataset"] == "M4FC_VAE", "image_id"]
)

manifest_m4fc = set(m4fc["image_id"])

print("\nM4FC comparison:")
print("Original accepted manifest:", len(manifest_m4fc))
print("JPEG85 retained:", len(jpeg_m4fc))
print("Excluded during JPEG85/face filtering:",
      sorted(manifest_m4fc - jpeg_m4fc))

# Check overlap between VAE and SDXL IDs
vae_ids = set(
    pairs.loc[
        pairs["dataset"].isin(["VERITE_VAE", "M4FC_VAE"]),
        "final_id"
    ]
)

sdxl_ids = set(
    pairs.loc[pairs["dataset"] == "SDXL", "final_id"]
)

print("\nVAE-SDXL shared source IDs:", len(vae_ids & sdxl_ids))
print("Examples:", sorted(vae_ids & sdxl_ids)[:10])

print("\nNo files modified.")

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")
VISION = ROOT / "data/copies/vision"

# Final JPEG85 original/twin pairs
jpeg_path = VISION / "jpeg85_final/jpeg85_final_manifest.csv"
jpeg = pd.read_csv(jpeg_path)

originals = jpeg[jpeg["role"] == "original"][
    ["dataset", "image_id", "output_path"]
].rename(columns={"output_path": "source_image_path"})

twins = jpeg[jpeg["role"] == "twin"][
    ["dataset", "image_id", "output_path"]
].rename(columns={"output_path": "twin_image_path"})

pairs = originals.merge(
    twins,
    on=["dataset", "image_id"],
    validate="one_to_one"
)

def stem(value):
    return Path(str(value)).stem

pairs["join_id"] = pairs["image_id"].apply(stem)
pairs["id"] = pairs["join_id"] + "_V"

def relative_path(path):
    return Path(path).resolve().relative_to(ROOT.resolve()).as_posix()

# -------------------------
# Build VAE records
# -------------------------

verite = pd.read_csv(
    ROOT / "features/vision/verite/image_twins/"
    "verite_face_filtered_vae_twin_manifest.csv"
)
m4fc = pd.read_csv(
    ROOT / "features/vision/m4fc/image_twins/"
    "m4fc_face_filtered_vae_twin_manifest.csv"
)

verite["join_id"] = verite["image_id"].apply(stem)
m4fc["join_id"] = m4fc["image_id"].apply(stem)

verite_info = verite[["join_id", "split", "siglip2_similarity"]].copy()
verite_info["dataset"] = "VERITE"
verite_info["similarity"] = verite_info["siglip2_similarity"]
verite_info = verite_info.drop(columns=["siglip2_similarity"])

m4fc_info = m4fc[["join_id", "split", "siglip2_cosine_similarity"]].copy()
m4fc_info["dataset"] = "M4FC"
m4fc_info["similarity"] = m4fc_info["siglip2_cosine_similarity"]
m4fc_info = m4fc_info.drop(columns=["siglip2_cosine_similarity"])

vae_info = pd.concat([verite_info, m4fc_info], ignore_index=True)

vae_pairs = pairs[
    pairs["dataset"].isin(["VERITE_VAE", "M4FC_VAE"])
].copy()

vae_pairs["dataset"] = vae_pairs["dataset"].map({
    "VERITE_VAE": "VERITE",
    "M4FC_VAE": "M4FC"
})

vae_pairs = vae_pairs.merge(
    vae_info,
    on=["join_id", "dataset"],
    how="left",
    validate="one_to_one"
)

assert vae_pairs["split"].notna().all(), "Missing VAE split metadata"
assert vae_pairs["similarity"].notna().all(), "Missing VAE similarity metadata"

vae_pairs["method"] = "VAE"

# -------------------------
# Build SDXL records
# -------------------------

sdxl = pd.read_csv(
    VISION / "sdxl_face_filtered_accepted_manifest.csv"
)

sdxl["join_id"] = sdxl["id"].apply(stem)

sdxl_pairs = pairs[pairs["dataset"] == "SDXL"].copy()

sdxl_pairs = sdxl_pairs.merge(
    sdxl[["join_id", "split", "similarity"]],
    on="join_id",
    how="left",
    validate="one_to_one"
)

assert sdxl_pairs["split"].notna().all(), "Missing SDXL split metadata"
assert sdxl_pairs["similarity"].notna().all(), "Missing SDXL similarity metadata"

sdxl_pairs["dataset"] = sdxl_pairs["split"].map(
    lambda _: None
) if False else sdxl_pairs["dataset"]

sdxl_pairs["method"] = "SDXL"

# -------------------------
# Prepare JSONL records
# -------------------------

def make_records(frame):
    records = []
    for _, row in frame.iterrows():
        records.append({
            "id": row["id"],
            "dataset": row["dataset"],
            "split": row["split"],
            "method": row["method"],
            "source_image_path": relative_path(row["source_image_path"]),
            "twin_image_path": relative_path(row["twin_image_path"]),
            "similarity": float(row["similarity"])
        })
    return records

vae_records = make_records(vae_pairs)
sdxl_records = make_records(sdxl_pairs)

# -------------------------
# Validate before writing
# -------------------------

assert len(vae_records) == 452, f"Expected 452 VAE pairs, got {len(vae_records)}"
assert len(sdxl_records) == 148, f"Expected 148 SDXL pairs, got {len(sdxl_records)}"

assert len({r["id"] for r in vae_records}) == len(vae_records), "Duplicate VAE IDs"
assert len({r["id"] for r in sdxl_records}) == len(sdxl_records), "Duplicate SDXL IDs"

for record in vae_records + sdxl_records:
    assert (ROOT / record["source_image_path"]).is_file(), record["source_image_path"]
    assert (ROOT / record["twin_image_path"]).is_file(), record["twin_image_path"]

assert not any(
    r["dataset"] == "VERITE" and r["split"] == "train"
    for r in sdxl_records
), "SDXL unexpectedly contains VERITE training sources"

# -------------------------
# Write JSONL files
# -------------------------

vae_output = VISION / "vae.jsonl"
sdxl_output = VISION / "sdxl.jsonl"

for output_path, records in [
    (vae_output, vae_records),
    (sdxl_output, sdxl_records)
]:
    with open(output_path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Read back and verify
for output_path, expected_count in [
    (vae_output, 452),
    (sdxl_output, 148)
]:
    with open(output_path, "r", encoding="utf-8") as f:
        loaded = [json.loads(line) for line in f if line.strip()]
    assert len(loaded) == expected_count
    print(f"{output_path.name}: {len(loaded)} records | verified")

print("\nVAE breakdown:")
print(vae_pairs.groupby("dataset").size().to_string())

print("\nSDXL breakdown:")
print(sdxl_pairs.groupby(["dataset", "split"]).size().to_string())

print("\nSDXL VERITE training records:",
      sum(r["dataset"] == "VERITE" and r["split"] == "train"
          for r in sdxl_records))

print("\nJSONL files created successfully.")
print("No image files were modified or copied by this cell.")

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")
VISION = ROOT / "data/copies/vision"

files = {
    "current_sources.csv": VISION / "sources.csv",
    "approved_sdxl_sources.csv": VISION / "sdxl_sources.csv",
    "face_excluded_sources.csv": VISION / "sdxl_face_excluded_sources.csv",
}

tables = {}

for name, path in files.items():
    if not path.exists():
        print(f"\n{name}: FILE NOT FOUND")
        continue

    df = pd.read_csv(path)
    tables[name] = df

    print(f"\n--- {name} ---")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    for col in ["dataset", "split", "label"]:
        if col in df.columns:
            print(f"\n{col} counts:")
            print(df[col].value_counts(dropna=False).to_string())

# Compare the current and approved source lists
if (
    "current_sources.csv" in tables
    and "approved_sdxl_sources.csv" in tables
):
    current = tables["current_sources.csv"]
    approved = tables["approved_sdxl_sources.csv"]

    print("\n--- SOURCE LIST COMPARISON ---")
    print("Current source rows:", len(current))
    print("Approved source rows:", len(approved))

    if "id" in current.columns and "id" in approved.columns:
        current_ids = set(current["id"].astype(str))
        approved_ids = set(approved["id"].astype(str))

        print("Current IDs not in approved list:",
              sorted(current_ids - approved_ids))
        print("Approved IDs missing from current list:",
              sorted(approved_ids - current_ids))

    if "id" in approved.columns:
        print("Approved IDs unique:",
              approved["id"].is_unique)

print("\nInspection only: no files modified.")

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")
VISION = ROOT / "data/copies/vision"

current_path = VISION / "sources.csv"
approved_path = VISION / "sdxl_sources.csv"
backup_path = VISION / "sources_pre_face_exclusion_backup.csv"

required_columns = [
    "id", "dataset", "split", "label", "source_image_path"
]

current = pd.read_csv(current_path)
approved = pd.read_csv(approved_path)

# Validate the approved source list before changing anything
assert len(current) == 257
assert len(approved) == 249
assert approved["id"].is_unique
assert set(required_columns).issubset(approved.columns)

assert not (
    (approved["dataset"] == "VERITE") &
    (approved["split"] == "train")
).any(), "Unexpected VERITE training source"

# Confirm that exactly the eight excluded IDs were removed
removed_ids = set(current["id"].astype(str)) - set(approved["id"].astype(str))
assert len(removed_ids) == 8
assert set(approved["id"].astype(str)).issubset(
    set(current["id"].astype(str))
)

# Preserve the original file, then write the approved list
if not backup_path.exists():
    shutil.copy2(current_path, backup_path)

final_sources = approved[required_columns].copy()
final_sources.to_csv(current_path, index=False)

# Read back and verify
check = pd.read_csv(current_path)

assert len(check) == 249
assert check.columns.tolist() == required_columns
assert check["id"].is_unique
assert set(check["id"].astype(str)) == set(approved["id"].astype(str))

print("sources.csv successfully updated.")
print("Final rows:", len(check))
print("Columns:", check.columns.tolist())
print("\nDataset counts:")
print(check["dataset"].value_counts().to_string())
print("\nSplit counts:")
print(check["split"].value_counts().to_string())
print("\nExcluded source IDs:", sorted(removed_ids))
print("Backup preserved:", backup_path.exists())
print("No image files modified.")

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")

files = [
    ROOT / "features/vision/F2_completion_audit.json",
    ROOT / "features/vision/F2_completion_report.md",
]

for path in files:
    print("\n" + "=" * 80)
    print(path.relative_to(ROOT))
    print("=" * 80)

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("FILE NOT FOUND")

print("\nInspection only: no files modified.")

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/content/drive/MyDrive/twincheck")
FEATURES = ROOT / "features/vision"
VISION = ROOT / "data/copies/vision"

audit_path = FEATURES / "F2_completion_audit.json"
report_path = FEATURES / "F2_completion_report.md"

# Final file locations
jpeg_manifest = VISION / "jpeg85_final/jpeg85_final_manifest.csv"
jpeg_excluded = VISION / "jpeg85_final/jpeg85_excluded_manifest.csv"

siglip_embeddings = FEATURES / "jpeg85_final/siglip2/jpeg85_siglip2_embeddings.npy"
siglip_manifest = FEATURES / "jpeg85_final/siglip2/jpeg85_siglip2_embedding_manifest.csv"

clip_embeddings = FEATURES / "jpeg85_final/clip_vit_l14/jpeg85_clip_vit_l14_embeddings.npy"
clip_manifest = FEATURES / "jpeg85_final/clip_vit_l14/jpeg85_clip_vit_l14_embedding_manifest.csv"

vae_jsonl = VISION / "vae.jsonl"
sdxl_jsonl = VISION / "sdxl.jsonl"
sources_csv = VISION / "sources.csv"

# Validate required final deliverables exist
required_files = [
    jpeg_manifest, jpeg_excluded,
    siglip_embeddings, siglip_manifest,
    clip_embeddings, clip_manifest,
    vae_jsonl, sdxl_jsonl, sources_csv
]

missing = [str(p) for p in required_files if not p.is_file()]
assert not missing, "Missing required files:\n" + "\n".join(missing)

# Validate final manifest counts
import pandas as pd

jpeg = pd.read_csv(jpeg_manifest)
assert len(jpeg) == 1200
assert jpeg["role"].value_counts().to_dict() == {
    "original": 600, "twin": 600
}

pair_counts = (
    jpeg.groupby(["dataset", "role"])
    .size()
    .unstack(fill_value=0)
)

expected_pairs = {
    "VERITE_VAE": 311,
    "M4FC_VAE": 141,
    "SDXL": 148
}

for dataset, expected in expected_pairs.items():
    assert pair_counts.loc[dataset, "original"] == expected
    assert pair_counts.loc[dataset, "twin"] == expected

assert len(pd.read_csv(jpeg_excluded)) == 2
assert len(pd.read_csv(sources_csv)) == 249

def count_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

assert count_jsonl(vae_jsonl) == 452
assert count_jsonl(sdxl_jsonl) == 148

# Verify embedding array dimensions
import numpy as np

siglip = np.load(siglip_embeddings, mmap_mode="r")
clip = np.load(clip_embeddings, mmap_mode="r")

assert siglip.shape == (1200, 1152)
assert clip.shape == (1200, 768)

# Build audit
audit = {
    "task": "F2 - Vision features and image twins",
    "last_updated_utc": datetime.now(timezone.utc).isoformat(),
    "embedding_audit": {
        "VERITE": {
            "siglip2_images": 663,
            "siglip2_captions": 1001,
            "clip_vit_l14_images": 663
        },
        "GenImage": {
            "siglip2_images": 18000,
            "clip_vit_l14_images": 18000,
            "float16_siglip2_file": "features/vision/genimage/genimage_siglip2_image_projected_embeddings_float16.npy",
            "float16_clip_file": "features/vision/genimage/genimage_clip_vit_l14_image_projected_embeddings_float16.npy",
            "manifest_split_column": "GenImage source folder (train / validation), not the experiment split",
            "experiment_splits": {
                "source": "data/processed/vision/genimage_{train,validation,test}_manifest.csv (F1)",
                "join_key": "image_id = md5",
                "train": "4,000 real + 5,000 BigGAN",
                "validation": "1,000 real + 1,000 BigGAN",
                "test": "1,000 real + 1,000 Stable Diffusion v1.4 (unseen generator)",
                "images_with_embeddings": 13000
            }
        },
        "M4FC": {
            "siglip2_images": 603,
            "clip_vit_l14_images": 603,
            "siglip2_arabic_claim_embeddings": 770
        },
        "jpeg85_final": {
            "image_count": 1200,
            "siglip2_shape": [
                1200,
                1152
            ],
            "clip_vit_l14_shape": [
                1200,
                768
            ],
            "siglip2_embeddings": "features/vision/jpeg85_final/siglip2/jpeg85_siglip2_embeddings.npy",
            "clip_embeddings": "features/vision/jpeg85_final/clip_vit_l14/jpeg85_clip_vit_l14_embeddings.npy",
            "all_embeddings_finite_and_normalized": True
        },
        "all_checked_image_embeddings_passed": True
    },
    "vae_twins": {
        "VERITE": {
            "generated": 338,
            "accepted_before_final_face_screen": 311,
            "final_retained_pairs": 311
        },
        "M4FC": {
            "generated": 156,
            "accepted_before_final_face_screen": 142,
            "final_retained_pairs": 141,
            "additional_face_exclusion": "m4fc_000478"
        },
        "total_generated": 494,
        "final_retained_pairs": 452,
        "jsonl": "data/copies/vision/vae.jsonl",
        "jsonl_records": 903,
        "twin_images": 452
    },
    "sdxl_twins": {
        "strength": 0.15,
        "seed": 42,
        "sources_approved": 249,
        "generated": 249,
        "similarity_accepted_before_manual_face_review": 157,
        "similarity_rejected": 92,
        "manually_excluded_for_faces": 9,
        "final_retained_pairs": 148,
        "visual_pairs_manually_inspected": 30,
        "final_manifest": "data/copies/vision/sdxl_face_filtered_accepted_manifest.csv",
        "jsonl": "data/copies/vision/sdxl.jsonl",
        "jsonl_records": 295,
        "sources_csv": "data/copies/vision/sources.csv",
        "sources_csv_records": 249,
        "verite_training_sources_included": 0,
        "twin_images": 148
    },
    "jpeg85_final": {
        "conversion": "Originals and twins re-saved from decoded RGB pixels as JPEG quality 85",
        "pairs_before_additional_automatic_face_screen": 601,
        "pairs_excluded_after_screen": 1,
        "excluded_pair_id": "m4fc_000478",
        "final_retained_pairs": 600,
        "final_files": 1200,
        "dataset_pair_counts": {
            "VERITE_VAE": 311,
            "M4FC_VAE": 141,
            "SDXL": 148
        },
        "excluded_manifest": "data/copies/vision/jpeg85_final/jpeg85_excluded_manifest.csv",
        "final_manifest": "data/copies/vision/jpeg85_final/jpeg85_final_manifest.csv",
        "quality_85_verified_by_conversion_process": True
    },
    "face_screening": {
        "method": "OpenCV Haar frontal-face cascade",
        "twins_screened": 601,
        "detections_over_10_percent": 4,
        "confirmed_face_exclusions": 1,
        "confirmed_exclusion_id": "m4fc_000478",
        "false_positive_detections": 3,
        "manual_review_scope": "The four automatic detections were visually reviewed; SDXL also had a separate manual review of 30 pairs and nine face exclusions.",
        "limitation": "Haar frontal-face detection is not a comprehensive detector for all faces or poses."
    },
    "pr_deliverables": {
        "vae_jsonl_records": 903,
        "sdxl_jsonl_records": 295,
        "sources_csv_records": 249,
        "actual_twin_image_files_to_commit": 0
    },
    "limitations_and_pending_items": [
        "M4FC text embeddings represent Arabic claims, not image captions.",
        "SDXL manual visual inspection covered 30 pairs, not every generated pair.",
        "The automatic Haar face detector is limited and does not establish exhaustive face removal."
    ],
    "status": "COMPLETED: the core F2 image embeddings, image-twin generation, JPEG85 originals/twins, final embeddings, manifests, and shared-format records are generated and validated. Quality-review limitations remain documented.",
    "final_twin_count_summary": {
        "VERITE_VAE": 311,
        "M4FC_VAE": 141,
        "SDXL": 148,
        "total_retained_pairs_across_methods": 600,
        "note": "The total counts outputs across methods; the same source may have both a VAE twin and an SDXL twin."
    }
}

# Build report
report = """# F2 — Vision Features and Image Twins

## Status

**COMPLETED (core F2 pipeline).** The final JPEG-quality-85 image-twin pairs,
their SigLIP 2 and CLIP ViT-L/14 embeddings, and the requested PR metadata
files have been generated and validated. Quality-review limitations are
documented below.

## 1. Image embeddings

| Dataset | SigLIP 2 image embeddings | SigLIP 2 text embeddings | CLIP ViT-L/14 image embeddings |
|---|---:|---:|---:|
| VERITE | 663 | 1,001 captions | 663 |
| GenImage | 18,000 | Not generated | 18,000 |
| M4FC | 603 | 770 Arabic claim embeddings | 603 |

All checked image embeddings passed the recorded embedding audit.

The GenImage float16 embedding copies are available:
- `features/vision/genimage/genimage_siglip2_image_projected_embeddings_float16.npy`
- `features/vision/genimage/genimage_clip_vit_l14_image_projected_embeddings_float16.npy`

Only the float16 copies are in git. The float32 originals (55 MB and 83 MB)
stay on Drive.

**Row manifests (which row is which image or claim).** Every `.npy` has a
`*_manifest.csv` next to it with the same row order. Paths are repo-relative.

| Embedding file | Manifest key | Notes |
|---|---|---|
| VERITE SigLIP 2 / CLIP images (663) | `image_id` (e.g. `true_0.jpg`) | Same order in both; the split matches `verite.jsonl` for all 663 |
| VERITE SigLIP 2 captions (1,001) | `caption_id` | Row *i* = record `verite_{i:06d}` in `verite.jsonl` (checked) |
| M4FC SigLIP 2 / CLIP images (603) | `id` (image id) | `split` here is M4FC's own split; our M4FC records are all `test` |
| M4FC SigLIP 2 Arabic claims (770) | `id`, `embedding_row` | One extra row, `m4fc_000027`, is no longer in `m4fc_ar_test.jsonl` (769). Join by `id`. |
| GenImage (18,000) | `image_id` (= `md5` in the F1 manifests) | 5,000 / 1,000 per class (real, SD v1.4, BigGAN) from the GenImage train / validation folders |

**GenImage experiment splits.** The `split` column in the GenImage embedding
manifest is the GenImage source folder, not the experiment split. The
experiment splits come from F1 (`data/processed/vision/genimage_{train,validation,test}_manifest.csv`):

| Split | Real | AI-generated |
|---|---:|---|
| Train | 4,000 | 5,000 BigGAN |
| Validation | 1,000 | 1,000 BigGAN |
| Test (unseen generator) | 1,000 | 1,000 Stable Diffusion v1.4 |

All 13,000 images in these splits have embeddings; join on `image_id` = `md5`.
The 5,000 Stable Diffusion v1.4 images from the GenImage train folder are
embedded but not used in the experiment.

The final JPEG85 image embeddings cover 1,200 images (600 original/twin
pairs):
- SigLIP 2: shape `(1200, 1152)`.
- CLIP ViT-L/14: shape `(1200, 768)`.

Both embedding sets were checked for finite, normalized vectors.

**Important distinction:** M4FC's 770 text embeddings represent Arabic claims,
not image captions.

## 2. VAE image twins

The VAE encode-decode pipeline generated 494 twins.

| Dataset | Generated | Accepted before final face screen | Final retained |
|---|---:|---:|---:|
| VERITE | 338 | 311 | 311 |
| M4FC | 156 | 142 | 141 |
| **Total** | **494** | **453** | **452** |

The final M4FC count excludes `m4fc_000478`, which was identified as
containing a large face during the additional automatic-screening and
manual-review process.

## 3. SDXL image twins

The SDXL img2img run used strength 0.15 and seed 42.

| Stage | Count |
|---|---:|
| Approved sources processed | 249 |
| Similarity accepted before manual face review | 157 |
| Similarity rejected | 92 |
| Manually excluded for faces | 9 |
| **Final retained** | **148** |

The approved source set contains 152 M4FC and 97 VERITE images:
201 test and 48 validation sources. No VERITE training sources were used.

Manual visual inspection covered 30 pairs; it was not an exhaustive review
of every generated pair.

## 4. JPEG quality 85 conversion and final face screening

The original and twin images were re-saved from decoded RGB pixels as JPEG
quality 85. The conversion process recorded the quality setting and
produced the final manifest.

| Dataset/method | Pairs retained |
|---|---:|
| VERITE VAE | 311 |
| M4FC VAE | 141 |
| SDXL | 148 |
| **Total** | **600** |

The final manifest contains 1,200 files: 600 originals and 600 twins.
One additional pair, `m4fc_000478`, was excluded after face screening.

An OpenCV Haar frontal-face cascade screened all 601 twins before that
exclusion. Four detections exceeded the 10% image-area threshold. Manual
inspection identified one genuine large face and three false positives.
The Haar cascade is not a comprehensive detector for all face sizes,
orientations, or poses.

The final JPEG85 embeddings were extracted after conversion and screening.

## 5. PR metadata deliverables

The twin files are in the shared record format (CLAUDE.md §4). Each line is
the source record from `data/processed/vision/verite.jsonl` or
`data/processed/ar/m4fc_ar_test.jsonl` (same claim, label, evidence, language
and split) with `id = <source id>_V` and `image_path` = the twin image.
`method`, `similarity` and `orig_image_path` are extra fields. A twin image
gets one record per claim that uses the original photo. For example, a
VERITE true photo has a true caption and a miscaptioned one.

| File | Twin images | Records |
|---|---:|---:|
| `data/copies/vision/vae.jsonl` | 452 | 903 (VERITE 622, M4FC 281) |
| `data/copies/vision/sdxl.jsonl` | 148 | 295 (VERITE 106, M4FC 189) |
| `data/copies/vision/sources.csv` | 249 approved SDXL sources (148 with a kept twin) | — |

All three pass `python -m src.eval.loaders --check`. Built by
`python -m src.vision.make_twin_records` from the F2 notebook manifests in
`data/copies/vision/manifests/`. `data/processed/vision/verite.jsonl` is built by
`python -m src.vision.load_verite` (ids `verite_000000`…, labels true →
supported, miscaptioned / out-of-context → contradicted, F1 split by image:
701 / 149 / 151 records). Paths are repo-relative. The image files themselves
are not in git (the repo is public and these are news photos).

IDs can occur in both JSONL files when the same source has both a VAE twin
and an SDXL twin; uniqueness is checked within each file.

## 6. Quality-review limitations

1. SDXL manual visual inspection covered 30 pairs, not all generated pairs.
2. Haar frontal-face detection is limited and does not establish exhaustive
   face removal.
3. M4FC's text embeddings represent Arabic claims, not image captions.

## 7. Final assessment

The final JPEG85 twin set contains 600 retained pairs across the two
generation methods. The corresponding 1,200 images have SigLIP 2 and
CLIP ViT-L/14 embeddings. The JSONL and CSV PR metadata files have been
created and validated.

The core F2 pipeline is **completed**. The remaining notes are quality-review
limitations and do not represent missing core F2 deliverables.
"""

# Write files
with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

report_path.write_text(report, encoding="utf-8")

# Read back and validate
with open(audit_path, encoding="utf-8") as f:
    audit_check = json.load(f)

report_check = report_path.read_text(encoding="utf-8")

assert audit_check["final_twin_count_summary"]["total_retained_pairs_across_methods"] == 600
assert audit_check["jpeg85_final"]["quality_85_verified_by_conversion_process"] is True
assert "COMPLETED" in report_check
assert "600" in report_check
assert "JPEG quality 85" in report_check

print("F2 audit and report written.")
print("Audit status:", audit_check["status"])
print("Final retained pairs:", audit_check["final_twin_count_summary"]["total_retained_pairs_across_methods"])
print("Final JPEG85 files:", audit_check["jpeg85_final"]["final_files"])
print("VAE JSONL records:", count_jsonl(vae_jsonl))
print("SDXL JSONL records:", count_jsonl(sdxl_jsonl))
print("Sources CSV records:", len(pd.read_csv(sources_csv)))
print("No image files modified.")

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

ROOT = Path("/content/drive/MyDrive/twincheck")

# The 12 approved files and their relative folder structure
file_list = [
    "data/copies/vision/vae.jsonl",
    "data/copies/vision/sdxl.jsonl",
    "data/copies/vision/sources.csv",

    "features/vision/jpeg85_final/siglip2/jpeg85_siglip2_embeddings.npy",
    "features/vision/jpeg85_final/siglip2/jpeg85_siglip2_embedding_manifest.csv",

    "features/vision/jpeg85_final/clip_vit_l14/jpeg85_clip_vit_l14_embeddings.npy",
    "features/vision/jpeg85_final/clip_vit_l14/jpeg85_clip_vit_l14_embedding_manifest.csv",

    "features/vision/genimage/genimage_siglip2_image_projected_embeddings_float16.npy",
    "features/vision/genimage/genimage_clip_vit_l14_image_projected_embeddings_float16.npy",
    "features/vision/genimage/genimage_image_embedding_manifest.csv",

    "features/vision/F2_completion_audit.json",
    "features/vision/F2_completion_report.md"
]

# Verify all files exist
missing = [f for f in file_list if not (ROOT / f).is_file()]
assert not missing, "Missing files:\n" + "\n".join(missing)

# Create ZIP
zip_path = "/content/F2_GitHub_Upload_Files.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for relative_path in file_list:
        z.write(ROOT / relative_path, arcname=relative_path)

# Verify ZIP contents
with zipfile.ZipFile(zip_path, "r") as z:
    assert len(z.namelist()) == 12
    assert z.testzip() is None

print("ZIP created successfully.")
print("Files included:", len(file_list))
print("ZIP size: {:.2f} MB".format(os.path.getsize(zip_path) / (1024**2)))

# Download to your computer
files.download(zip_path)

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/twincheck")

folders = [
    ROOT / "data/copies/vision/jpeg85_final"
]

total_bytes = 0
file_count = 0

for folder in folders:
    for path in folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            total_bytes += path.stat().st_size
            file_count += 1

print("JPEG image files:", file_count)
print("Total size: {:.2f} MB".format(total_bytes / (1024**2)))
print("Average file size: {:.2f} KB".format(
    total_bytes / max(file_count, 1) / 1024
))
print("Team's stated image limit: 100 MB")

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/twincheck")

manifest_path = (
    ROOT / "data/copies/vision/jpeg85_final/jpeg85_final_manifest.csv"
)

import pandas as pd

df = pd.read_csv(manifest_path)

twins = df[df["role"] == "twin"]

total_bytes = 0
missing = []

for relative_path in twins["output_path"]:
    path = Path(relative_path)

    if not path.is_absolute():
        path = ROOT / path

    if path.exists():
        total_bytes += path.stat().st_size
    else:
        missing.append(str(path))

print("Twin image count:", len(twins))
print("Twin image total size: {:.2f} MB".format(total_bytes / (1024**2)))
print("Missing twin files:", len(missing))
print("Team limit: 100 MB")

In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
from google.colab import files

ROOT = Path("/content/drive/MyDrive/twincheck")

manifest_path = ROOT / "data/copies/vision/jpeg85_final/jpeg85_final_manifest.csv"
df = pd.read_csv(manifest_path)

# Select only twin images
twins = df[df["role"] == "twin"]

zip_path = Path("/content/jpeg85_twin_images.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for relative_path in twins["output_path"]:
        image_path = Path(relative_path)

        if not image_path.is_absolute():
            image_path = ROOT / image_path

        if image_path.exists():
            z.write(image_path, arcname=image_path.name)
        else:
            print("Missing:", image_path)

print("Twin images:", len(twins))
print("ZIP size:", round(zip_path.stat().st_size / (1024 * 1024), 2), "MB")

# Download ZIP to your computer
files.download(str(zip_path))